# ResNet50 End-to-End Transfer Learning

This notebook implements ResNet50 transfer learning for four-class brain tumour MRI classification using the predefined five-fold cross-validation assignments.

In [ ]:
import sys
import tensorflow as tf


# ============================================================
# VERIFY THE GOOGLE COLAB ENVIRONMENT
# ============================================================

print("=" * 70)
print("COLAB ENVIRONMENT")
print("=" * 70)
print("Python version:", sys.version)
print("TensorFlow version:", tf.__version__)
print("\n--- TensorFlow devices ---")
print("CPU devices:", tf.config.list_physical_devices("CPU"))
print("GPU devices:",tf.config.list_physical_devices("GPU"))
print("\n--- NVIDIA GPU information ---")

!nvidia-smi

COLAB ENVIRONMENT
Python version: 3.13.15 (main, Aug  6 2026, 11:06:23) [GCC 11.4.0]
TensorFlow version: 2.20.0

--- TensorFlow devices ---
CPU devices: [PhysicalDevice(name='/physical_device:CPU:0', device_type='CPU')]
GPU devices: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]

--- NVIDIA GPU information ---
Wed Aug 26 05:09:44 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+================

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
from google.colab import files

uploaded = files.upload()

Saving repeated_stratified_5fold_assignments.csv to repeated_stratified_5fold_assignments.csv


In [ ]:
import shutil
import tarfile
from pathlib import Path


# ============================================================
# RESTORE THE PROJECT DATASET TO THE COLAB RUNTIME
# ============================================================

# Create the path to the project archive stored in Google Drive
DRIVE_ARCHIVE = Path(
    "/content/drive/MyDrive/"
    "brain_tumour_colab/"
    "brain_tumour_colab_bundle.tar"
)


# Create the local path where the archive will temporarily be copied
LOCAL_ARCHIVE = Path("/content/brain_tumour_colab_bundle.tar")


# Create the path where the project will be restored
PROJECT_ROOT = Path("/content/brain-tumour-mri-classification")


print("=" * 70)
print("RESTORE PROJECT DATA")
print("=" * 70)
print("Drive archive:", DRIVE_ARCHIVE)
print("Drive archive exists:", DRIVE_ARCHIVE.exists())
print("Project root:", PROJECT_ROOT)


# Stop if the project archive cannot be found in Google Drive
if not DRIVE_ARCHIVE.exists():
    raise FileNotFoundError(f"Archive not found: {DRIVE_ARCHIVE}")


# Restore the project only if it is not already present in the runtime
if not PROJECT_ROOT.exists():

    print("\nCopying archive to Colab runtime...")
    shutil.copy2(DRIVE_ARCHIVE, LOCAL_ARCHIVE)


    # Create the project directory
    PROJECT_ROOT.mkdir(parents=True, exist_ok=True)


    print("Extracting archive...")

    with tarfile.open(LOCAL_ARCHIVE, "r") as archive:
        archive.extractall(PROJECT_ROOT, filter="data")

else:
    print("\nProject directory already exists. Skipping archive extraction.")


# ============================================================
# VERIFY THE RESTORED PROJECT
# ============================================================

# Create the path to the fixed five-fold cross-validation file
FOLDS_FILE = PROJECT_ROOT / "splits" / "five_fold_cross_validation.csv"


# Create the path to the cropped Training and Testing dataset
DATA_DIR = PROJECT_ROOT / "processed_data_cropped"


# Count every PNG image in the cropped dataset
png_count = len(list(DATA_DIR.rglob("*.png")))

print("\n--- Verification ---")
print("Project root exists:", PROJECT_ROOT.exists())
print("Fold file exists:", FOLDS_FILE.exists())
print("Dataset folder exists:", DATA_DIR.exists())
print("PNG images found:", png_count)


# Stop if the project was not restored correctly
if not FOLDS_FILE.exists():
    raise FileNotFoundError(f"Five-fold cross-validation file not found: {FOLDS_FILE}")


if not DATA_DIR.exists():
    raise FileNotFoundError(f"Cropped dataset folder not found: {DATA_DIR}")


if png_count != 7198:
    raise RuntimeError("Expected 7,198 PNG images, "f"but found {png_count}.")

print("\nProject restored successfully.")

RESTORE PROJECT DATA
Drive archive: /content/drive/MyDrive/brain_tumour_colab/brain_tumour_colab_bundle.tar
Drive archive exists: True
Project root: /content/brain-tumour-mri-classification

Copying archive to Colab runtime...
Extracting archive...

--- Verification ---
Project root exists: True
Fold file exists: True
Dataset folder exists: True
PNG images found: 7198

Project restored successfully.


In [ ]:
# ============================================================
# VALIDATE THE REPEATED 10 x 5 CROSS-VALIDATION ASSIGNMENTS
# ============================================================

from pathlib import Path
import shutil
import pandas as pd


UPLOADED_SPLIT_FILE = Path(
    "/content/repeated_stratified_5fold_assignments.csv"
)

REPEATED_SPLITS_DIR = (
    PROJECT_ROOT
    / "repeated_cv"
    / "splits"
)

REPEATED_SPLITS_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

REPEATED_FOLDS_FILE = (
    REPEATED_SPLITS_DIR
    / "repeated_stratified_5fold_assignments.csv"
)


# Copy the uploaded file into the restored project structure
shutil.copy2(
    UPLOADED_SPLIT_FILE,
    REPEATED_FOLDS_FILE,
)


# Load the assignments
repeated_folds_df = pd.read_csv(
    REPEATED_FOLDS_FILE
)


# ------------------------------------------------------------
# Validation
# ------------------------------------------------------------

assert len(repeated_folds_df) == 56000

assert repeated_folds_df["repeat"].nunique() == 10

assert set(
    repeated_folds_df["repeat"]
) == set(range(1, 11))

assert set(
    repeated_folds_df["fold"]
) == {1, 2, 3, 4, 5}

assert (
    repeated_folds_df
    .groupby(
        ["repeat", "fold"]
    )
    .size()
    .eq(1120)
    .all()
)

assert (
    repeated_folds_df
    .groupby(
        ["repeat", "fold", "class"]
    )
    .size()
    .eq(280)
    .all()
)

assert (
    repeated_folds_df[
        "relative_path"
    ]
    .str.startswith("Training/")
    .all()
)


print(
    "Repeated split file:",
    REPEATED_FOLDS_FILE,
)

print(
    "Rows:",
    len(repeated_folds_df),
)

print(
    "Repeats:",
    repeated_folds_df[
        "repeat"
    ].nunique(),
)

print(
    "Folds per repeat:",
    repeated_folds_df[
        "fold"
    ].nunique(),
)

print(
    "\nSplit seeds:"
)

print(
    repeated_folds_df[
        ["repeat", "split_seed"]
    ]
    .drop_duplicates()
    .sort_values("repeat")
    .to_string(index=False)
)

print(
    "\nRepeated 10 x 5 split validation PASSED."
)

Repeated split file: /content/brain-tumour-mri-classification/repeated_cv/splits/repeated_stratified_5fold_assignments.csv
Rows: 56000
Repeats: 10
Folds per repeat: 5

Split seeds:
 repeat  split_seed
      1      202601
      2      202602
      3      202603
      4      202604
      5      202605
      6      202606
      7      202607
      8      202608
      9      202609
     10      202610

Repeated 10 x 5 split validation PASSED.


In [ ]:
# ============================================================
# Step 5. REPEATED NESTED-CV RESNET50 EXPERIMENT SETUP
# ============================================================

import gc
import json
import os
import random
import time
from pathlib import Path

import numpy as np
import pandas as pd
import tensorflow as tf


# ------------------------------------------------------------
# Dataset
# ------------------------------------------------------------

DATA_DIR = (
    PROJECT_ROOT
    / "processed_data_cropped"
)

assert DATA_DIR.exists()


# ------------------------------------------------------------
# Persistent Google Drive results
# ------------------------------------------------------------

DRIVE_ROOT = Path(
    "/content/drive/MyDrive/brain_tumour_colab"
)

REPEATED_RESNET50_DIR = (
    DRIVE_ROOT
    / "results"
    / "repeated_nested_cv"
    / "resnet50"
)

REPEATED_RESNET50_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


# ------------------------------------------------------------
# Image / class configuration
# ------------------------------------------------------------

IMAGE_HEIGHT = 224
IMAGE_WIDTH = 224
IMAGE_CHANNELS = 3

NUMBER_OF_CLASSES = 4

CLASS_NAMES = [
    "glioma",
    "meningioma",
    "notumor",
    "pituitary",
]

CLASS_TO_INDEX = {
    class_name: index
    for index, class_name
    in enumerate(CLASS_NAMES)
}

INDEX_TO_CLASS = {
    index: class_name
    for class_name, index
    in CLASS_TO_INDEX.items()
}


# ------------------------------------------------------------
# Repeated nested-CV configuration
# ------------------------------------------------------------

NUMBER_OF_REPEATS = 10
NUMBER_OF_OUTER_FOLDS = 5

INNER_VALIDATION_FRACTION = 0.10


# Different deterministic seeds for every repeat/fold
BASE_MODEL_SEED = 505000
BASE_INNER_SPLIT_SEED = 606000


def get_resnet_model_seed(
    repeat_number,
    fold_number,
):
    return (
        BASE_MODEL_SEED
        + repeat_number * 100
        + fold_number
    )


def get_resnet_inner_split_seed(
    repeat_number,
    fold_number,
):
    return (
        BASE_INNER_SPLIT_SEED
        + repeat_number * 100
        + fold_number
    )


# ------------------------------------------------------------
# TensorFlow reproducibility
# ------------------------------------------------------------

try:
    tf.config.experimental.enable_op_determinism()
    determinism_status = "enabled"

except Exception as error:
    determinism_status = (
        f"requested but unavailable: {error}"
    )


tf.keras.backend.set_floatx(
    "float32"
)


# ------------------------------------------------------------
# Safety checks
# ------------------------------------------------------------

gpu_devices = (
    tf.config.list_physical_devices(
        "GPU"
    )
)

assert gpu_devices, (
    "No GPU detected."
)

assert len(
    repeated_folds_df
) == 56000


# ------------------------------------------------------------
# Display configuration
# ------------------------------------------------------------

print("=" * 70)
print("REPEATED NESTED-CV RESNET50 SETUP")
print("=" * 70)

print(
    "GPU:",
    gpu_devices,
)

print(
    "Dataset:",
    DATA_DIR,
)

print(
    "Results directory:",
    REPEATED_RESNET50_DIR,
)

print(
    "\nRepeats:",
    NUMBER_OF_REPEATS,
)

print(
    "Outer folds per repeat:",
    NUMBER_OF_OUTER_FOLDS,
)

print(
    "Total outer evaluations:",
    NUMBER_OF_REPEATS
    * NUMBER_OF_OUTER_FOLDS,
)

print(
    "Internal validation fraction:",
    INNER_VALIDATION_FRACTION,
)

print(
    "\nRepeat 1 / Fold 1 model seed:",
    get_resnet_model_seed(
        1,
        1,
    ),
)

print(
    "Repeat 1 / Fold 1 inner-split seed:",
    get_resnet_inner_split_seed(
        1,
        1,
    ),
)

print(
    "\nTensorFlow deterministic operations:",
    determinism_status,
)

print(
    "\nResNet50 repeated-CV setup PASSED."
)

REPEATED NESTED-CV RESNET50 SETUP
GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
Dataset: /content/brain-tumour-mri-classification/processed_data_cropped
Results directory: /content/drive/MyDrive/brain_tumour_colab/results/repeated_nested_cv/resnet50

Repeats: 10
Outer folds per repeat: 5
Total outer evaluations: 50
Internal validation fraction: 0.1

Repeat 1 / Fold 1 model seed: 505101
Repeat 1 / Fold 1 inner-split seed: 606101

TensorFlow deterministic operations: enabled

ResNet50 repeated-CV setup PASSED.


In [ ]:
# ============================================================
# Step 6. PREPARE REPEATED RESNET50 ASSIGNMENTS
# ============================================================

# Work from the already validated 10 x 5 split table
resnet_repeated_assignments = (
    repeated_folds_df
    .copy()
)

# Add the fixed numeric class index
resnet_repeated_assignments[
    "class_index"
] = (
    resnet_repeated_assignments[
        "class"
    ]
    .map(
        CLASS_TO_INDEX
    )
)

# Add the full image path
resnet_repeated_assignments[
    "image_path"
] = (
    resnet_repeated_assignments[
        "relative_path"
    ]
    .apply(
        lambda relative_path:
            DATA_DIR
            / Path(relative_path)
    )
)


# ------------------------------------------------------------
# Validation
# ------------------------------------------------------------

assert len(
    resnet_repeated_assignments
) == 56000

assert (
    resnet_repeated_assignments[
        "class_index"
    ]
    .notna()
    .all()
)

assert (
    resnet_repeated_assignments[
        "relative_path"
    ]
    .str.startswith("Training/")
    .all()
)

assert all(
    image_path.exists()
    for image_path
    in resnet_repeated_assignments[
        "image_path"
    ]
)

# Every repeat must contain the same 5,600 unique images
images_per_repeat = (
    resnet_repeated_assignments
    .groupby("repeat")[
        "relative_path"
    ]
    .nunique()
)

assert (
    images_per_repeat
    == 5600
).all()


# ------------------------------------------------------------
# Display
# ------------------------------------------------------------

print(
    "Rows:",
    len(
        resnet_repeated_assignments
    ),
)

print(
    "Unique images per repeat:"
)

print(
    images_per_repeat.to_dict()
)

print(
    "\nClass mapping:",
    CLASS_TO_INDEX,
)

print(
    "\nMissing image files: 0"
)

print(
    "\nRepeated ResNet50 assignment preparation PASSED."
)

Rows: 56000
Unique images per repeat:
{1: 5600, 2: 5600, 3: 5600, 4: 5600, 5: 5600, 6: 5600, 7: 5600, 8: 5600, 9: 5600, 10: 5600}

Class mapping: {'glioma': 0, 'meningioma': 1, 'notumor': 2, 'pituitary': 3}

Missing image files: 0

Repeated ResNet50 assignment preparation PASSED.


In [ ]:
# ============================================================
# STEP 7: CORRECT RESNET50 PREPROCESSING AND DATASET FUNCTIONS
# ============================================================

def load_and_preprocess_resnet50_image(
    image_path,
    class_index,
):
    """
    Load one stored 224x224 grayscale PNG and prepare it
    for ImageNet-pretrained ResNet50.
    """

    # Read image file
    image_bytes = tf.io.read_file(
        image_path
    )

    # Decode as one-channel grayscale
    grayscale_image = tf.io.decode_png(
        image_bytes,
        channels=1,
    )

    grayscale_image = tf.ensure_shape(
        grayscale_image,
        (
            IMAGE_HEIGHT,
            IMAGE_WIDTH,
            1,
        ),
    )

    # Convert grayscale -> 3-channel pseudo-RGB
    pseudo_rgb_image = (
        tf.image.grayscale_to_rgb(
            grayscale_image
        )
    )

    pseudo_rgb_image = tf.cast(
        pseudo_rgb_image,
        tf.float32,
    )

    # Apply official ImageNet ResNet50 preprocessing
    resnet50_image = (
        tf.keras.applications.resnet50.preprocess_input(
            pseudo_rgb_image
        )
    )

    resnet50_image = tf.ensure_shape(
        resnet50_image,
        (
            IMAGE_HEIGHT,
            IMAGE_WIDTH,
            IMAGE_CHANNELS,
        ),
    )

    class_index = tf.cast(
        class_index,
        tf.int32,
    )

    return (
        resnet50_image,
        class_index,
    )


def make_resnet50_dataset(
    dataframe,
    training,
    seed,
    batch_size,
):
    """
    Build a deterministic TensorFlow dataset.
    """

    image_paths = (
        dataframe[
            "image_path"
        ]
        .astype(str)
        .to_numpy()
    )

    class_indices = (
        dataframe[
            "class_index"
        ]
        .astype(np.int32)
        .to_numpy()
    )

    dataset = (
        tf.data.Dataset
        .from_tensor_slices(
            (
                image_paths,
                class_indices,
            )
        )
    )

    options = tf.data.Options()

    options.experimental_deterministic = True

    dataset = dataset.with_options(
        options
    )

    if training:

        dataset = dataset.shuffle(
            buffer_size=len(dataframe),
            seed=seed,
            reshuffle_each_iteration=True,
        )

    dataset = dataset.map(
        load_and_preprocess_resnet50_image,
        num_parallel_calls=tf.data.AUTOTUNE,
        deterministic=True,
    )

    dataset = dataset.batch(
        batch_size,
        drop_remainder=False,
    )

    dataset = dataset.prefetch(
        tf.data.AUTOTUNE
    )

    return dataset


# ------------------------------------------------------------
# Test preprocessing on one actual Training image
# ------------------------------------------------------------

test_row = (
    resnet_repeated_assignments
    .iloc[0]
)

test_image, test_label = (
    load_and_preprocess_resnet50_image(
        str(
            test_row[
                "image_path"
            ]
        ),
        int(
            test_row[
                "class_index"
            ]
        ),
    )
)


assert test_image.shape == (
    224,
    224,
    3,
)

assert (
    test_image.dtype
    == tf.float32
)

assert (
    test_label.dtype
    == tf.int32
)

assert bool(
    tf.reduce_all(
        tf.math.is_finite(
            test_image
        )
    )
)


print(
    "Test image shape:",
    test_image.shape,
)

print(
    "Test image dtype:",
    test_image.dtype,
)

print(
    "Test label:",
    int(
        test_label.numpy()
    ),
)

print(
    "All values finite:",
    bool(
        tf.reduce_all(
            tf.math.is_finite(
                test_image
            )
        )
    ),
)

print(
    "\nStep 7 preprocessing validation PASSED."
)

Test image shape: (224, 224, 3)
Test image dtype: <dtype: 'float32'>
Test label: 0
All values finite: True

Step 7 preprocessing validation PASSED.


In [ ]:
# ============================================================
# STEP 8: CREATE THE 90/10 INTERNAL VALIDATION SPLIT
# ============================================================

from sklearn.model_selection import StratifiedShuffleSplit


def make_repeated_resnet_partitions(
    repeat_number,
    fold_number,
):
    """
    For one repeated outer fold, create:

    - 4,032 model-training images
    -   448 internal-validation images
    - 1,120 untouched outer-validation images
    """

    # --------------------------------------------------------
    # Get the 5,600 Training images for this repeat
    # --------------------------------------------------------

    repeat_assignments = (
        resnet_repeated_assignments[
            resnet_repeated_assignments["repeat"]
            == repeat_number
        ]
        .copy()
        .reset_index(drop=True)
    )


    # --------------------------------------------------------
    # Outer validation = the selected outer fold
    # --------------------------------------------------------

    outer_validation = (
        repeat_assignments[
            repeat_assignments["fold"]
            == fold_number
        ]
        .copy()
        .reset_index(drop=True)
    )


    # --------------------------------------------------------
    # Outer training = the other four outer folds
    # --------------------------------------------------------

    outer_training = (
        repeat_assignments[
            repeat_assignments["fold"]
            != fold_number
        ]
        .copy()
        .reset_index(drop=True)
    )


    # --------------------------------------------------------
    # Create a different deterministic 90/10 split
    # for every repeat/fold
    # --------------------------------------------------------

    inner_split_seed = (
        get_resnet_inner_split_seed(
            repeat_number,
            fold_number,
        )
    )

    splitter = StratifiedShuffleSplit(
        n_splits=1,
        test_size=0.10,
        random_state=inner_split_seed,
    )

    (
        model_training_positions,
        internal_validation_positions,
    ) = next(
        splitter.split(
            outer_training,
            outer_training["class_index"],
        )
    )


    # --------------------------------------------------------
    # 90% model-training partition
    # --------------------------------------------------------

    model_training = (
        outer_training
        .iloc[model_training_positions]
        .copy()
        .reset_index(drop=True)
    )


    # --------------------------------------------------------
    # 10% internal-validation partition
    # --------------------------------------------------------

    internal_validation = (
        outer_training
        .iloc[internal_validation_positions]
        .copy()
        .reset_index(drop=True)
    )


    # --------------------------------------------------------
    # Label the three partitions
    # --------------------------------------------------------

    model_training["partition"] = (
        "model_training"
    )

    internal_validation["partition"] = (
        "internal_validation"
    )

    outer_validation["partition"] = (
        "outer_validation"
    )


    # --------------------------------------------------------
    # Combined manifest
    # --------------------------------------------------------

    partition_manifest = pd.concat(
        [
            model_training,
            internal_validation,
            outer_validation,
        ],
        ignore_index=True,
    )


    # --------------------------------------------------------
    # Safety checks
    # --------------------------------------------------------

    assert len(outer_training) == 4480

    assert len(model_training) == 4032

    assert len(internal_validation) == 448

    assert len(outer_validation) == 1120

    assert len(partition_manifest) == 5600


    # No image may occur in more than one partition
    assert not (
        partition_manifest[
            "relative_path"
        ]
        .duplicated()
        .any()
    )


    # Exact class balance
    assert (
        model_training[
            "class"
        ]
        .value_counts()
        .eq(1008)
        .all()
    )

    assert (
        internal_validation[
            "class"
        ]
        .value_counts()
        .eq(112)
        .all()
    )

    assert (
        outer_validation[
            "class"
        ]
        .value_counts()
        .eq(280)
        .all()
    )


    return (
        model_training,
        internal_validation,
        outer_validation,
        partition_manifest,
    )


# ============================================================
# TEST ONLY: Repeat 1 / Fold 1
# ============================================================

(
    test_model_training,
    test_internal_validation,
    test_outer_validation,
    test_partition_manifest,
) = make_repeated_resnet_partitions(
    repeat_number=1,
    fold_number=1,
)


print("=" * 70)
print("REPEAT 1 / FOLD 1 — 90/10 PARTITION CHECK")
print("=" * 70)

print(
    "Outer-training images:",
    len(test_model_training)
    + len(test_internal_validation),
)

print(
    "Model-training images:",
    len(test_model_training),
)

print(
    "Internal-validation images:",
    len(test_internal_validation),
)

print(
    "Untouched outer-validation images:",
    len(test_outer_validation),
)

print(
    "\nModel-training class counts:"
)

print(
    test_model_training[
        "class"
    ]
    .value_counts()
    .sort_index()
)

print(
    "\nInternal-validation class counts:"
)

print(
    test_internal_validation[
        "class"
    ]
    .value_counts()
    .sort_index()
)

print(
    "\nOuter-validation class counts:"
)

print(
    test_outer_validation[
        "class"
    ]
    .value_counts()
    .sort_index()
)

print(
    "\nInner split seed:",
    get_resnet_inner_split_seed(
        1,
        1,
    ),
)

print(
    "\nStep 8 90/10 partition validation PASSED."
)

REPEAT 1 / FOLD 1 — 90/10 PARTITION CHECK
Outer-training images: 4480
Model-training images: 4032
Internal-validation images: 448
Untouched outer-validation images: 1120

Model-training class counts:
class
glioma        1008
meningioma    1008
notumor       1008
pituitary     1008
Name: count, dtype: int64

Internal-validation class counts:
class
glioma        112
meningioma    112
notumor       112
pituitary     112
Name: count, dtype: int64

Outer-validation class counts:
class
glioma        280
meningioma    280
notumor       280
pituitary     280
Name: count, dtype: int64

Inner split seed: 606101

Step 8 90/10 partition validation PASSED.


In [ ]:
# ============================================================
# STEP 9: DEFINE RESNET50 SEARCH AND REPEAT-LEVEL OUTPUTS
# ============================================================

import json
from itertools import product
from pathlib import Path


# ------------------------------------------------------------
# 1. Hyperparameter search space
#
# These 12 configurations will be searched independently
# inside EVERY outer fold using only:
#
#   4,032 model-training images
#     448 internal-validation images
#
# The 1,120 outer-validation images are never used
# to choose these parameters.
# ------------------------------------------------------------

RESNET50_BATCH_SIZES = [
    8,
    16,
    32,
]

RESNET50_HEAD_LEARNING_RATES = [
    1e-3,
    3e-4,
]

RESNET50_FINE_TUNE_LEARNING_RATES = [
    1e-5,
    3e-6,
]


# ------------------------------------------------------------
# 2. Fixed architecture/training settings
# ------------------------------------------------------------

RESNET50_DROPOUT_RATE = 0.30

RESNET50_FINE_TUNE_FROM_LAYER = (
    "conv5_block1_1_conv"
)

RESNET50_HEAD_MAX_EPOCHS = 15

RESNET50_FINE_TUNE_MAX_EPOCHS = 20

RESNET50_EARLY_STOPPING_PATIENCE = 3

RESNET50_HEAD_MIN_LR = 1e-6

RESNET50_FINE_TUNE_MIN_LR = 1e-7


# ------------------------------------------------------------
# 3. Build the 12 candidate configurations
# ------------------------------------------------------------

resnet50_search_configurations = []

configuration_number = 0


for (
    batch_size,
    head_learning_rate,
    fine_tune_learning_rate,
) in product(
    RESNET50_BATCH_SIZES,
    RESNET50_HEAD_LEARNING_RATES,
    RESNET50_FINE_TUNE_LEARNING_RATES,
):

    configuration_number += 1

    configuration_id = (
        f"config_{configuration_number:02d}"
        f"_bs{batch_size}"
        f"_headlr{head_learning_rate:.0e}"
        f"_ftlr{fine_tune_learning_rate:.0e}"
    )

    resnet50_search_configurations.append(
        {
            "configuration_number":
                configuration_number,

            "configuration_id":
                configuration_id,

            "batch_size":
                batch_size,

            "head_learning_rate":
                head_learning_rate,

            "fine_tune_learning_rate":
                fine_tune_learning_rate,
        }
    )


resnet50_search_table = pd.DataFrame(
    resnet50_search_configurations
)


assert len(
    resnet50_search_table
) == 12


# ------------------------------------------------------------
# 4. Configuration-selection rule
#
# Each outer fold will choose ONE winning configuration using
# only its 448-image INTERNAL validation set.
# ------------------------------------------------------------

RESNET50_SELECTION_RULE = [
    "highest internal-validation macro F1",
    "highest internal-validation balanced accuracy",
    "lowest internal-validation log loss",
    "lowest configuration number",
]


# ------------------------------------------------------------
# 5. Persistent output structure
#
# Fold results are saved separately for resume protection.
# After all 5 folds finish, they are aggregated into ONE
# repeat/separation-level result.
# ------------------------------------------------------------

for repeat_number in range(
    1,
    NUMBER_OF_REPEATS + 1,
):

    repeat_directory = (
        REPEATED_RESNET50_DIR
        / f"repeat_{repeat_number:02d}"
    )

    repeat_directory.mkdir(
        parents=True,
        exist_ok=True,
    )


# Final table used later for paired statistical testing.
#
# It will eventually contain exactly 10 rows:
# one row for each repeat/separation.
REPEAT_LEVEL_SUMMARY_PATH = (
    REPEATED_RESNET50_DIR
    / "repeat_level_summary.csv"
)


# ------------------------------------------------------------
# 6. Save experiment definition
# ------------------------------------------------------------

SEARCH_DEFINITION_PATH = (
    REPEATED_RESNET50_DIR
    / "repeated_cv_search_definition.json"
)


search_definition = {
    "model":
        "ResNet50",

    "number_of_repeats":
        10,

    "outer_folds_per_repeat":
        5,

    "total_outer_evaluations":
        50,

    "outer_training_images":
        4480,

    "internal_model_training_images":
        4032,

    "internal_validation_images":
        448,

    "outer_validation_images":
        1120,

    "internal_validation_fraction":
        0.10,

    "number_of_candidate_configurations":
        12,

    "batch_sizes":
        RESNET50_BATCH_SIZES,

    "head_learning_rates":
        RESNET50_HEAD_LEARNING_RATES,

    "fine_tune_learning_rates":
        RESNET50_FINE_TUNE_LEARNING_RATES,

    "dropout_rate":
        RESNET50_DROPOUT_RATE,

    "fine_tune_from_layer":
        RESNET50_FINE_TUNE_FROM_LAYER,

    "head_max_epochs":
        RESNET50_HEAD_MAX_EPOCHS,

    "fine_tune_max_epochs":
        RESNET50_FINE_TUNE_MAX_EPOCHS,

    "early_stopping_patience":
        RESNET50_EARLY_STOPPING_PATIENCE,

    "selection_rule":
        RESNET50_SELECTION_RULE,

    "final_statistical_unit":
        (
            "mean of 5 outer-fold results "
            "within each repeat"
        ),

    "expected_repeat_level_results":
        10,

    "testing_images_used":
        False,
}


temporary_definition_path = (
    SEARCH_DEFINITION_PATH
    .with_name(
        SEARCH_DEFINITION_PATH.name
        + ".tmp"
    )
)


with temporary_definition_path.open(
    "w",
    encoding="utf-8",
) as file:

    json.dump(
        search_definition,
        file,
        indent=4,
    )


temporary_definition_path.replace(
    SEARCH_DEFINITION_PATH
)


# ------------------------------------------------------------
# 7. Display
# ------------------------------------------------------------

print("=" * 70)
print("RESNET50 REPEATED-CV SEARCH DEFINITION")
print("=" * 70)

print(
    "Candidate configurations:",
    len(
        resnet50_search_configurations
    ),
)

display(
    resnet50_search_table
)

print(
    "\nOuter folds per repeat:",
    NUMBER_OF_OUTER_FOLDS,
)

print(
    "Repeats/separations:",
    NUMBER_OF_REPEATS,
)

print(
    "Total outer evaluations:",
    NUMBER_OF_REPEATS
    * NUMBER_OF_OUTER_FOLDS,
)

print(
    "\nParameters are re-selected "
    "inside EVERY outer fold."
)

print(
    "\nFold-level results will be saved "
    "for checkpoint/resume protection."
)

print(
    "Five folds will then be averaged "
    "into ONE result per repeat."
)

print(
    "\nFinal Wilcoxon table will contain:",
    10,
    "ResNet50 values.",
)

print(
    "\nRepeat-level summary path:"
)

print(
    REPEAT_LEVEL_SUMMARY_PATH
)

print(
    "\nStep 9 search/output definition PASSED."
)

RESNET50 REPEATED-CV SEARCH DEFINITION
Candidate configurations: 12


,configuration_number,configuration_id,batch_size,head_learning_rate,fine_tune_learning_rate
0,1,config_01_bs8_headlr1e-03_ftlr1e-05,8,0.0010,0.000010
1,2,config_02_bs8_headlr1e-03_ftlr3e-06,8,0.0010,0.000003
2,3,config_03_bs8_headlr3e-04_ftlr1e-05,8,0.0003,0.000010
3,4,config_04_bs8_headlr3e-04_ftlr3e-06,8,0.0003,0.000003
4,5,config_05_bs16_headlr1e-03_ftlr1e-05,16,0.0010,0.000010
5,6,config_06_bs16_headlr1e-03_ftlr3e-06,16,0.0010,0.000003
6,7,config_07_bs16_headlr3e-04_ftlr1e-05,16,0.0003,0.000010
7,8,config_08_bs16_headlr3e-04_ftlr3e-06,16,0.0003,0.000003
8,9,config_09_bs32_headlr1e-03_ftlr1e-05,32,0.0010,0.000010
9,10,config_10_bs32_headlr1e-03_ftlr3e-06,32,0.0010,0.000003



Outer folds per repeat: 5
Repeats/separations: 10
Total outer evaluations: 50

Parameters are re-selected inside EVERY outer fold.

Fold-level results will be saved for checkpoint/resume protection.
Five folds will then be averaged into ONE result per repeat.

Final Wilcoxon table will contain: 10 ResNet50 values.

Repeat-level summary path:
/content/drive/MyDrive/brain_tumour_colab/results/repeated_nested_cv/resnet50/repeat_level_summary.csv

Step 9 search/output definition PASSED.


In [ ]:
# ============================================================
# STEP 10: DEFINE RESNET50 MODEL AND TRAINING UTILITIES
# ============================================================

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    log_loss,
    precision_recall_fscore_support,
)


# ------------------------------------------------------------
# 1. Build a fresh ImageNet-pretrained ResNet50
# ------------------------------------------------------------

def build_repeated_resnet50_model():
    """
    Build a fresh ResNet50 transfer-learning model.

    Stage 1:
        - ImageNet backbone frozen
        - new classification head trained

    Stage 2:
        - upper ResNet50 layers unfrozen
        - Batch Normalization layers remain frozen
    """

    inputs = tf.keras.Input(
        shape=(
            IMAGE_HEIGHT,
            IMAGE_WIDTH,
            IMAGE_CHANNELS,
        ),
        name="input_image",
    )

    base_model = (
        tf.keras.applications.ResNet50(
            include_top=False,
            weights="imagenet",
            input_shape=(
                IMAGE_HEIGHT,
                IMAGE_WIDTH,
                IMAGE_CHANNELS,
            ),
            pooling="avg",
        )
    )

    # Stage 1 starts with complete backbone frozen
    base_model.trainable = False

    # Keep pretrained BatchNorm behaviour in inference mode
    features = base_model(
        inputs,
        training=False,
    )

    features = tf.keras.layers.Dropout(
        rate=RESNET50_DROPOUT_RATE,
        name="classifier_dropout",
    )(
        features
    )

    outputs = tf.keras.layers.Dense(
        units=NUMBER_OF_CLASSES,
        activation="softmax",
        dtype="float32",
        name="class_probabilities",
    )(
        features
    )

    model = tf.keras.Model(
        inputs=inputs,
        outputs=outputs,
        name="resnet50_repeated_cv",
    )

    return model, base_model


# ------------------------------------------------------------
# 2. Compile model
# ------------------------------------------------------------

def compile_repeated_resnet50(
    model,
    learning_rate,
):
    model.compile(
        optimizer=tf.keras.optimizers.Adam(
            learning_rate=learning_rate
        ),
        loss=(
            tf.keras.losses
            .SparseCategoricalCrossentropy()
        ),
        metrics=[
            tf.keras.metrics
            .SparseCategoricalAccuracy(
                name="accuracy"
            )
        ],
    )


# ------------------------------------------------------------
# 3. Enable Stage-2 fine-tuning
# ------------------------------------------------------------

def enable_repeated_resnet50_fine_tuning(
    base_model,
):
    """
    Unfreeze layers beginning at conv5_block1_1_conv.

    Batch Normalization layers remain frozen.
    """

    base_model.trainable = True

    start_found = False

    for layer in base_model.layers:

        if (
            layer.name
            == RESNET50_FINE_TUNE_FROM_LAYER
        ):
            start_found = True

        layer.trainable = (
            start_found
            and not isinstance(
                layer,
                tf.keras.layers.BatchNormalization,
            )
        )

    if not start_found:
        raise ValueError(
            "Fine-tuning start layer "
            "was not found: "
            f"{RESNET50_FINE_TUNE_FROM_LAYER}"
        )

    trainable_layers = [
        layer.name
        for layer in base_model.layers
        if layer.trainable
    ]

    if not trainable_layers:
        raise RuntimeError(
            "No ResNet50 backbone layers "
            "were enabled for fine-tuning."
        )

    return trainable_layers


# ------------------------------------------------------------
# 4. Training callbacks
# ------------------------------------------------------------

def create_repeated_resnet50_callbacks(
    checkpoint_path,
    minimum_learning_rate,
):
    """
    All checkpoint/early-stopping decisions use ONLY the
    448-image internal-validation partition.
    """

    return [
        tf.keras.callbacks.ModelCheckpoint(
            filepath=str(
                checkpoint_path
            ),
            monitor="val_loss",
            mode="min",
            save_best_only=True,
            save_weights_only=True,
            verbose=0,
        ),

        tf.keras.callbacks.EarlyStopping(
            monitor="val_loss",
            mode="min",
            patience=(
                RESNET50_EARLY_STOPPING_PATIENCE
            ),
            restore_best_weights=True,
            verbose=0,
        ),

        tf.keras.callbacks.ReduceLROnPlateau(
            monitor="val_loss",
            mode="min",
            factor=0.2,
            patience=2,
            min_lr=minimum_learning_rate,
            verbose=0,
        ),

        tf.keras.callbacks.TerminateOnNaN(),
    ]


# ------------------------------------------------------------
# 5. Find best epoch and validation loss
# ------------------------------------------------------------

def get_repeated_resnet50_best_epoch(
    history,
):
    validation_losses = np.asarray(
        history.history["val_loss"],
        dtype=np.float64,
    )

    if validation_losses.size == 0:
        raise RuntimeError(
            "No validation losses were recorded."
        )

    if not np.isfinite(
        validation_losses
    ).all():
        raise RuntimeError(
            "Non-finite validation loss detected."
        )

    best_position = int(
        np.argmin(
            validation_losses
        )
    )

    return {
        "best_epoch":
            best_position + 1,

        "best_validation_loss":
            float(
                validation_losses[
                    best_position
                ]
            ),
    }


# ------------------------------------------------------------
# 6. Classification metrics
# ------------------------------------------------------------

def calculate_repeated_resnet50_metrics(
    true_labels,
    predicted_labels,
    probabilities,
):
    """
    Used for both internal-validation model selection
    and final untouched outer-fold evaluation.
    """

    macro_scores = (
        precision_recall_fscore_support(
            true_labels,
            predicted_labels,
            average="macro",
            zero_division=0,
        )
    )

    return {
        "accuracy":
            float(
                accuracy_score(
                    true_labels,
                    predicted_labels,
                )
            ),

        "balanced_accuracy":
            float(
                balanced_accuracy_score(
                    true_labels,
                    predicted_labels,
                )
            ),

        "macro_precision":
            float(
                macro_scores[0]
            ),

        "macro_recall":
            float(
                macro_scores[1]
            ),

        "macro_f1":
            float(
                macro_scores[2]
            ),

        "log_loss":
            float(
                log_loss(
                    true_labels,
                    probabilities,
                    labels=list(
                        range(
                            NUMBER_OF_CLASSES
                        )
                    ),
                )
            ),
    }


# ------------------------------------------------------------
# 7. Basic model-construction test
# ------------------------------------------------------------

tf.keras.backend.clear_session()

test_model, test_base_model = (
    build_repeated_resnet50_model()
)

assert (
    test_model.output_shape
    == (None, 4)
)

assert (
    test_base_model.trainable
    is False
)


print("=" * 70)
print("RESNET50 MODEL UTILITY CHECK")
print("=" * 70)

print(
    "Model output shape:",
    test_model.output_shape,
)

print(
    "Backbone initially trainable:",
    test_base_model.trainable,
)

print(
    "Dropout rate:",
    RESNET50_DROPOUT_RATE,
)

print(
    "Fine-tuning starts from:",
    RESNET50_FINE_TUNE_FROM_LAYER,
)

print(
    "\nStep 10 model/training utilities PASSED."
)


# Release the temporary test model
del test_model
del test_base_model

tf.keras.backend.clear_session()
gc.collect()

94765736/94765736 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step
RESNET50 MODEL UTILITY CHECK
Model output shape: (None, 4)
Backbone initially trainable: False
Dropout rate: 0.3
Fine-tuning starts from: conv5_block1_1_conv

Step 10 model/training utilities PASSED.


0

In [ ]:
# ============================================================
# STEP 11: DEFINE RESNET50 CANDIDATE SAVE / RESUME UTILITIES
# ============================================================

import json
from pathlib import Path


# ------------------------------------------------------------
# Atomic JSON save
# ------------------------------------------------------------

def save_json_atomic(
    data,
    destination,
):
    destination = Path(
        destination
    )

    temporary_path = (
        destination.with_name(
            destination.name + ".tmp"
        )
    )

    with temporary_path.open(
        "w",
        encoding="utf-8",
    ) as file:

        json.dump(
            data,
            file,
            indent=4,
            default=str,
        )

    temporary_path.replace(
        destination
    )


# ------------------------------------------------------------
# Atomic CSV save
# ------------------------------------------------------------

def save_dataframe_atomic(
    dataframe,
    destination,
    index=False,
):
    destination = Path(
        destination
    )

    temporary_path = (
        destination.with_name(
            destination.name + ".tmp"
        )
    )

    dataframe.to_csv(
        temporary_path,
        index=index,
    )

    temporary_path.replace(
        destination
    )


# ------------------------------------------------------------
# Candidate output directory
# ------------------------------------------------------------

def get_resnet50_candidate_directory(
    repeat_number,
    fold_number,
    configuration_id,
):
    return (
        REPEATED_RESNET50_DIR
        / f"repeat_{repeat_number:02d}"
        / f"fold_{fold_number:02d}"
        / "configurations"
        / configuration_id
    )


# ------------------------------------------------------------
# Check whether one candidate completed successfully
# ------------------------------------------------------------

def validate_completed_resnet50_candidate(
    repeat_number,
    fold_number,
    configuration_id,
):
    candidate_directory = (
        get_resnet50_candidate_directory(
            repeat_number,
            fold_number,
            configuration_id,
        )
    )

    configuration_path = (
        candidate_directory
        / "configuration.json"
    )

    metrics_path = (
        candidate_directory
        / "internal_validation_metrics.json"
    )

    history_path = (
        candidate_directory
        / "training_history.csv"
    )

    completion_path = (
        candidate_directory
        / "complete.json"
    )

    required_paths = [
        configuration_path,
        metrics_path,
        history_path,
        completion_path,
    ]

    if not all(
        path.exists()
        for path in required_paths
    ):
        return False

    try:

        with completion_path.open(
            "r",
            encoding="utf-8",
        ) as file:

            completion = json.load(
                file
            )

        if (
            completion.get("status")
            != "completed"
        ):
            return False

        if (
            int(
                completion["repeat"]
            )
            != repeat_number
        ):
            return False

        if (
            int(
                completion["fold"]
            )
            != fold_number
        ):
            return False

        if (
            completion[
                "configuration_id"
            ]
            != configuration_id
        ):
            return False

        with metrics_path.open(
            "r",
            encoding="utf-8",
        ) as file:

            metrics = json.load(
                file
            )

        history = pd.read_csv(
            history_path
        )

        if history.empty:
            return False

        if (
            not np.isfinite(
                float(
                    metrics[
                        "internal_macro_f1"
                    ]
                )
            )
        ):
            return False

        return True

    except Exception:

        return False


# ------------------------------------------------------------
# Test directory construction
# ------------------------------------------------------------

test_candidate_directory = (
    get_resnet50_candidate_directory(
        repeat_number=1,
        fold_number=1,
        configuration_id=(
            resnet50_search_configurations[
                0
            ][
                "configuration_id"
            ]
        ),
    )
)


print("=" * 70)
print("RESNET50 SAVE / RESUME STRUCTURE")
print("=" * 70)

print(
    "Example candidate directory:"
)

print(
    test_candidate_directory
)

print(
    "\nCandidate completion currently:",
    validate_completed_resnet50_candidate(
        repeat_number=1,
        fold_number=1,
        configuration_id=(
            resnet50_search_configurations[
                0
            ][
                "configuration_id"
            ]
        ),
    ),
)

print(
    "\nEach configuration will save:"
)

print(
    "  configuration.json"
)

print(
    "  internal_validation_metrics.json"
)

print(
    "  training_history.csv"
)

print(
    "  complete.json"
)

print(
    "\nStep 11 candidate persistence setup PASSED."
)

RESNET50 SAVE / RESUME STRUCTURE
Example candidate directory:
/content/drive/MyDrive/brain_tumour_colab/results/repeated_nested_cv/resnet50/repeat_01/fold_01/configurations/config_01_bs8_headlr1e-03_ftlr1e-05

Candidate completion currently: True

Each configuration will save:
  configuration.json
  internal_validation_metrics.json
  training_history.csv
  complete.json

Step 11 candidate persistence setup PASSED.


In [ ]:
# ============================================================
# STEP 12: TEST ONE RESNET50 CANDIDATE
#
# Repeat 1 / Fold 1 / Configuration 1 ONLY
#
# IMPORTANT:
#   - Outer-validation images are NOT evaluated here.
#   - This tests only the inner model-selection procedure.
# ============================================================

import gc
import random
import shutil
import time

from sklearn.metrics import (
    balanced_accuracy_score,
    f1_score,
    log_loss,
)


TEST_REPEAT = 1
TEST_FOLD = 1

TEST_CONFIGURATION = (
    resnet50_search_configurations[0]
)

TEST_CONFIGURATION_ID = (
    TEST_CONFIGURATION[
        "configuration_id"
    ]
)

TEST_BATCH_SIZE = int(
    TEST_CONFIGURATION[
        "batch_size"
    ]
)

TEST_HEAD_LR = float(
    TEST_CONFIGURATION[
        "head_learning_rate"
    ]
)

TEST_FINE_TUNE_LR = float(
    TEST_CONFIGURATION[
        "fine_tune_learning_rate"
    ]
)


# ------------------------------------------------------------
# 1. Create nested partitions
# ------------------------------------------------------------

(
    model_training_dataframe,
    internal_validation_dataframe,
    outer_validation_dataframe,
    partition_manifest,
) = make_repeated_resnet_partitions(
    repeat_number=TEST_REPEAT,
    fold_number=TEST_FOLD,
)


assert len(model_training_dataframe) == 4032
assert len(internal_validation_dataframe) == 448
assert len(outer_validation_dataframe) == 1120


# ------------------------------------------------------------
# 2. Candidate output directory
# ------------------------------------------------------------

candidate_directory = (
    get_resnet50_candidate_directory(
        repeat_number=TEST_REPEAT,
        fold_number=TEST_FOLD,
        configuration_id=TEST_CONFIGURATION_ID,
    )
)


# Remove any incomplete previous attempt
if candidate_directory.exists():

    if not validate_completed_resnet50_candidate(
        TEST_REPEAT,
        TEST_FOLD,
        TEST_CONFIGURATION_ID,
    ):

        print(
            "Removing incomplete previous candidate output..."
        )

        shutil.rmtree(
            candidate_directory
        )


candidate_directory.mkdir(
    parents=True,
    exist_ok=True,
)


configuration_path = (
    candidate_directory
    / "configuration.json"
)

metrics_path = (
    candidate_directory
    / "internal_validation_metrics.json"
)

history_path = (
    candidate_directory
    / "training_history.csv"
)

completion_path = (
    candidate_directory
    / "complete.json"
)

stage_1_checkpoint = (
    candidate_directory
    / "stage_1_best.weights.h5"
)

stage_2_checkpoint = (
    candidate_directory
    / "stage_2_best.weights.h5"
)


# ------------------------------------------------------------
# 3. Reproducible model/training seed
# ------------------------------------------------------------

candidate_seed = (
    get_resnet_model_seed(
        TEST_REPEAT,
        TEST_FOLD,
    )
)


tf.keras.backend.clear_session()

random.seed(
    candidate_seed
)

np.random.seed(
    candidate_seed
)

tf.keras.utils.set_random_seed(
    candidate_seed
)

gc.collect()


# ------------------------------------------------------------
# 4. Build TensorFlow datasets
# ------------------------------------------------------------

training_dataset = (
    make_resnet50_dataset(
        model_training_dataframe,
        training=True,
        seed=candidate_seed,
        batch_size=TEST_BATCH_SIZE,
    )
)

internal_validation_dataset = (
    make_resnet50_dataset(
        internal_validation_dataframe,
        training=False,
        seed=candidate_seed,
        batch_size=TEST_BATCH_SIZE,
    )
)


# ------------------------------------------------------------
# 5. Build fresh ImageNet ResNet50
# ------------------------------------------------------------

model, base_model = (
    build_repeated_resnet50_model()
)


candidate_start_time = (
    time.perf_counter()
)


# ============================================================
# STAGE 1 — FROZEN BACKBONE
# ============================================================

compile_repeated_resnet50(
    model,
    TEST_HEAD_LR,
)


print("=" * 70)

print(
    f"Repeat {TEST_REPEAT} / "
    f"Fold {TEST_FOLD} / "
    f"{TEST_CONFIGURATION_ID}"
)

print("=" * 70)

print(
    "Batch size:",
    TEST_BATCH_SIZE,
)

print(
    "Stage 1 learning rate:",
    TEST_HEAD_LR,
)

print(
    "Stage 2 learning rate:",
    TEST_FINE_TUNE_LR,
)

print(
    "\nModel-training images:",
    len(model_training_dataframe),
)

print(
    "Internal-validation images:",
    len(internal_validation_dataframe),
)

print(
    "Outer-validation images currently UNUSED:",
    len(outer_validation_dataframe),
)


print(
    "\nStage 1 — frozen-head training"
)

stage_1_start = (
    time.perf_counter()
)

stage_1_history = model.fit(
    training_dataset,

    validation_data=(
        internal_validation_dataset
    ),

    epochs=(
        RESNET50_HEAD_MAX_EPOCHS
    ),

    callbacks=(
        create_repeated_resnet50_callbacks(
            stage_1_checkpoint,
            RESNET50_HEAD_MIN_LR,
        )
    ),

    verbose=1,
)

stage_1_seconds = (
    time.perf_counter()
    - stage_1_start
)


stage_1_best = (
    get_repeated_resnet50_best_epoch(
        stage_1_history
    )
)


assert stage_1_checkpoint.exists()


# Restore the best Stage-1 checkpoint before fine-tuning
model.load_weights(
    stage_1_checkpoint
)


# ============================================================
# STAGE 2 — FINE-TUNING
# ============================================================

trainable_layers = (
    enable_repeated_resnet50_fine_tuning(
        base_model
    )
)

compile_repeated_resnet50(
    model,
    TEST_FINE_TUNE_LR,
)


print(
    "\nStage 2 — fine-tuning"
)

print(
    "Trainable backbone layers:",
    len(trainable_layers),
)


stage_2_start = (
    time.perf_counter()
)

stage_2_history = model.fit(
    training_dataset,

    validation_data=(
        internal_validation_dataset
    ),

    epochs=(
        RESNET50_FINE_TUNE_MAX_EPOCHS
    ),

    callbacks=(
        create_repeated_resnet50_callbacks(
            stage_2_checkpoint,
            RESNET50_FINE_TUNE_MIN_LR,
        )
    ),

    verbose=1,
)

stage_2_seconds = (
    time.perf_counter()
    - stage_2_start
)


stage_2_best = (
    get_repeated_resnet50_best_epoch(
        stage_2_history
    )
)


assert stage_2_checkpoint.exists()


# ------------------------------------------------------------
# 6. Select Stage 1 or Stage 2 using INTERNAL validation only
# ------------------------------------------------------------

if (
    stage_2_best[
        "best_validation_loss"
    ]
    <
    stage_1_best[
        "best_validation_loss"
    ]
):

    selected_stage = (
        "fine_tuning"
    )

    selected_checkpoint = (
        stage_2_checkpoint
    )

    selected_epoch = int(
        stage_2_best[
            "best_epoch"
        ]
    )

    selected_validation_loss = float(
        stage_2_best[
            "best_validation_loss"
        ]
    )

else:

    selected_stage = (
        "frozen_head"
    )

    selected_checkpoint = (
        stage_1_checkpoint
    )

    selected_epoch = int(
        stage_1_best[
            "best_epoch"
        ]
    )

    selected_validation_loss = float(
        stage_1_best[
            "best_validation_loss"
        ]
    )


model.load_weights(
    selected_checkpoint
)


# ------------------------------------------------------------
# 7. Evaluate selected candidate on INTERNAL validation
# ------------------------------------------------------------

internal_probabilities = (
    model.predict(
        internal_validation_dataset,
        verbose=1,
    )
)

internal_probabilities = np.asarray(
    internal_probabilities,
    dtype=np.float32,
)


assert internal_probabilities.shape == (
    448,
    NUMBER_OF_CLASSES,
)

assert np.isfinite(
    internal_probabilities
).all()


internal_probability_sums = (
    internal_probabilities.sum(
        axis=1
    )
)

assert np.allclose(
    internal_probability_sums,
    1.0,
    atol=1e-5,
)


internal_true_labels = (
    internal_validation_dataframe[
        "class_index"
    ]
    .to_numpy(
        dtype=np.int32
    )
)

internal_predicted_labels = (
    np.argmax(
        internal_probabilities,
        axis=1,
    )
    .astype(np.int32)
)


internal_macro_f1 = float(
    f1_score(
        internal_true_labels,
        internal_predicted_labels,
        average="macro",
        zero_division=0,
    )
)

internal_balanced_accuracy = float(
    balanced_accuracy_score(
        internal_true_labels,
        internal_predicted_labels,
    )
)

internal_log_loss = float(
    log_loss(
        internal_true_labels,
        internal_probabilities,
        labels=list(
            range(
                NUMBER_OF_CLASSES
            )
        ),
    )
)


candidate_total_seconds = (
    time.perf_counter()
    - candidate_start_time
)


# ------------------------------------------------------------
# 8. Save combined training history
# ------------------------------------------------------------

stage_1_history_df = pd.DataFrame(
    stage_1_history.history
)

stage_1_history_df.insert(
    0,
    "stage_epoch",
    np.arange(
        1,
        len(stage_1_history_df) + 1,
    ),
)

stage_1_history_df.insert(
    1,
    "stage",
    "frozen_head",
)


stage_2_history_df = pd.DataFrame(
    stage_2_history.history
)

stage_2_history_df.insert(
    0,
    "stage_epoch",
    np.arange(
        1,
        len(stage_2_history_df) + 1,
    ),
)

stage_2_history_df.insert(
    1,
    "stage",
    "fine_tuning",
)


combined_history_df = pd.concat(
    [
        stage_1_history_df,
        stage_2_history_df,
    ],
    ignore_index=True,
)


# ------------------------------------------------------------
# 9. Save candidate configuration
# ------------------------------------------------------------

candidate_configuration = {
    "model":
        "ResNet50",

    "repeat":
        TEST_REPEAT,

    "fold":
        TEST_FOLD,

    "configuration_number":
        int(
            TEST_CONFIGURATION[
                "configuration_number"
            ]
        ),

    "configuration_id":
        TEST_CONFIGURATION_ID,

    "batch_size":
        TEST_BATCH_SIZE,

    "head_learning_rate":
        TEST_HEAD_LR,

    "fine_tune_learning_rate":
        TEST_FINE_TUNE_LR,

    "model_seed":
        candidate_seed,

    "inner_split_seed":
        get_resnet_inner_split_seed(
            TEST_REPEAT,
            TEST_FOLD,
        ),

    "model_training_images":
        4032,

    "internal_validation_images":
        448,

    "outer_validation_images":
        1120,

    "outer_validation_used_for_selection":
        False,
}


candidate_metrics = {
    "model":
        "ResNet50",

    "repeat":
        TEST_REPEAT,

    "fold":
        TEST_FOLD,

    "configuration_id":
        TEST_CONFIGURATION_ID,

    "selected_stage":
        selected_stage,

    "selected_epoch":
        selected_epoch,

    "selected_validation_loss":
        selected_validation_loss,

    "stage_1_best_epoch":
        int(
            stage_1_best[
                "best_epoch"
            ]
        ),

    "stage_1_best_validation_loss":
        float(
            stage_1_best[
                "best_validation_loss"
            ]
        ),

    "stage_2_best_epoch":
        int(
            stage_2_best[
                "best_epoch"
            ]
        ),

    "stage_2_best_validation_loss":
        float(
            stage_2_best[
                "best_validation_loss"
            ]
        ),

    "internal_macro_f1":
        internal_macro_f1,

    "internal_balanced_accuracy":
        internal_balanced_accuracy,

    "internal_log_loss":
        internal_log_loss,

    "stage_1_training_seconds":
        float(
            stage_1_seconds
        ),

    "stage_2_training_seconds":
        float(
            stage_2_seconds
        ),

    "total_candidate_seconds":
        float(
            candidate_total_seconds
        ),
}


# ------------------------------------------------------------
# 10. Save everything atomically
# ------------------------------------------------------------

save_dataframe_atomic(
    combined_history_df,
    history_path,
    index=False,
)

save_json_atomic(
    candidate_configuration,
    configuration_path,
)

save_json_atomic(
    candidate_metrics,
    metrics_path,
)


# Verify before writing complete.json
assert history_path.exists()
assert configuration_path.exists()
assert metrics_path.exists()

saved_history = pd.read_csv(
    history_path
)

assert not saved_history.empty


# ------------------------------------------------------------
# 11. Completion marker LAST
# ------------------------------------------------------------

completion_information = {
    "status":
        "completed",

    "repeat":
        TEST_REPEAT,

    "fold":
        TEST_FOLD,

    "configuration_id":
        TEST_CONFIGURATION_ID,

    "internal_macro_f1":
        internal_macro_f1,

    "outer_validation_used":
        False,
}


save_json_atomic(
    completion_information,
    completion_path,
)


assert validate_completed_resnet50_candidate(
    TEST_REPEAT,
    TEST_FOLD,
    TEST_CONFIGURATION_ID,
)


# ------------------------------------------------------------
# 12. Remove large temporary checkpoints
# ------------------------------------------------------------

for checkpoint_path in [
    stage_1_checkpoint,
    stage_2_checkpoint,
]:

    if checkpoint_path.exists():
        checkpoint_path.unlink()


# ------------------------------------------------------------
# 13. Results
# ------------------------------------------------------------

print(
    "\n"
    + "=" * 70
)

print(
    "STEP 12 SINGLE-CANDIDATE TEST COMPLETED"
)

print(
    "=" * 70
)

print(
    "Configuration:",
    TEST_CONFIGURATION_ID,
)

print(
    "Selected stage:",
    selected_stage,
)

print(
    "Selected epoch:",
    selected_epoch,
)

print(
    "Internal macro F1:",
    round(
        internal_macro_f1,
        6,
    ),
)

print(
    "Internal balanced accuracy:",
    round(
        internal_balanced_accuracy,
        6,
    ),
)

print(
    "Internal log loss:",
    round(
        internal_log_loss,
        6,
    ),
)

print(
    "Stage 1 training time:",
    round(
        stage_1_seconds / 60,
        2,
    ),
    "minutes",
)

print(
    "Stage 2 training time:",
    round(
        stage_2_seconds / 60,
        2,
    ),
    "minutes",
)

print(
    "\nSaved to:"
)

print(
    candidate_directory
)

print(
    "\nOuter validation was NOT used."
)

print(
    "\nStep 12 single-candidate validation PASSED."
)


# ------------------------------------------------------------
# Cleanup
# ------------------------------------------------------------

del model
del base_model
del training_dataset
del internal_validation_dataset
del internal_probabilities
del stage_1_history
del stage_2_history

tf.keras.backend.clear_session()

gc.collect()

In [ ]:
# ============================================================
# STEP 13: FULL RESNET50 REPEATED NESTED-CV EXPERIMENT
#
# 10 repeats × 5 outer folds = 50 outer evaluations
#
# For EACH outer fold:
#
#   4,480 outer-training images
#       ↓
#   4,032 model training
#     448 internal validation
#       ↓
#   search all 12 ResNet50 configurations
#       ↓
#   select winner using INTERNAL validation only
#       ↓
#   rebuild fresh ResNet50
#       ↓
#   refit winner on ALL 4,480 outer-training images
#       ↓
#   evaluate ONCE on untouched 1,120 outer-validation images
#
# After 5 folds:
#   aggregate into ONE repeat/separation result.
#
# Final statistical table:
#   10 rows = 10 repeats/separations
# ============================================================

import gc
import json
import random
import shutil
import time
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
import tensorflow as tf

from sklearn.metrics import (
    classification_report,
    confusion_matrix,
)


# ============================================================
# 1. ADDITIONAL REPRODUCIBLE REFIT SEED
# ============================================================

BASE_REFIT_SEED = 707000


def get_resnet_refit_seed(
    repeat_number,
    fold_number,
):
    return (
        BASE_REFIT_SEED
        + repeat_number * 100
        + fold_number
    )


# ============================================================
# 2. FINAL FOLD OUTPUT VALIDATION
# ============================================================

def validate_completed_resnet50_fold(
    repeat_number,
    fold_number,
):
    """
    A fold counts as completed only when all required
    final outputs exist and pass basic validation.
    """

    fold_directory = (
        REPEATED_RESNET50_DIR
        / f"repeat_{repeat_number:02d}"
        / f"fold_{fold_number:02d}"
    )

    metrics_path = (
        fold_directory
        / "outer_fold_metrics.json"
    )

    selected_configuration_path = (
        fold_directory
        / "selected_configuration.json"
    )

    predictions_path = (
        fold_directory
        / "outer_validation_predictions.csv"
    )

    confusion_matrix_path = (
        fold_directory
        / "confusion_matrix.csv"
    )

    classification_report_path = (
        fold_directory
        / "classification_report.csv"
    )

    partition_manifest_path = (
        fold_directory
        / "partition_manifest.csv"
    )

    candidate_leaderboard_path = (
        fold_directory
        / "candidate_leaderboard.csv"
    )

    final_history_path = (
        fold_directory
        / "final_refit_history.csv"
    )

    final_weights_path = (
        fold_directory
        / "final_selected_model.weights.h5"
    )

    completion_path = (
        fold_directory
        / "fold_complete.json"
    )

    required_paths = [
        metrics_path,
        selected_configuration_path,
        predictions_path,
        confusion_matrix_path,
        classification_report_path,
        partition_manifest_path,
        candidate_leaderboard_path,
        final_history_path,
        final_weights_path,
        completion_path,
    ]

    if not all(
        path.exists()
        for path in required_paths
    ):
        return False

    try:

        with completion_path.open(
            "r",
            encoding="utf-8",
        ) as file:

            completion = json.load(
                file
            )

        if (
            completion.get("status")
            != "completed"
        ):
            return False

        if (
            int(completion["repeat"])
            != repeat_number
        ):
            return False

        if (
            int(completion["fold"])
            != fold_number
        ):
            return False

        with metrics_path.open(
            "r",
            encoding="utf-8",
        ) as file:

            metrics = json.load(
                file
            )

        if (
            int(metrics["repeat"])
            != repeat_number
        ):
            return False

        if (
            int(metrics["fold"])
            != fold_number
        ):
            return False

        predictions = pd.read_csv(
            predictions_path
        )

        if len(predictions) != 1120:
            return False

        if (
            predictions[
                "relative_path"
            ]
            .nunique()
            != 1120
        ):
            return False

        leaderboard = pd.read_csv(
            candidate_leaderboard_path
        )

        if len(leaderboard) != 12:
            return False

        history = pd.read_csv(
            final_history_path
        )

        if history.empty:
            return False

        return True

    except Exception:

        return False


# ============================================================
# 3. RUN ONE INNER CANDIDATE
# ============================================================

def run_resnet50_candidate(
    repeat_number,
    fold_number,
    configuration,
    model_training_dataframe,
    internal_validation_dataframe,
):
    """
    Train one candidate using only the 4,032/448
    internal train/validation split.

    Outer validation is NEVER used here.
    """

    configuration_id = (
        configuration[
            "configuration_id"
        ]
    )

    configuration_number = int(
        configuration[
            "configuration_number"
        ]
    )

    batch_size = int(
        configuration[
            "batch_size"
        ]
    )

    head_learning_rate = float(
        configuration[
            "head_learning_rate"
        ]
    )

    fine_tune_learning_rate = float(
        configuration[
            "fine_tune_learning_rate"
        ]
    )


    # --------------------------------------------------------
    # Candidate directory
    # --------------------------------------------------------

    candidate_directory = (
        get_resnet50_candidate_directory(
            repeat_number,
            fold_number,
            configuration_id,
        )
    )


    # --------------------------------------------------------
    # Resume
    # --------------------------------------------------------

    if validate_completed_resnet50_candidate(
        repeat_number,
        fold_number,
        configuration_id,
    ):

        print(
            f"  {configuration_id}: "
            "already completed — skipping."
        )

        metrics_path = (
            candidate_directory
            / "internal_validation_metrics.json"
        )

        with metrics_path.open(
            "r",
            encoding="utf-8",
        ) as file:

            return json.load(
                file
            )


    # --------------------------------------------------------
    # Delete incomplete candidate only
    # --------------------------------------------------------

    if candidate_directory.exists():

        print(
            f"  {configuration_id}: "
            "incomplete output found — rerunning."
        )

        shutil.rmtree(
            candidate_directory
        )


    candidate_directory.mkdir(
        parents=True,
        exist_ok=True,
    )


    configuration_path = (
        candidate_directory
        / "configuration.json"
    )

    metrics_path = (
        candidate_directory
        / "internal_validation_metrics.json"
    )

    history_path = (
        candidate_directory
        / "training_history.csv"
    )

    completion_path = (
        candidate_directory
        / "complete.json"
    )

    stage_1_checkpoint = (
        candidate_directory
        / "stage_1_best.weights.h5"
    )

    stage_2_checkpoint = (
        candidate_directory
        / "stage_2_best.weights.h5"
    )


    # --------------------------------------------------------
    # Same deterministic model seed for all candidates
    # within the same outer fold.
    #
    # This prevents random initialization differences
    # from unfairly favouring one hyperparameter set.
    # --------------------------------------------------------

    candidate_seed = (
        get_resnet_model_seed(
            repeat_number,
            fold_number,
        )
    )


    tf.keras.backend.clear_session()

    random.seed(
        candidate_seed
    )

    np.random.seed(
        candidate_seed
    )

    tf.keras.utils.set_random_seed(
        candidate_seed
    )

    gc.collect()


    # --------------------------------------------------------
    # Datasets
    # --------------------------------------------------------

    training_dataset = (
        make_resnet50_dataset(
            model_training_dataframe,
            training=True,
            seed=candidate_seed,
            batch_size=batch_size,
        )
    )

    internal_validation_dataset = (
        make_resnet50_dataset(
            internal_validation_dataframe,
            training=False,
            seed=candidate_seed,
            batch_size=batch_size,
        )
    )


    # --------------------------------------------------------
    # Fresh ResNet50
    # --------------------------------------------------------

    model, base_model = (
        build_repeated_resnet50_model()
    )


    candidate_start = (
        time.perf_counter()
    )


    # ========================================================
    # STAGE 1 — frozen backbone
    # ========================================================

    compile_repeated_resnet50(
        model,
        head_learning_rate,
    )


    stage_1_start = (
        time.perf_counter()
    )


    stage_1_history = model.fit(
        training_dataset,

        validation_data=(
            internal_validation_dataset
        ),

        epochs=(
            RESNET50_HEAD_MAX_EPOCHS
        ),

        callbacks=(
            create_repeated_resnet50_callbacks(
                stage_1_checkpoint,
                RESNET50_HEAD_MIN_LR,
            )
        ),

        verbose=0,
    )


    stage_1_seconds = (
        time.perf_counter()
        - stage_1_start
    )


    stage_1_best = (
        get_repeated_resnet50_best_epoch(
            stage_1_history
        )
    )


    if not stage_1_checkpoint.exists():

        raise RuntimeError(
            "Stage 1 checkpoint was not saved."
        )


    model.load_weights(
        stage_1_checkpoint
    )


    # ========================================================
    # STAGE 2 — fine-tuning
    # ========================================================

    trainable_layers = (
        enable_repeated_resnet50_fine_tuning(
            base_model
        )
    )


    compile_repeated_resnet50(
        model,
        fine_tune_learning_rate,
    )


    stage_2_start = (
        time.perf_counter()
    )


    stage_2_history = model.fit(
        training_dataset,

        validation_data=(
            internal_validation_dataset
        ),

        epochs=(
            RESNET50_FINE_TUNE_MAX_EPOCHS
        ),

        callbacks=(
            create_repeated_resnet50_callbacks(
                stage_2_checkpoint,
                RESNET50_FINE_TUNE_MIN_LR,
            )
        ),

        verbose=0,
    )


    stage_2_seconds = (
        time.perf_counter()
        - stage_2_start
    )


    stage_2_best = (
        get_repeated_resnet50_best_epoch(
            stage_2_history
        )
    )


    if not stage_2_checkpoint.exists():

        raise RuntimeError(
            "Stage 2 checkpoint was not saved."
        )


    # ========================================================
    # SELECT STAGE USING INTERNAL VALIDATION ONLY
    # ========================================================

    if (
        stage_2_best[
            "best_validation_loss"
        ]
        <
        stage_1_best[
            "best_validation_loss"
        ]
    ):

        selected_stage = (
            "fine_tuning"
        )

        selected_checkpoint = (
            stage_2_checkpoint
        )

        selected_epoch = int(
            stage_2_best[
                "best_epoch"
            ]
        )

        selected_validation_loss = float(
            stage_2_best[
                "best_validation_loss"
            ]
        )

    else:

        selected_stage = (
            "frozen_head"
        )

        selected_checkpoint = (
            stage_1_checkpoint
        )

        selected_epoch = int(
            stage_1_best[
                "best_epoch"
            ]
        )

        selected_validation_loss = float(
            stage_1_best[
                "best_validation_loss"
            ]
        )


    model.load_weights(
        selected_checkpoint
    )


    # ========================================================
    # INTERNAL VALIDATION PREDICTIONS
    # ========================================================

    internal_probabilities = (
        model.predict(
            internal_validation_dataset,
            verbose=0,
        )
    )


    internal_probabilities = (
        np.asarray(
            internal_probabilities,
            dtype=np.float32,
        )
    )


    if (
        internal_probabilities.shape
        != (
            448,
            NUMBER_OF_CLASSES,
        )
    ):

        raise RuntimeError(
            "Unexpected internal-validation "
            "prediction shape."
        )


    if not np.isfinite(
        internal_probabilities
    ).all():

        raise RuntimeError(
            "Non-finite internal-validation "
            "probabilities detected."
        )


    internal_true_labels = (
        internal_validation_dataframe[
            "class_index"
        ]
        .to_numpy(
            dtype=np.int32
        )
    )


    internal_predicted_labels = (
        np.argmax(
            internal_probabilities,
            axis=1,
        )
        .astype(np.int32)
    )


    internal_metrics = (
        calculate_repeated_resnet50_metrics(
            internal_true_labels,
            internal_predicted_labels,
            internal_probabilities,
        )
    )


    candidate_total_seconds = (
        time.perf_counter()
        - candidate_start
    )


    # ========================================================
    # SAVE TRAINING HISTORY
    # ========================================================

    stage_1_history_df = (
        pd.DataFrame(
            stage_1_history.history
        )
    )

    stage_1_history_df.insert(
        0,
        "stage_epoch",
        np.arange(
            1,
            len(stage_1_history_df) + 1,
        ),
    )

    stage_1_history_df.insert(
        1,
        "stage",
        "frozen_head",
    )


    stage_2_history_df = (
        pd.DataFrame(
            stage_2_history.history
        )
    )

    stage_2_history_df.insert(
        0,
        "stage_epoch",
        np.arange(
            1,
            len(stage_2_history_df) + 1,
        ),
    )

    stage_2_history_df.insert(
        1,
        "stage",
        "fine_tuning",
    )


    combined_history_df = pd.concat(
        [
            stage_1_history_df,
            stage_2_history_df,
        ],
        ignore_index=True,
    )


    # ========================================================
    # CANDIDATE RECORD
    # ========================================================

    candidate_configuration = {
        "model":
            "ResNet50",

        "repeat":
            int(repeat_number),

        "fold":
            int(fold_number),

        "configuration_number":
            configuration_number,

        "configuration_id":
            configuration_id,

        "batch_size":
            batch_size,

        "head_learning_rate":
            head_learning_rate,

        "fine_tune_learning_rate":
            fine_tune_learning_rate,

        "model_seed":
            int(candidate_seed),

        "inner_split_seed":
            int(
                get_resnet_inner_split_seed(
                    repeat_number,
                    fold_number,
                )
            ),

        "model_training_images":
            4032,

        "internal_validation_images":
            448,

        "outer_validation_used_for_selection":
            False,
    }


    candidate_metrics = {
        "model":
            "ResNet50",

        "repeat":
            int(repeat_number),

        "fold":
            int(fold_number),

        "configuration_number":
            configuration_number,

        "configuration_id":
            configuration_id,

        "batch_size":
            batch_size,

        "head_learning_rate":
            head_learning_rate,

        "fine_tune_learning_rate":
            fine_tune_learning_rate,

        "selected_stage":
            selected_stage,

        "selected_epoch":
            selected_epoch,

        "selected_validation_loss":
            selected_validation_loss,

        "stage_1_best_epoch":
            int(
                stage_1_best[
                    "best_epoch"
                ]
            ),

        "stage_1_best_validation_loss":
            float(
                stage_1_best[
                    "best_validation_loss"
                ]
            ),

        "stage_2_best_epoch":
            int(
                stage_2_best[
                    "best_epoch"
                ]
            ),

        "stage_2_best_validation_loss":
            float(
                stage_2_best[
                    "best_validation_loss"
                ]
            ),

        "internal_accuracy":
            float(
                internal_metrics[
                    "accuracy"
                ]
            ),

        "internal_balanced_accuracy":
            float(
                internal_metrics[
                    "balanced_accuracy"
                ]
            ),

        "internal_macro_precision":
            float(
                internal_metrics[
                    "macro_precision"
                ]
            ),

        "internal_macro_recall":
            float(
                internal_metrics[
                    "macro_recall"
                ]
            ),

        "internal_macro_f1":
            float(
                internal_metrics[
                    "macro_f1"
                ]
            ),

        "internal_log_loss":
            float(
                internal_metrics[
                    "log_loss"
                ]
            ),

        "stage_1_training_seconds":
            float(
                stage_1_seconds
            ),

        "stage_2_training_seconds":
            float(
                stage_2_seconds
            ),

        "total_candidate_seconds":
            float(
                candidate_total_seconds
            ),

        "trainable_backbone_layers":
            len(
                trainable_layers
            ),
    }


    # ========================================================
    # SAVE CANDIDATE
    # ========================================================

    save_dataframe_atomic(
        combined_history_df,
        history_path,
        index=False,
    )

    save_json_atomic(
        candidate_configuration,
        configuration_path,
    )

    save_json_atomic(
        candidate_metrics,
        metrics_path,
    )


    # Completion marker LAST
    completion_information = {
        "status":
            "completed",

        "repeat":
            int(repeat_number),

        "fold":
            int(fold_number),

        "configuration_id":
            configuration_id,

        "internal_macro_f1":
            float(
                internal_metrics[
                    "macro_f1"
                ]
            ),

        "outer_validation_used":
            False,

        "completed_at":
            datetime.now().isoformat(),
    }


    save_json_atomic(
        completion_information,
        completion_path,
    )


    if not validate_completed_resnet50_candidate(
        repeat_number,
        fold_number,
        configuration_id,
    ):

        raise RuntimeError(
            "Candidate persistence validation failed."
        )


    # Candidate checkpoints no longer needed.
    for checkpoint_path in [
        stage_1_checkpoint,
        stage_2_checkpoint,
    ]:

        if checkpoint_path.exists():

            checkpoint_path.unlink()


    print(
        f"  {configuration_id}: "
        f"macro F1="
        f"{internal_metrics['macro_f1']:.4f} "
        f"stage={selected_stage}"
    )


    # Cleanup
    del model
    del base_model
    del training_dataset
    del internal_validation_dataset
    del internal_probabilities
    del stage_1_history
    del stage_2_history

    tf.keras.backend.clear_session()

    gc.collect()


    return candidate_metrics


# ============================================================
# 4. BUILD/UPDATE GLOBAL 10-REPEAT SUMMARY
# ============================================================

def update_resnet50_repeat_level_summary():
    """
    Reconstruct the global statistical table from all completed
    repeat_summary.json files.

    This makes the summary safe to rebuild after interruption.
    """

    records = []


    for repeat_number in range(
        1,
        NUMBER_OF_REPEATS + 1,
    ):

        repeat_summary_path = (
            REPEATED_RESNET50_DIR
            / f"repeat_{repeat_number:02d}"
            / "repeat_summary.json"
        )


        if not repeat_summary_path.exists():

            continue


        try:

            with repeat_summary_path.open(
                "r",
                encoding="utf-8",
            ) as file:

                record = json.load(
                    file
                )

            if (
                record.get("status")
                == "completed"
            ):

                records.append(
                    record
                )

        except Exception:

            continue


    if not records:

        return


    repeat_summary_df = pd.DataFrame(
        records
    )


    repeat_summary_df = (
        repeat_summary_df
        .sort_values(
            "repeat"
        )
        .reset_index(
            drop=True
        )
    )


    save_dataframe_atomic(
        repeat_summary_df,
        REPEAT_LEVEL_SUMMARY_PATH,
        index=False,
    )


# ============================================================
# 5. MAIN 10 × 5 LOOP
# ============================================================

for repeat_number in range(
    1,
    NUMBER_OF_REPEATS + 1,
):

    print(
        "\n"
        + "#" * 80
    )

    print(
        f"RESNET50 — REPEAT / SEPARATION "
        f"{repeat_number:02d} OF "
        f"{NUMBER_OF_REPEATS}"
    )

    print(
        "#" * 80
    )


    repeat_directory = (
        REPEATED_RESNET50_DIR
        / f"repeat_{repeat_number:02d}"
    )


    repeat_directory.mkdir(
        parents=True,
        exist_ok=True,
    )


    # ========================================================
    # FIVE OUTER FOLDS
    # ========================================================

    for fold_number in range(
        1,
        NUMBER_OF_OUTER_FOLDS + 1,
    ):

        print(
            "\n"
            + "=" * 80
        )

        print(
            f"REPEAT {repeat_number:02d} "
            f"/ FOLD {fold_number:02d}"
        )

        print(
            "=" * 80
        )


        fold_directory = (
            repeat_directory
            / f"fold_{fold_number:02d}"
        )


        fold_directory.mkdir(
            parents=True,
            exist_ok=True,
        )


        # ----------------------------------------------------
        # Complete fold -> skip
        # ----------------------------------------------------

        if validate_completed_resnet50_fold(
            repeat_number,
            fold_number,
        ):

            print(
                "Valid completed fold found — skipping."
            )

            continue


        # ----------------------------------------------------
        # Create 4032 / 448 / 1120 partitions
        # ----------------------------------------------------

        (
            model_training_dataframe,
            internal_validation_dataframe,
            outer_validation_dataframe,
            partition_manifest,
        ) = make_repeated_resnet_partitions(
            repeat_number,
            fold_number,
        )


        # The complete 4,480 outer-training data.
        outer_training_dataframe = (
            pd.concat(
                [
                    model_training_dataframe,
                    internal_validation_dataframe,
                ],
                ignore_index=True,
            )
        )


        assert (
            len(
                outer_training_dataframe
            )
            == 4480
        )

        assert (
            outer_training_dataframe[
                "relative_path"
            ]
            .nunique()
            == 4480
        )


        # ----------------------------------------------------
        # Save exact partition manifest
        # ----------------------------------------------------

        partition_manifest_path = (
            fold_directory
            / "partition_manifest.csv"
        )


        manifest_columns = [
            "relative_path",
            "class",
            "class_index",
            "repeat",
            "fold",
            "split_seed",
            "partition",
        ]


        save_dataframe_atomic(
            partition_manifest[
                manifest_columns
            ],
            partition_manifest_path,
            index=False,
        )


        print(
            "Model training:",
            len(
                model_training_dataframe
            ),
        )

        print(
            "Internal validation:",
            len(
                internal_validation_dataframe
            ),
        )

        print(
            "Untouched outer validation:",
            len(
                outer_validation_dataframe
            ),
        )


        # ====================================================
        # SEARCH ALL 12 CONFIGURATIONS
        # ====================================================

        print(
            "\nSearching 12 configurations..."
        )


        candidate_records = []


        for configuration in (
            resnet50_search_configurations
        ):

            candidate_metrics = (
                run_resnet50_candidate(
                    repeat_number,
                    fold_number,
                    configuration,
                    model_training_dataframe,
                    internal_validation_dataframe,
                )
            )

            candidate_records.append(
                candidate_metrics
            )


        # ====================================================
        # CANDIDATE LEADERBOARD
        # ====================================================

        candidate_leaderboard = (
            pd.DataFrame(
                candidate_records
            )
        )


        if len(
            candidate_leaderboard
        ) != 12:

            raise RuntimeError(
                "Expected 12 completed candidate results."
            )


        # Predetermined selection:
        #
        # 1 highest internal macro F1
        # 2 highest balanced accuracy
        # 3 lowest log loss
        # 4 lowest configuration number
        candidate_leaderboard = (
            candidate_leaderboard
            .sort_values(
                by=[
                    "internal_macro_f1",
                    "internal_balanced_accuracy",
                    "internal_log_loss",
                    "configuration_number",
                ],

                ascending=[
                    False,
                    False,
                    True,
                    True,
                ],

                kind="mergesort",
            )
            .reset_index(
                drop=True
            )
        )


        candidate_leaderboard.insert(
            0,
            "rank",
            np.arange(
                1,
                len(candidate_leaderboard) + 1,
            ),
        )


        candidate_leaderboard_path = (
            fold_directory
            / "candidate_leaderboard.csv"
        )


        save_dataframe_atomic(
            candidate_leaderboard,
            candidate_leaderboard_path,
            index=False,
        )


        # ====================================================
        # WINNING CONFIGURATION
        # ====================================================

        winning_row = (
            candidate_leaderboard.iloc[0]
        )


        selected_configuration_id = str(
            winning_row[
                "configuration_id"
            ]
        )

        selected_batch_size = int(
            winning_row[
                "batch_size"
            ]
        )

        selected_head_lr = float(
            winning_row[
                "head_learning_rate"
            ]
        )

        selected_fine_tune_lr = float(
            winning_row[
                "fine_tune_learning_rate"
            ]
        )

        selected_stage = str(
            winning_row[
                "selected_stage"
            ]
        )

        selected_stage_1_epochs = int(
            winning_row[
                "stage_1_best_epoch"
            ]
        )

        selected_stage_2_epochs = int(
            winning_row[
                "stage_2_best_epoch"
            ]
        )


        print(
            "\nSelected configuration:",
            selected_configuration_id,
        )

        print(
            "Batch size:",
            selected_batch_size,
        )

        print(
            "Head learning rate:",
            selected_head_lr,
        )

        print(
            "Fine-tuning learning rate:",
            selected_fine_tune_lr,
        )

        print(
            "Selected stage:",
            selected_stage,
        )

        print(
            "Internal macro F1:",
            f"{winning_row['internal_macro_f1']:.6f}",
        )


        # ====================================================
        # SAVE WINNING CONFIGURATION
        # ====================================================

        selected_configuration = {
            "model":
                "ResNet50",

            "repeat":
                int(
                    repeat_number
                ),

            "fold":
                int(
                    fold_number
                ),

            "configuration_id":
                selected_configuration_id,

            "configuration_number":
                int(
                    winning_row[
                        "configuration_number"
                    ]
                ),

            "batch_size":
                selected_batch_size,

            "head_learning_rate":
                selected_head_lr,

            "fine_tune_learning_rate":
                selected_fine_tune_lr,

            "selected_stage":
                selected_stage,

            "stage_1_refit_epochs":
                selected_stage_1_epochs,

            "stage_2_refit_epochs":
                (
                    selected_stage_2_epochs
                    if selected_stage
                    == "fine_tuning"
                    else 0
                ),

            "internal_macro_f1":
                float(
                    winning_row[
                        "internal_macro_f1"
                    ]
                ),

            "internal_balanced_accuracy":
                float(
                    winning_row[
                        "internal_balanced_accuracy"
                    ]
                ),

            "internal_log_loss":
                float(
                    winning_row[
                        "internal_log_loss"
                    ]
                ),

            "outer_validation_used_for_selection":
                False,
        }


        selected_configuration_path = (
            fold_directory
            / "selected_configuration.json"
        )


        save_json_atomic(
            selected_configuration,
            selected_configuration_path,
        )


        # ====================================================
        # FINAL REFIT ON ALL 4,480 OUTER-TRAINING IMAGES
        # ====================================================

        print(
            "\nRefitting selected configuration "
            "on all 4,480 outer-training images..."
        )


        refit_seed = (
            get_resnet_refit_seed(
                repeat_number,
                fold_number,
            )
        )


        tf.keras.backend.clear_session()

        random.seed(
            refit_seed
        )

        np.random.seed(
            refit_seed
        )

        tf.keras.utils.set_random_seed(
            refit_seed
        )

        gc.collect()


        final_training_dataset = (
            make_resnet50_dataset(
                outer_training_dataframe,
                training=True,
                seed=refit_seed,
                batch_size=(
                    selected_batch_size
                ),
            )
        )


        outer_validation_dataset = (
            make_resnet50_dataset(
                outer_validation_dataframe,
                training=False,
                seed=refit_seed,
                batch_size=(
                    selected_batch_size
                ),
            )
        )


        final_model, final_base_model = (
            build_repeated_resnet50_model()
        )


        final_refit_start = (
            time.perf_counter()
        )


        # ====================================================
        # FINAL STAGE 1
        # ====================================================

        compile_repeated_resnet50(
            final_model,
            selected_head_lr,
        )


        final_stage_1_start = (
            time.perf_counter()
        )


        final_stage_1_history = (
            final_model.fit(
                final_training_dataset,

                epochs=(
                    selected_stage_1_epochs
                ),

                callbacks=[
                    tf.keras.callbacks
                    .TerminateOnNaN()
                ],

                verbose=0,
            )
        )


        final_stage_1_seconds = (
            time.perf_counter()
            - final_stage_1_start
        )


        final_stage_2_history = None
        final_stage_2_seconds = 0.0


        # ====================================================
        # FINAL STAGE 2 if selected
        # ====================================================

        if (
            selected_stage
            == "fine_tuning"
        ):

            enable_repeated_resnet50_fine_tuning(
                final_base_model
            )


            compile_repeated_resnet50(
                final_model,
                selected_fine_tune_lr,
            )


            final_stage_2_start = (
                time.perf_counter()
            )


            final_stage_2_history = (
                final_model.fit(
                    final_training_dataset,

                    epochs=(
                        selected_stage_2_epochs
                    ),

                    callbacks=[
                        tf.keras.callbacks
                        .TerminateOnNaN()
                    ],

                    verbose=0,
                )
            )


            final_stage_2_seconds = (
                time.perf_counter()
                - final_stage_2_start
            )


        final_refit_seconds = (
            time.perf_counter()
            - final_refit_start
        )


        # ====================================================
        # SAVE FINAL REFIT HISTORY
        # ====================================================

        final_stage_1_history_df = (
            pd.DataFrame(
                final_stage_1_history.history
            )
        )


        final_stage_1_history_df.insert(
            0,
            "stage_epoch",
            np.arange(
                1,
                len(
                    final_stage_1_history_df
                ) + 1,
            ),
        )


        final_stage_1_history_df.insert(
            1,
            "stage",
            "frozen_head",
        )


        final_history_frames = [
            final_stage_1_history_df
        ]


        if (
            final_stage_2_history
            is not None
        ):

            final_stage_2_history_df = (
                pd.DataFrame(
                    final_stage_2_history.history
                )
            )


            final_stage_2_history_df.insert(
                0,
                "stage_epoch",
                np.arange(
                    1,
                    len(
                        final_stage_2_history_df
                    ) + 1,
                ),
            )


            final_stage_2_history_df.insert(
                1,
                "stage",
                "fine_tuning",
            )


            final_history_frames.append(
                final_stage_2_history_df
            )


        final_refit_history = pd.concat(
            final_history_frames,
            ignore_index=True,
        )


        final_history_path = (
            fold_directory
            / "final_refit_history.csv"
        )


        save_dataframe_atomic(
            final_refit_history,
            final_history_path,
            index=False,
        )


        # ====================================================
        # SAVE FINAL FOLD MODEL WEIGHTS
        # ====================================================

        final_weights_path = (
            fold_directory
            / "final_selected_model.weights.h5"
        )


        temporary_weights_path = (
            fold_directory
            / "final_selected_model.tmp.weights.h5"
        )


        if temporary_weights_path.exists():

            temporary_weights_path.unlink()


        final_model.save_weights(
            temporary_weights_path
        )


        temporary_weights_path.replace(
            final_weights_path
        )


        # ====================================================
        # OUTER VALIDATION — USED ONCE
        # ====================================================

        print(
            "Evaluating untouched outer fold..."
        )


        inference_start = (
            time.perf_counter()
        )


        outer_probabilities = (
            final_model.predict(
                outer_validation_dataset,
                verbose=0,
            )
        )


        inference_seconds = (
            time.perf_counter()
            - inference_start
        )


        outer_probabilities = (
            np.asarray(
                outer_probabilities,
                dtype=np.float32,
            )
        )


        expected_shape = (
            1120,
            NUMBER_OF_CLASSES,
        )


        if (
            outer_probabilities.shape
            != expected_shape
        ):

            raise RuntimeError(
                "Unexpected outer-validation "
                "probability shape."
            )


        if not np.isfinite(
            outer_probabilities
        ).all():

            raise RuntimeError(
                "Non-finite outer-validation "
                "probabilities detected."
            )


        if not np.allclose(
            outer_probabilities.sum(
                axis=1
            ),
            1.0,
            atol=1e-5,
        ):

            raise RuntimeError(
                "Outer probability rows "
                "do not sum to one."
            )


        outer_true_labels = (
            outer_validation_dataframe[
                "class_index"
            ]
            .to_numpy(
                dtype=np.int32
            )
        )


        outer_predicted_labels = (
            np.argmax(
                outer_probabilities,
                axis=1,
            )
            .astype(np.int32)
        )


        outer_metrics = (
            calculate_repeated_resnet50_metrics(
                outer_true_labels,
                outer_predicted_labels,
                outer_probabilities,
            )
        )


        # ====================================================
        # OUTER PREDICTIONS
        # ====================================================

        predictions_dataframe = (
            outer_validation_dataframe[
                [
                    "relative_path",
                    "class",
                    "class_index",
                    "repeat",
                    "fold",
                ]
            ]
            .copy()
        )


        predictions_dataframe = (
            predictions_dataframe.rename(
                columns={
                    "class":
                        "true_class",

                    "class_index":
                        "true_class_index",
                }
            )
        )


        predictions_dataframe[
            "predicted_class_index"
        ] = (
            outer_predicted_labels
        )


        predictions_dataframe[
            "predicted_class"
        ] = [
            INDEX_TO_CLASS[
                int(index)
            ]
            for index
            in outer_predicted_labels
        ]


        predictions_dataframe[
            "correct"
        ] = (
            predictions_dataframe[
                "true_class_index"
            ]
            ==
            predictions_dataframe[
                "predicted_class_index"
            ]
        )


        for (
            class_index,
            class_name,
        ) in enumerate(
            CLASS_NAMES
        ):

            predictions_dataframe[
                f"probability_{class_name}"
            ] = (
                outer_probabilities[
                    :,
                    class_index,
                ]
            )


        predictions_path = (
            fold_directory
            / "outer_validation_predictions.csv"
        )


        save_dataframe_atomic(
            predictions_dataframe,
            predictions_path,
            index=False,
        )


        # ====================================================
        # CONFUSION MATRIX
        # ====================================================

        fold_confusion_matrix = (
            confusion_matrix(
                outer_true_labels,
                outer_predicted_labels,
                labels=list(
                    range(
                        NUMBER_OF_CLASSES
                    )
                ),
            )
        )


        confusion_matrix_dataframe = (
            pd.DataFrame(
                fold_confusion_matrix,
                index=CLASS_NAMES,
                columns=CLASS_NAMES,
            )
        )


        confusion_matrix_path = (
            fold_directory
            / "confusion_matrix.csv"
        )


        save_dataframe_atomic(
            confusion_matrix_dataframe,
            confusion_matrix_path,
            index=True,
        )


        # ====================================================
        # CLASSIFICATION REPORT
        # ====================================================

        fold_report = (
            classification_report(
                outer_true_labels,
                outer_predicted_labels,

                labels=list(
                    range(
                        NUMBER_OF_CLASSES
                    )
                ),

                target_names=(
                    CLASS_NAMES
                ),

                output_dict=True,

                zero_division=0,
            )
        )


        classification_report_dataframe = (
            pd.DataFrame(
                fold_report
            )
            .transpose()
        )


        classification_report_path = (
            fold_directory
            / "classification_report.csv"
        )


        save_dataframe_atomic(
            classification_report_dataframe,
            classification_report_path,
            index=True,
        )


        # ====================================================
        # FINAL OUTER-FOLD METRICS
        # ====================================================

        candidate_search_seconds = float(
            candidate_leaderboard[
                "total_candidate_seconds"
            ].sum()
        )


        fold_metrics = {
            "model":
                "ResNet50",

            "repeat":
                int(
                    repeat_number
                ),

            "fold":
                int(
                    fold_number
                ),

            "outer_split_seed":
                int(
                    partition_manifest[
                        "split_seed"
                    ].iloc[0]
                ),

            "inner_split_seed":
                int(
                    get_resnet_inner_split_seed(
                        repeat_number,
                        fold_number,
                    )
                ),

            "candidate_model_seed":
                int(
                    get_resnet_model_seed(
                        repeat_number,
                        fold_number,
                    )
                ),

            "final_refit_seed":
                int(
                    refit_seed
                ),

            "selected_configuration_id":
                selected_configuration_id,

            "selected_batch_size":
                selected_batch_size,

            "selected_head_learning_rate":
                selected_head_lr,

            "selected_fine_tune_learning_rate":
                selected_fine_tune_lr,

            "selected_stage":
                selected_stage,

            "selected_stage_1_epochs":
                selected_stage_1_epochs,

            "selected_stage_2_epochs":
                (
                    selected_stage_2_epochs
                    if selected_stage
                    == "fine_tuning"
                    else 0
                ),

            "outer_train_images":
                4480,

            "internal_model_training_images":
                4032,

            "internal_validation_images":
                448,

            "outer_validation_images":
                1120,

            "number_of_candidates":
                12,

            "accuracy":
                float(
                    outer_metrics[
                        "accuracy"
                    ]
                ),

            "balanced_accuracy":
                float(
                    outer_metrics[
                        "balanced_accuracy"
                    ]
                ),

            "macro_precision":
                float(
                    outer_metrics[
                        "macro_precision"
                    ]
                ),

            "macro_recall":
                float(
                    outer_metrics[
                        "macro_recall"
                    ]
                ),

            "macro_f1":
                float(
                    outer_metrics[
                        "macro_f1"
                    ]
                ),

            "log_loss":
                float(
                    outer_metrics[
                        "log_loss"
                    ]
                ),

            "candidate_search_seconds":
                candidate_search_seconds,

            "final_stage_1_seconds":
                float(
                    final_stage_1_seconds
                ),

            "final_stage_2_seconds":
                float(
                    final_stage_2_seconds
                ),

            "final_refit_seconds":
                float(
                    final_refit_seconds
                ),

            "outer_inference_seconds":
                float(
                    inference_seconds
                ),

            "outer_validation_used_for_selection":
                False,

            "testing_images_used":
                False,
        }


        metrics_path = (
            fold_directory
            / "outer_fold_metrics.json"
        )


        save_json_atomic(
            fold_metrics,
            metrics_path,
        )


        # ====================================================
        # VERIFY FOLD OUTPUTS
        # ====================================================

        required_fold_outputs = [
            metrics_path,
            selected_configuration_path,
            predictions_path,
            confusion_matrix_path,
            classification_report_path,
            partition_manifest_path,
            candidate_leaderboard_path,
            final_history_path,
            final_weights_path,
        ]


        if not all(
            path.exists()
            for path
            in required_fold_outputs
        ):

            raise RuntimeError(
                "One or more required "
                "fold outputs are missing."
            )


        saved_predictions = (
            pd.read_csv(
                predictions_path
            )
        )


        if (
            len(
                saved_predictions
            )
            != 1120
        ):

            raise RuntimeError(
                "Saved prediction count "
                "is incorrect."
            )


        # ====================================================
        # FOLD COMPLETE MARKER — LAST
        # ====================================================

        fold_completion = {
            "status":
                "completed",

            "model":
                "ResNet50",

            "repeat":
                int(
                    repeat_number
                ),

            "fold":
                int(
                    fold_number
                ),

            "selected_configuration_id":
                selected_configuration_id,

            "macro_f1":
                float(
                    outer_metrics[
                        "macro_f1"
                    ]
                ),

            "balanced_accuracy":
                float(
                    outer_metrics[
                        "balanced_accuracy"
                    ]
                ),

            "outer_validation_used_once":
                True,

            "outer_validation_used_for_selection":
                False,

            "testing_images_used":
                False,

            "completed_at":
                datetime.now().isoformat(),
        }


        completion_path = (
            fold_directory
            / "fold_complete.json"
        )


        save_json_atomic(
            fold_completion,
            completion_path,
        )


        if not validate_completed_resnet50_fold(
            repeat_number,
            fold_number,
        ):

            raise RuntimeError(
                "Final fold persistence "
                "validation failed."
            )


        print(
            "\nFold completed successfully."
        )

        print(
            "Outer accuracy:",
            f"{outer_metrics['accuracy']:.6f}",
        )

        print(
            "Outer balanced accuracy:",
            f"{outer_metrics['balanced_accuracy']:.6f}",
        )

        print(
            "Outer macro F1:",
            f"{outer_metrics['macro_f1']:.6f}",
        )

        print(
            "Saved:",
            fold_directory,
        )


        # ====================================================
        # CLEAN GPU MEMORY
        # ====================================================

        del final_model
        del final_base_model
        del final_training_dataset
        del outer_validation_dataset
        del outer_probabilities
        del final_stage_1_history

        if (
            final_stage_2_history
            is not None
        ):

            del final_stage_2_history


        tf.keras.backend.clear_session()

        gc.collect()


    # ========================================================
    # 6. AGGREGATE FIVE FOLDS INTO ONE REPEAT RESULT
    # ========================================================

    print(
        "\nAggregating Repeat",
        repeat_number,
    )


    repeat_fold_records = []


    for fold_number in range(
        1,
        NUMBER_OF_OUTER_FOLDS + 1,
    ):

        if not validate_completed_resnet50_fold(
            repeat_number,
            fold_number,
        ):

            raise RuntimeError(
                f"Repeat {repeat_number} "
                f"cannot be aggregated because "
                f"Fold {fold_number} is incomplete."
            )


        fold_metrics_path = (
            repeat_directory
            / f"fold_{fold_number:02d}"
            / "outer_fold_metrics.json"
        )


        with fold_metrics_path.open(
            "r",
            encoding="utf-8",
        ) as file:

            repeat_fold_records.append(
                json.load(
                    file
                )
            )


    repeat_fold_metrics = (
        pd.DataFrame(
            repeat_fold_records
        )
        .sort_values(
            "fold"
        )
        .reset_index(
            drop=True
        )
    )


    repeat_fold_metrics_path = (
        repeat_directory
        / "repeat_fold_metrics.csv"
    )


    save_dataframe_atomic(
        repeat_fold_metrics,
        repeat_fold_metrics_path,
        index=False,
    )


    # --------------------------------------------------------
    # ONE statistical observation for this separation/repeat
    # --------------------------------------------------------

    repeat_summary = {
        "status":
            "completed",

        "model":
            "ResNet50",

        "repeat":
            int(
                repeat_number
            ),

        "completed_outer_folds":
            5,

        "mean_accuracy":
            float(
                repeat_fold_metrics[
                    "accuracy"
                ].mean()
            ),

        "sd_accuracy":
            float(
                repeat_fold_metrics[
                    "accuracy"
                ].std(
                    ddof=1
                )
            ),

        "mean_balanced_accuracy":
            float(
                repeat_fold_metrics[
                    "balanced_accuracy"
                ].mean()
            ),

        "sd_balanced_accuracy":
            float(
                repeat_fold_metrics[
                    "balanced_accuracy"
                ].std(
                    ddof=1
                )
            ),

        "mean_macro_precision":
            float(
                repeat_fold_metrics[
                    "macro_precision"
                ].mean()
            ),

        "sd_macro_precision":
            float(
                repeat_fold_metrics[
                    "macro_precision"
                ].std(
                    ddof=1
                )
            ),

        "mean_macro_recall":
            float(
                repeat_fold_metrics[
                    "macro_recall"
                ].mean()
            ),

        "sd_macro_recall":
            float(
                repeat_fold_metrics[
                    "macro_recall"
                ].std(
                    ddof=1
                )
            ),

        "mean_macro_f1":
            float(
                repeat_fold_metrics[
                    "macro_f1"
                ].mean()
            ),

        "sd_macro_f1":
            float(
                repeat_fold_metrics[
                    "macro_f1"
                ].std(
                    ddof=1
                )
            ),

        "mean_log_loss":
            float(
                repeat_fold_metrics[
                    "log_loss"
                ].mean()
            ),

        "sd_log_loss":
            float(
                repeat_fold_metrics[
                    "log_loss"
                ].std(
                    ddof=1
                )
            ),

        "statistical_unit":
            (
                "mean of five outer folds "
                "within this repeat"
            ),

        "wilcoxon_value":
            float(
                repeat_fold_metrics[
                    "macro_f1"
                ].mean()
            ),

        "testing_images_used":
            False,

        "completed_at":
            datetime.now().isoformat(),
    }


    repeat_summary_path = (
        repeat_directory
        / "repeat_summary.json"
    )


    save_json_atomic(
        repeat_summary,
        repeat_summary_path,
    )


    # Update master 10-repeat table immediately.
    update_resnet50_repeat_level_summary()


    print(
        "\n"
        + "-" * 80
    )

    print(
        f"REPEAT {repeat_number:02d} COMPLETED"
    )

    print(
        "-" * 80
    )

    print(
        "Mean five-fold accuracy:",
        f"{repeat_summary['mean_accuracy']:.6f}",
    )

    print(
        "Mean five-fold balanced accuracy:",
        f"{repeat_summary['mean_balanced_accuracy']:.6f}",
    )

    print(
        "Mean five-fold macro F1:",
        f"{repeat_summary['mean_macro_f1']:.6f}",
    )

    print(
        "This is ONE Wilcoxon observation."
    )


# ============================================================
# 7. FINAL 10-REPEAT VALIDATION
# ============================================================

update_resnet50_repeat_level_summary()


final_repeat_summary = pd.read_csv(
    REPEAT_LEVEL_SUMMARY_PATH
)


if len(
    final_repeat_summary
) != 10:

    raise RuntimeError(
        "Expected exactly 10 completed "
        "repeat-level results, but found "
        f"{len(final_repeat_summary)}."
    )


if (
    final_repeat_summary[
        "repeat"
    ].nunique()
    != 10
):

    raise RuntimeError(
        "Repeat-level summary does not "
        "contain 10 unique repeats."
    )


print(
    "\n"
    + "#" * 80
)

print(
    "RESNET50 REPEATED NESTED-CV COMPLETED"
)

print(
    "#" * 80
)

print(
    "Outer evaluations:",
    50,
)

print(
    "Completed repeats:",
    10,
)

print(
    "Wilcoxon observations:",
    len(
        final_repeat_summary
    ),
)

print(
    "\nRepeat-level statistical results:"
)

display(
    final_repeat_summary[
        [
            "repeat",
            "mean_accuracy",
            "mean_balanced_accuracy",
            "mean_macro_f1",
            "sd_macro_f1",
            "wilcoxon_value",
        ]
    ]
)

print(
    "\nSaved to:"
)

print(
    REPEAT_LEVEL_SUMMARY_PATH
)

print(
    "\nTesting images were never used."
)

print(
    "\nStep 13 full repeated ResNet50 experiment PASSED."
)


################################################################################
RESNET50 — REPEAT / SEPARATION 01 OF 10
################################################################################

REPEAT 01 / FOLD 01
Valid completed fold found — skipping.

REPEAT 01 / FOLD 02
Valid completed fold found — skipping.

REPEAT 01 / FOLD 03
Valid completed fold found — skipping.

REPEAT 01 / FOLD 04
Valid completed fold found — skipping.

REPEAT 01 / FOLD 05
Valid completed fold found — skipping.

Aggregating Repeat 1

--------------------------------------------------------------------------------
REPEAT 01 COMPLETED
--------------------------------------------------------------------------------
Mean five-fold accuracy: 0.953571
Mean five-fold balanced accuracy: 0.953571
Mean five-fold macro F1: 0.953605
This is ONE Wilcoxon observation.

################################################################################
RESNET50 — REPEAT / SEPARATION 02 OF 10
#########################

,repeat,mean_accuracy,mean_balanced_accuracy,mean_macro_f1,sd_macro_f1,wilcoxon_value
0,1,0.953571,0.953571,0.953605,0.014229,0.953605
1,2,0.957321,0.957321,0.957350,0.009010,0.957350
2,3,0.956071,0.956071,0.956216,0.001722,0.956216
3,4,0.963393,0.963393,0.963404,0.005033,0.963404
4,5,0.957857,0.957857,0.957928,0.008288,0.957928
5,6,0.953571,0.953571,0.953590,0.014032,0.953590
6,7,0.954286,0.954286,0.954118,0.012524,0.954118
7,8,0.949821,0.949821,0.949517,0.015862,0.949517
8,9,0.941429,0.941429,0.941120,0.016231,0.941120
9,10,0.955536,0.955536,0.955493,0.009092,0.955493



Saved to:
/content/drive/MyDrive/brain_tumour_colab/results/repeated_nested_cv/resnet50/repeat_level_summary.csv

Testing images were never used.

Step 13 full repeated ResNet50 experiment PASSED.


In [ ]:
# ============================================================
# RESNET50 EXPERIMENT SETUP AND REPRODUCIBILITY
# ============================================================

import gc
import json
import os
import random
import sys
import time
from pathlib import Path
import numpy as np
import pandas as pd
import tensorflow as tf


# ============================================================
# 1. Reproducibility
# ============================================================

RANDOM_SEED = 42

# Set Python's hash seed
os.environ["PYTHONHASHSEED"] = str(RANDOM_SEED)

# Set Python, NumPy, and TensorFlow random seeds
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
tf.keras.utils.set_random_seed(RANDOM_SEED)


# Request deterministic TensorFlow operations where supported
try:

    tf.config.experimental.enable_op_determinism()
    determinism_status = "enabled"

except Exception as error:
    determinism_status = ("requested, but TensorFlow returned: "f"{error}")


# Keep TensorFlow/Keras calculations in float32
tf.keras.backend.set_floatx("float32")


# ============================================================
# 2. Project paths
# ============================================================

# PROJECT_ROOT was created and verified in the previous project-restoration cell.

DATA_DIR = PROJECT_ROOT / "processed_data_cropped"
FOLDS_FILE = PROJECT_ROOT / "splits" / "five_fold_cross_validation.csv"


# Create the persistent Google Drive results directory
DRIVE_ROOT = Path("/content/drive/MyDrive/brain_tumour_colab")


ARCHITECTURE_RESULTS_DIR = DRIVE_ROOT / "results" / "resnet50_transfer_learning"


# Store the complete ResNet50 hyperparameter-search results here
CV_RESULTS_DIR = ARCHITECTURE_RESULTS_DIR / "resnet50_hyperparameter_search_no_augmentation_v2"


# Store the selected final ResNet50 model here
FINAL_RESULTS_DIR = ARCHITECTURE_RESULTS_DIR / "resnet50_selected_final_no_augmentation_v2"


CV_RESULTS_DIR.mkdir(parents=True, exist_ok=True)

FINAL_RESULTS_DIR.mkdir(parents=True, exist_ok=True)


# ============================================================
# 3. Fixed experiment configuration
# ============================================================

IMAGE_HEIGHT = 224
IMAGE_WIDTH = 224
IMAGE_CHANNELS = 3

NUMBER_OF_CLASSES = 4


CLASS_NAMES = ["glioma", "meningioma", "notumor", "pituitary"]


# Create mappings between class names and their numerical class indices
CLASS_TO_INDEX = {class_name: index for index, class_name in enumerate(CLASS_NAMES)}
INDEX_TO_CLASS = {index: class_name for class_name, index in CLASS_TO_INDEX.items()}


# ============================================================
# 4. Verify experiment setup
# ============================================================

print("=" * 70)
print("RESNET50 EXPERIMENT SETUP")
print("=" * 70)
print("Python version:", sys.version)
print("TensorFlow version:", tf.__version__)
print("Keras version:", tf.keras.__version__)
print("\n--- Reproducibility ---")
print("Random seed:", RANDOM_SEED)
print("Deterministic TensorFlow operations:", determinism_status)
print("Keras float type:", tf.keras.backend.floatx())
print("\n--- Project paths ---")
print("Project root:", PROJECT_ROOT)
print("Dataset folder:", DATA_DIR)
print("Fold file:", FOLDS_FILE)
print("Hyperparameter-search results:", CV_RESULTS_DIR)
print("Selected final-model results:", FINAL_RESULTS_DIR)
print("\n--- Dataset configuration ---")
print("Input shape:", (IMAGE_HEIGHT, IMAGE_WIDTH, IMAGE_CHANNELS))
print("Number of classes:", NUMBER_OF_CLASSES)
print("Class mapping:", CLASS_TO_INDEX)
print("Data augmentation:", False)


# ============================================================
# 5. Safety checks
# ============================================================

if not DATA_DIR.exists():
    raise FileNotFoundError(f"Dataset directory not found: "f"{DATA_DIR}")


if not FOLDS_FILE.exists():
    raise FileNotFoundError(f"Five-fold cross-validation file not found: "f"{FOLDS_FILE}")


if not tf.config.list_physical_devices("GPU"):
    raise RuntimeError("No GPU was detected. Enable a GPU runtime before running the experiment.")

print("\nResNet50 experiment setup verified successfully.")

RESNET50 EXPERIMENT SETUP
Python version: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
TensorFlow version: 2.20.0
Keras version: 3.13.2

--- Reproducibility ---
Random seed: 42
Deterministic TensorFlow operations: enabled
Keras float type: float32

--- Project paths ---
Project root: /content/brain-tumour-mri-classification
Dataset folder: /content/brain-tumour-mri-classification/processed_data_cropped
Fold file: /content/brain-tumour-mri-classification/splits/five_fold_cross_validation.csv
Hyperparameter-search results: /content/drive/MyDrive/brain_tumour_colab/results/resnet50_transfer_learning/resnet50_hyperparameter_search_no_augmentation_v2
Selected final-model results: /content/drive/MyDrive/brain_tumour_colab/results/resnet50_transfer_learning/resnet50_selected_final_no_augmentation_v2

--- Dataset configuration ---
Input shape: (224, 224, 3)
Number of classes: 4
Class mapping: {'glioma': 0, 'meningioma': 1, 'notumor': 2, 'pituitary': 3}
Data augmentation: False

ResNet50 

In [ ]:
# ============================================================
# LOAD AND VALIDATE THE FIXED FIVE-FOLD ASSIGNMENTS
# ============================================================

# Load the fixed five-fold Training assignments
assignments = pd.read_csv(FOLDS_FILE)


# ============================================================
# 1. Validate the table structure
# ============================================================

# Define the columns that must exist in the fold-assignment file
required_columns = {"relative_path", "class", "fold",}


# Find any required columns that are missing
missing_columns = required_columns - set(assignments.columns)


# Stop if any required column is missing
if missing_columns:
    raise ValueError("The fold file is missing columns: "f"{sorted(missing_columns)}")


# ============================================================
# 2. Create class indices and complete image paths
# ============================================================

# Assign the fixed numeric class index to every Training image
assignments["class_index"] = assignments["class"].map(CLASS_TO_INDEX)


# Create the complete path to every Training image
assignments["image_path"] = assignments["relative_path"].apply(lambda path:DATA_DIR / Path(path))



# ============================================================
# 3. Validate paths and assignments
# ============================================================

# Find any Training images that do not exist
missing_image_paths = [image_path for image_path in assignments["image_path"] if not image_path.exists()]


# Find any class names that do not belong to the four expected classes
unexpected_classes = sorted(set(assignments["class"]) - set(CLASS_NAMES))


# Find any fold values outside the fixed folds 1 to 5
unexpected_folds = sorted(set(assignments["fold"]) - {1, 2, 3, 4, 5})


# Count duplicate Training image paths
duplicate_paths = assignments["relative_path"].duplicated().sum()


# Count missing values in the required columns
missing_values = assignments[["relative_path", "class", "fold"]].isna().sum()


# Check whether any class failed to receive a numeric class index
missing_class_indices = assignments["class_index"].isna().sum()


# ============================================================
# 4. Verify the fixed fold distribution
# ============================================================

# Count how many images from every class belong to each outer fold
fold_distribution = pd.crosstab(assignments["fold"], assignments["class"])


# Put the class columns into the fixed project class order
fold_distribution = fold_distribution.reindex(columns=CLASS_NAMES, fill_value=0)


# Add the total number of images in each fold
fold_distribution["total"] = (fold_distribution.sum(axis=1))


# Each of the five fixed folds must contain 280 images from every class
expected_fold_distribution = pd.DataFrame(
    {
        class_name: [280] * 5
        for class_name in CLASS_NAMES
    },
    index=[1, 2, 3, 4, 5]
)

expected_fold_distribution["total"] = 1120


# ============================================================
# 5. Display the validation results
# ============================================================

print("=" * 70)
print("FIXED FIVE-FOLD ASSIGNMENT VALIDATION")
print("=" * 70)
print("Rows:", len(assignments))
print("Columns:", assignments.columns.tolist())
print("\n--- Assignment checks ---")
print("Duplicate relative paths:", duplicate_paths)
print("Missing image files:", len(missing_image_paths))
print("Unexpected classes:", unexpected_classes)
print("Unexpected fold values:", unexpected_folds)
print("Missing class indices:", missing_class_indices)
print("\nMissing values:")
print(missing_values)
print("\n--- Images per fold and class ---")
display(fold_distribution)
print("\n--- Sample records ---")
display(assignments[["relative_path", "class", "class_index", "fold", "image_path"]].head())


# ============================================================
# 6. Stop if any validation check failed
# ============================================================

# The fold file must contain exactly 5,600 Training images
if len(assignments) != 5600:
    raise ValueError("Expected 5,600 Training images, "f"but found {len(assignments)}.")


# Every Training image path must be unique
if duplicate_paths != 0:
    raise ValueError("Duplicate paths were found in the fold file.")


# Every Training image listed in the CSV must exist
if missing_image_paths:
    raise FileNotFoundError(f"{len(missing_image_paths)} Training image files are missing.")


# Only the expected four classes are allowed
if unexpected_classes:
    raise ValueError(f"Unexpected classes found: "f"{unexpected_classes}")


# Only folds 1 to 5 are allowed
if unexpected_folds:
    raise ValueError(f"Unexpected fold values found: "f"{unexpected_folds}")


# Required columns must not contain missing values
if missing_values.sum() != 0:
    raise ValueError("Missing values were found in the fold file.")


# Every class must map successfully to its fixed numeric index
if missing_class_indices != 0:
    raise ValueError("One or more Training classes could not be mapped to a class index.")


# Verify the exact fixed five-fold class distribution
if not fold_distribution.equals(expected_fold_distribution):
    raise ValueError("The fixed five-fold class distribution does not match the expected "
        "280 images per class and 1,120 images per fold.")


print("\nFixed five-fold assignment validation passed.")

FIXED FIVE-FOLD ASSIGNMENT VALIDATION
Rows: 5600
Columns: ['relative_path', 'class', 'fold', 'class_index', 'image_path']

--- Assignment checks ---
Duplicate relative paths: 0
Missing image files: 0
Unexpected classes: []
Unexpected fold values: []
Missing class indices: 0

Missing values:
relative_path    0
class            0
fold             0
dtype: int64

--- Images per fold and class ---


class,glioma,meningioma,notumor,pituitary,total
fold,,,,,
1,280,280,280,280,1120
2,280,280,280,280,1120
3,280,280,280,280,1120
4,280,280,280,280,1120
5,280,280,280,280,1120



--- Sample records ---


,relative_path,class,class_index,fold,image_path
0,Training/glioma/Tr-gl_100.png,glioma,0,1,/content/brain-tumour-mri-classification/proce...
1,Training/glioma/Tr-gl_1001.png,glioma,0,1,/content/brain-tumour-mri-classification/proce...
2,Training/glioma/Tr-gl_1003.png,glioma,0,1,/content/brain-tumour-mri-classification/proce...
3,Training/glioma/Tr-gl_1014.png,glioma,0,1,/content/brain-tumour-mri-classification/proce...
4,Training/glioma/Tr-gl_1015.png,glioma,0,1,/content/brain-tumour-mri-classification/proce...



Fixed five-fold assignment validation passed.


In [ ]:

# ============================================================
# DEFINE THE RESNET50 IMAGE PREPROCESSING FUNCTION
# ============================================================

def load_and_preprocess_resnet50_image(image_path, class_index):
    """
    Load one stored grayscale PNG and prepare it for ResNet50.
    """

    # Read the image file from its path
    image_bytes = tf.io.read_file(image_path)


    # Decode the PNG image as a single-channel grayscale image
    grayscale_image = tf.io.decode_png(image_bytes, channels=1)


    # Confirm the stored image has the expected height,
    # width, and one grayscale channel
    grayscale_image = tf.ensure_shape(image, (IMAGE_HEIGHT, IMAGE_WIDTH, 1))


    # Convert the grayscale image into a three-channel
    # pseudo-RGB image
    pseudo_rgb_image = tf.image.grayscale_to_rgb(image)


    # Convert the image pixel values to float32
    resnet50_image = tf.cast(image, tf.float32)


    # Apply the preprocessing required by ImageNet ResNet50
    resenet50_image = tf.keras.applications.resnet50.preprocess_input(image)


    # Confirm the final ResNet50 input shape
    resnet50_image = tf.ensure_shape(image, (IMAGE_HEIGHT, IMAGE_WIDTH, IMAGE_CHANNELS))


    # Ensure the class label uses int32
    class_index = tf.cast(class_index, tf.int32)

    return resnet50_image, class_index


# ============================================================
# DEFINE THE RESNET50 TENSORFLOW DATASET FUNCTION
# ============================================================

def make_dataset(dataframe, training, seed, batch_size):
    """
    Create a deterministic TensorFlow dataset from image paths
    and numeric class labels.
    """

    # Convert all image paths into strings that TensorFlow can read
    image_paths = (dataframe["image_path"].astype(str).to_numpy())


    # Convert all class indices into int32 values
    class_indices = (dataframe["class_index"].astype(np.int32).to_numpy())


    # Create a TensorFlow dataset containing image paths and labels
    dataset = tf.data.Dataset.from_tensor_slices((image_paths, class_indices))


    # Create deterministic dataset options
    dataset_options = tf.data.Options()

    dataset_options.experimental_deterministic = True

    dataset = dataset.with_options(dataset_options)


    # Shuffle model-training datasets only
    if training:
        dataset = dataset.shuffle(
            buffer_size=len(dataframe),
            seed=seed,
            reshuffle_each_iteration=True,
        )


    # Load and preprocess every image for ResNet50
    dataset = dataset.map(
        load_and_preprocess_resnet50_image,
        num_parallel_calls=tf.data.AUTOTUNE,
        deterministic=True)


    # Group images and labels into batches
    dataset = dataset.batch(batch_size, drop_remainder=False,)


    # Prepare upcoming batches while the model processes
    # the current batch
    dataset = dataset.prefetch(tf.data.AUTOTUNE)


    return dataset

In [ ]:
# ============================================================
# DEFINE THE RESNET50 TRANSFER-LEARNING MODEL
# ============================================================

# Set the fixed dropout rate used in the classification head
DROPOUT_RATE = 0.30


def build_model():
    """
    Create a ResNet50 transfer-learning model with an ImageNet-pretrained convolutional base and a new
    four-class classification head.
    """

    # Create the model input for 224 x 224 pseudo-RGB images
    inputs = tf.keras.Input(shape=(IMAGE_HEIGHT, IMAGE_WIDTH, IMAGE_CHANNELS), name="input_image")


    # Load the ImageNet-pretrained ResNet50 convolutional base
    base_model = tf.keras.applications.ResNet50(
        include_top=False,
        weights="imagenet",
        input_shape=(IMAGE_HEIGHT, IMAGE_WIDTH, IMAGE_CHANNELS),
        pooling="avg")


    # Freeze the complete pretrained ResNet50 base for Stage 1
    base_model.trainable = False


    # Run the frozen pretrained base in inference mode
    features = base_model(inputs, training=False)


    # Apply dropout before the final classification layer
    features = tf.keras.layers.Dropout(rate=DROPOUT_RATE, name="classifier_dropout")(features)


    # Create the four-class softmax classification layer
    outputs = tf.keras.layers.Dense(
        units=NUMBER_OF_CLASSES,
        activation="softmax",
        dtype="float32",
        name="class_probabilities",
    )(
        features
    )


    # Create the complete transfer-learning model
    model = tf.keras.Model(inputs=inputs, outputs=outputs, name="resnet50_transfer_learning")


    return model, base_model

In [ ]:
# ============================================================
# DEFINE RESNET50 CROSS-VALIDATION UTILITY FUNCTIONS
# ============================================================

from sklearn.metrics import (accuracy_score, balanced_accuracy_score, classification_report, confusion_matrix,
    log_loss, matthews_corrcoef, precision_recall_fscore_support)

from sklearn.model_selection import StratifiedShuffleSplit)


# ============================================================
# 1. Fixed cross-validation settings
# ============================================================

OUTER_FOLDS = [1, 2, 3, 4, 5]

INNER_VALIDATION_FRACTION = 0.10

FINE_TUNE_FROM_LAYER = ("conv5_block1_1_conv")

EARLY_STOPPING_PATIENCE = 3


# ============================================================
# 2. Create the training, internal-validation,
#    and outer-validation partitions
# ============================================================

def make_partitions(outer_fold,):
    """
    Create three non-overlapping partitions:

    1. Model-training data
    2. Internal-validation data
    3. Untouched outer-validation data
    """

    # Create the untouched outer-validation partition
    outer_validation = (assignments[assignments["fold"] == outer_fold]
        .copy()
        .reset_index(drop=True))


    # Create the complete outer-training partition
    outer_training = (assignments[assignments["fold"] != outer_fold]
        .copy()
        .reset_index(drop=True))


    # Create a deterministic stratified 10% internal
    # validation split from the outer-training partition
    splitter = StratifiedShuffleSplit(
        n_splits=1,
        test_size=(INNER_VALIDATION_FRACTION),
        random_state=(RANDOM_SEED + outer_fold))


    training_positions, internal_validation_positions = next(
        splitter.split(outer_training, outer_training["class_index"]))


    # Create the model-training partition
    model_training = outer_training.iloc[training_positions].copy().reset_index(drop=True)



    # Create the internal-validation partition
    internal_validation = outer_training.iloc[internal_validation_positions].copy().reset_index(drop=True)



    # Label each partition for the saved manifest
    model_training["partition"] = "model_training"
    internal_validation["partition"] = "internal_validation"
    outer_validation["partition"] = "outer_validation"



    # Combine the three partitions into one manifest
    manifest = pd.concat(
        [model_training, internal_validation, outer_validation], ignore_index=True)


    # Verify the expected partition sizes
    expected_sizes = (4032, 448, 1120)
    actual_sizes = (len(model_training), len(internal_validation), len(outer_validation))


    if actual_sizes != expected_sizes:
        raise RuntimeError("Unexpected partition sizes. "f"Found {actual_sizes}; "f"expected {expected_sizes}.")


    # Verify that no image appears in more than one partition
    if manifest["relative_path"].duplicated().any():
        raise RuntimeError("The model-training, internal-validation, and outer-validation partitions overlap.")

    return model_training, internal_validation, outer_validation, manifest


# ============================================================
# 3. Compile the ResNet50 model
# ============================================================

def compile_model(model, learning_rate):
    """
    Compile the ResNet50 transfer-learning model.
    """

    model.compile(
        optimizer=(tf.keras.optimizers.Adam(learning_rate=(learning_rate))),
        loss=(tf.keras.losses.SparseCategoricalCrossentropy()),
        metrics=[tf.keras.metrics.SparseCategoricalAccuracy(name="accuracy")])


# ============================================================
# 4. Enable ResNet50 fine-tuning
# ============================================================

def enable_fine_tuning(base_model):
    """
    Unfreeze the selected upper ResNet50 layers while
    keeping Batch Normalization layers frozen.
    """

    # Allow selected ResNet50 layers to become trainable
    base_model.trainable = True
    start_found = False


    # Freeze every layer before the selected fine-tuning point
    for layer in base_model.layers:
        if layer.name == FINE_TUNE_FROM_LAYER:
            start_found = True

        # Keep Batch Normalization layers frozen
        layer.trainable = (start_found and not isinstance(layer,tf.keras.layers.BatchNormalization))


    # Stop if the requested fine-tuning layer does not exist
    if not start_found:
        raise ValueError("Fine-tuning layer was not found: "f"{FINE_TUNE_FROM_LAYER}")


    # Record the ResNet50 layers enabled for fine-tuning
    trainable_layers = [layer.name for layer in base_model.layers if layer.trainable]


    if not trainable_layers:
        raise RuntimeError("No ResNet50 layers were enabled for fine-tuning.")


    return trainable_layers


# ============================================================
# 5. Create training callbacks
# ============================================================

def create_callbacks(checkpoint_path, minimum_learning_rate):
    """
    Create the callbacks used during both transfer-learning stages.
    """

    return [
        tf.keras.callbacks.ModelCheckpoint(
            filepath=checkpoint_path,
            monitor="val_loss",
            mode="min",
            save_best_only=True,
            save_weights_only=True,
            verbose=1,
        ),

        tf.keras.callbacks.EarlyStopping(
            monitor="val_loss",
            mode="min",
            patience=(EARLY_STOPPING_PATIENCE),
            restore_best_weights=True,
            verbose=1,
        ),

        tf.keras.callbacks.ReduceLROnPlateau(
            monitor="val_loss",
            mode="min",
            factor=0.2,
            patience=2,
            min_lr=minimum_learning_rate,
            verbose=1,
        ),

        tf.keras.callbacks.TerminateOnNaN(),
    ]


# ============================================================
# 6. Calculate classification metrics
# ============================================================

def calculate_metrics(y_true, y_predicted, probabilities):
    """
    Calculate the classification metrics used to evaluate an untouched outer-validation fold.
    """

    # Calculate macro-averaged precision, recall, and F1
    macro_scores = (
        precision_recall_fscore_support(y_true, y_predicted, average="macro", zero_division=0))


    # Calculate weighted precision, recall, and F1
    weighted_scores = (
        precision_recall_fscore_support(y_true, y_predicted, average="weighted", zero_division=0))


    return {
        "accuracy": float(accuracy_score(y_true, y_predicted)),
        "balanced_accuracy": float(balanced_accuracy_score(y_true, y_predicted)),
        "macro_precision": float(macro_scores[0]),
        "macro_recall": float(macro_scores[1]),
        "macro_f1": float(macro_scores[2]),
        "weighted_precision": float(weighted_scores[0]),
        "weighted_recall": float(weighted_scores[1]),
        "weighted_f1": float(weighted_scores[2]),
        "matthews_correlation_coefficient": (float(matthews_corrcoef(y_true, y_predicted))),
        "log_loss": float(log_loss(y_true, probabilities, labels=list(range(NUMBER_OF_CLASSES)))),
    }


print("ResNet50 cross-validation utility functions defined successfully.")

ResNet50 cross-validation utility functions defined successfully.


In [ ]:
# ============================================================
# RESNET50 HYPERPARAMETER SEARCH WITH FIVE-FOLD
# CROSS-VALIDATION
# ============================================================
#
# Primary controlled experiment:
#   - No image augmentation
#   - Fixed five outer folds
#   - Deterministic stratified 10% internal-validation split
#     inside each outer-training partition
#   - Outer-validation folds are not used for early stopping
#     or checkpoint selection
#   - Held-out Testing images are not loaded
#
# Search:
#   Batch size:
#       8, 16, 32
#
#   Stage 1 learning rate:
#       1e-3, 3e-4
#
#   Stage 2 learning rate:
#       1e-5, 3e-6
#
#   12 configurations × 5 folds = 60 runs
#
# Configuration selection:
#   1. Highest mean five-fold macro F1
#   2. Highest mean five-fold balanced accuracy
#   3. Lowest five-fold macro-F1 standard deviation
#   4. Lowest mean five-fold log loss
#
# Completed folds are skipped safely when this cell is rerun.
# Incomplete folds are deleted and rerun from the beginning.
# ============================================================

import gc
import json
import shutil
import time
from datetime import datetime
from pathlib import Path
import numpy as np
import pandas as pd
import tensorflow as tf


# ============================================================
# 1. Hyperparameter-search configuration
# ============================================================

BATCH_SIZES = [8, 16, 32]
HEAD_LEARNING_RATES = [1e-3, 3e-4]
FINE_TUNE_LEARNING_RATES = [1e-5, 3e-6]
HEAD_MAX_EPOCHS = 15
FINE_TUNE_MAX_EPOCHS = 20
HEAD_MINIMUM_LEARNING_RATE = 1e-6
FINE_TUNE_MINIMUM_LEARNING_RATE = 1e-7
NUMBER_OF_CONFIGURATIONS = (len(BATCH_SIZES) * len(HEAD_LEARNING_RATES) * len(FINE_TUNE_LEARNING_RATES))
NUMBER_OF_FOLD_RUNS = (NUMBER_OF_CONFIGURATIONS * len(OUTER_FOLDS))


# ============================================================
# 2. Search output paths
# ============================================================

SEARCH_CONFIGURATION_PATH = CV_RESULTS_DIR / "search_configuration.json"
ALL_FOLD_METRICS_PATH = CV_RESULTS_DIR / "all_fold_metrics.csv"
LEADERBOARD_PATH = CV_RESULTS_DIR / "hyperparameter_leaderboard.csv"
SELECTED_CONFIGURATION_PATH = CV_RESULTS_DIR / "selected_configuration.json"
SEARCH_COMPLETION_PATH = CV_RESULTS_DIR / "search_completion.json"
CV_RESULTS_DIR.mkdir(parents=True, exist_ok=True)


# ============================================================
# 3. Utility functions used by the search
# ============================================================

def save_json_atomic(data, destination):
    """
    Save JSON through a temporary file before replacing the destination.
    """

    destination = Path(destination)
    temporary_path = destination.with_name(destination.name + ".tmp")

    with temporary_path.open("w", encoding="utf-8") as file:
        json.dump(data, file, indent=4, default=str)

    temporary_path.replace(destination)


def get_best_epoch_and_loss(history):
    """
    Return the epoch and validation loss corresponding to the lowest internal-validation loss.
    """

    validation_losses = np.asarray(history.history["val_loss"], dtype=np.float64)

    if validation_losses.size == 0:
        raise RuntimeError("No internal-validation losses were recorded.")

    if not np.isfinite(validation_losses).all():
        raise RuntimeError("A non-finite internal-validation loss was recorded.")

    best_position = int(np.argmin(validation_losses))
    best_epoch = best_position + 1
    best_loss = float(validation_losses[best_position])


    return best_epoch, best_loss


# ============================================================
# 4. Save the complete search definition
# ============================================================

SEARCH_CONFIGURATION = {
    "experiment_name": (
        "resnet50_hyperparameter_search_"
        "no_augmentation_v2"
    ),
    "architecture": "ResNet50",
    "pretrained_weights": "ImageNet",
    "input_shape": [IMAGE_HEIGHT, IMAGE_WIDTH, IMAGE_CHANNELS],
    "number_of_classes": NUMBER_OF_CLASSES,
    "class_names": CLASS_NAMES,
    "class_to_index": CLASS_TO_INDEX,
    "outer_folds": OUTER_FOLDS,
    "inner_validation_fraction": INNER_VALIDATION_FRACTION,
    "batch_sizes": BATCH_SIZES,
    "head_learning_rates": HEAD_LEARNING_RATES,
    "fine_tune_learning_rates": FINE_TUNE_LEARNING_RATES,
    "head_max_epochs": HEAD_MAX_EPOCHS,
    "fine_tune_max_epochs": FINE_TUNE_MAX_EPOCHS,
    "dropout_rate": DROPOUT_RATE,
    "fine_tune_from_layer": FINE_TUNE_FROM_LAYER,
    "batch_normalisation_frozen": True,
    "backbone_called_with_training_false": True,
    "early_stopping_patience": EARLY_STOPPING_PATIENCE,
    "reduce_lr_factor": 0.2,
    "reduce_lr_patience": 2,
    "data_augmentation": False,
    "testing_images_loaded": False,
    "random_seed": RANDOM_SEED,
    "number_of_configurations": NUMBER_OF_CONFIGURATIONS,
    "number_of_fold_runs": NUMBER_OF_FOLD_RUNS,
    "selection_rule": [
        "highest mean five-fold macro F1",
        "highest mean five-fold balanced accuracy",
        "lowest five-fold macro-F1 standard deviation",
        "lowest mean five-fold log loss"],
    "tensorflow_version": tf.__version__,
    "keras_version": tf.keras.__version__,
}

save_json_atomic(SEARCH_CONFIGURATION, SEARCH_CONFIGURATION_PATH)


# ============================================================
# 5. Verify the search environment
# ============================================================

gpu_devices = tf.config.list_physical_devices("GPU")



if not gpu_devices:
    raise RuntimeError(
        "No TensorFlow GPU was detected. Enable a GPU runtime before starting "
        "the ResNet50 search.")


if len(assignments) != 5600:
    raise RuntimeError("Expected 5,600 validated Training assignments, "f"but found {len(assignments)}.")


print("=" * 70)
print("RESNET50 HYPERPARAMETER SEARCH")
print("=" * 70)
print("GPU devices:", gpu_devices)
print("Results directory:", CV_RESULTS_DIR)
print("Configurations:", NUMBER_OF_CONFIGURATIONS)
print("Outer folds:", len(OUTER_FOLDS))
print("Total fold runs:", NUMBER_OF_FOLD_RUNS)
print("\n--- Search space ---")
print("Batch sizes:", BATCH_SIZES)
print("Head learning rates:", HEAD_LEARNING_RATES)
print("Fine-tuning learning rates:", FINE_TUNE_LEARNING_RATES)
print("Maximum head-training epochs:", HEAD_MAX_EPOCHS)
print("Maximum fine-tuning epochs:", FINE_TUNE_MAX_EPOCHS)
print("\nData augmentation:", False)
print("Testing images loaded:", False)


# ============================================================
# 6. Build the 12 fixed hyperparameter configurations
# ============================================================

search_configurations = []
configuration_number = 0

for batch_size in BATCH_SIZES:
    for head_learning_rate in HEAD_LEARNING_RATES:
        for fine_tune_learning_rate in FINE_TUNE_LEARNING_RATES:
            configuration_number += 1
            configuration_id = (
                f"resnet50_bs{batch_size}"
                f"_headlr"
                f"{head_learning_rate:.0e}"
                f"_ftlr"
                f"{fine_tune_learning_rate:.0e}")


            search_configurations.append(
                {
                    "configuration_number": configuration_number,
                    "configuration_id": configuration_id,
                    "batch_size": batch_size,
                    "head_learning_rate": head_learning_rate,
                    "fine_tune_learning_rate": fine_tune_learning_rate
                    })


configuration_table = pd.DataFrame(search_configurations)

print("\n--- Configurations ---")
display(configuration_table)


# ============================================================
# 7. Run every configuration across all five outer folds
# ============================================================

search_start_time = (time.perf_counter())

for configuration in search_configurations:

    configuration_number = configuration["configuration_number"]
    configuration_id = configuration["configuration_id"]
    batch_size = configuration["batch_size"]
    head_learning_rate = configuration["head_learning_rate"]
    fine_tune_learning_rate = configuration["fine_tune_learning_rate"]

    configuration_directory = CV_RESULTS_DIR / configuration_id
    configuration_directory.mkdir(parents=True, exist_ok=True)

    configuration_information = {
        **configuration,
        "head_max_epochs": HEAD_MAX_EPOCHS,
        "fine_tune_max_epochs": FINE_TUNE_MAX_EPOCHS,
        "dropout_rate": DROPOUT_RATE,
        "fine_tune_from_layer": FINE_TUNE_FROM_LAYER,
        "data_augmentation": False,
        "testing_images_loaded": False,
    }


    save_json_atomic(configuration_information, configuration_directory / "configuration.json")

    print("\n" + "=" * 70)
    print(f"CONFIGURATION {configuration_number} OF {NUMBER_OF_CONFIGURATIONS}")
    print("=" * 70)
    print("Configuration:", configuration_id)
    print("Batch size:", batch_size)
    print("Head learning rate:", head_learning_rate)
    print("Fine-tuning learning rate:", fine_tune_learning_rate)


    for outer_fold in OUTER_FOLDS:

        fold_directory = configuration_directory / f"fold_{outer_fold}"
        completion_path = fold_directory / "fold_complete.json"
        fold_metrics_path = fold_directory / "fold_metrics.json"
        predictions_path = fold_directory / "outer_validation_predictions.csv"
        partition_manifest_path = fold_directory / "partition_manifest.csv"
        training_history_path = fold_directory / "training_history.csv"
        confusion_matrix_path = fold_directory / "confusion_matrix.csv"
        classification_report_path = fold_directory / "classification_report.csv"

        required_fold_outputs = [
            fold_metrics_path,
            predictions_path,
            partition_manifest_path,
            training_history_path,
            confusion_matrix_path,
            classification_report_path,
        ]


        # ----------------------------------------------------
        # Resume logic
        # ----------------------------------------------------

        fold_is_complete = (
            completion_path.exists()
            and all(path.exists() for path in required_fold_outputs))


        if fold_is_complete:
            print(f"\nConfiguration {configuration_number}/{NUMBER_OF_CONFIGURATIONS}, Fold {outer_fold}: ""already completed — skipping.")
            continue


        # If a completion marker or any other partial output
        # exists without the complete required output set,
        # rerun the fold cleanly.
        if fold_directory.exists():
            print(f"\nRemoving incomplete output for {configuration_id}, Fold {outer_fold}...")
            shutil.rmtree(fold_directory)


        fold_directory.mkdir(parents=True, exist_ok=True)

        print("\n" + "-" * 70)
        print(f"CONFIGURATION {configuration_number}/{NUMBER_OF_CONFIGURATIONS} — OUTER FOLD {outer_fold}/5")
        print("-" * 70)

        fold_start_time = time.perf_counter()


        # Use the same deterministic seed for every
        # configuration evaluated on the same outer fold.
        fold_seed = RANDOM_SEED + outer_fold

        tf.keras.backend.clear_session()

        tf.keras.utils.set_random_seed(fold_seed)

        gc.collect()


        # ----------------------------------------------------
        # Create the fixed nested partitions
        # ----------------------------------------------------

        (
            model_training_dataframe,
            internal_validation_dataframe,
            outer_validation_dataframe,
            partition_manifest,
        ) = make_partitions(outer_fold)


        partition_manifest[["relative_path", "class", "class_index", "fold", "partition"]].to_csv(
            partition_manifest_path, index=False)

        print("Model-training images:", len(model_training_dataframe))
        print("Internal-validation images:", len(internal_validation_dataframe))
        print("Outer-validation images:", len(outer_validation_dataframe))


        # ----------------------------------------------------
        # Create the TensorFlow datasets
        # ----------------------------------------------------

        training_dataset = make_dataset(model_training_dataframe, training=True, seed=fold_seed, batch_size=batch_size)

        internal_validation_dataset = make_dataset(internal_validation_dataframe, training=False, seed=fold_seed, batch_size=batch_size)

        outer_validation_dataset = make_dataset(outer_validation_dataframe, training=False, seed=fold_seed, batch_size=batch_size)



        # ----------------------------------------------------
        # Build a fresh ImageNet ResNet50
        # ----------------------------------------------------
        model, base_model = build_model()


        # ====================================================
        # STAGE 1 — TRAIN THE CLASSIFIER HEAD
        # ====================================================

        stage_1_checkpoint = fold_directory / "stage_1_best.weights.h5"

        compile_model(model, head_learning_rate)

        print("\nStage 1 of 2 — classifier-head training")
        print("Maximum epochs:", HEAD_MAX_EPOCHS)
        print("Learning rate:", head_learning_rate)

        stage_1_start_time = time.perf_counter()

        stage_1_history = model.fit(training_dataset, validation_data=(internal_validation_dataset),
            epochs=HEAD_MAX_EPOCHS,
            callbacks=create_callbacks(str(stage_1_checkpoint), minimum_learning_rate=(HEAD_MINIMUM_LEARNING_RATE)),
            verbose=1)


        stage_1_training_seconds = time.perf_counter() - stage_1_start_time

        stage_1_best_epoch, stage_1_best_validation_loss = get_best_epoch_and_loss(stage_1_history)

        if not stage_1_checkpoint.exists():
            raise RuntimeError("Stage 1 checkpoint was not saved.")


        # Explicitly restore the best Stage 1 checkpoint
        # before starting fine-tuning.
        model.load_weights(stage_1_checkpoint)

        print("Stage 1 best epoch:", stage_1_best_epoch)
        print("Stage 1 best internal-validation loss:", stage_1_best_validation_loss)


        # ====================================================
        # STAGE 2 — FINE-TUNE THE UPPER RESNET50 STAGE
        # ====================================================

        trainable_base_layers = enable_fine_tuning(base_model)

        stage_2_checkpoint = fold_directory / "stage_2_best.weights.h5"

        # Recompile after changing layer trainability.
        compile_model(model, fine_tune_learning_rate)

        print("\nStage 2 of 2 — ResNet50 fine-tuning")
        print("Fine-tuning starts from:", FINE_TUNE_FROM_LAYER)
        print("Trainable backbone layers:", len(trainable_base_layers))
        print("Maximum epochs:", FINE_TUNE_MAX_EPOCHS)
        print("Learning rate:", fine_tune_learning_rate)

        stage_2_start_time = time.perf_counter()

        stage_2_history = model.fit(training_dataset, validation_data=internal_validation_dataset,
            epochs=FINE_TUNE_MAX_EPOCHS,
            callbacks=create_callbacks(str(stage_2_checkpoint),minimum_learning_rate=(FINE_TUNE_MINIMUM_LEARNING_RATE)),
            verbose=1)


        stage_2_training_seconds = time.perf_counter() - stage_2_start_time

        stage_2_best_epoch, stage_2_best_validation_loss = get_best_epoch_and_loss(stage_2_history)


        if not stage_2_checkpoint.exists():
            raise RuntimeError("Stage 2 checkpoint was not saved.")

        print("Stage 2 best epoch:", stage_2_best_epoch)
        print("Stage 2 best internal-validation loss:", stage_2_best_validation_loss)


        # ====================================================
        # SELECT THE BETTER INTERNAL-VALIDATION STAGE
        # ====================================================

        if stage_2_best_validation_loss < stage_1_best_validation_loss:

            selected_stage = "fine_tuning"
            selected_epoch = stage_2_best_epoch
            selected_validation_loss = stage_2_best_validation_loss
            selected_checkpoint = stage_2_checkpoint

        else:
            selected_stage = "frozen_head"
            selected_epoch = stage_1_best_epoch
            selected_validation_loss = stage_1_best_validation_loss
            selected_checkpoint = stage_1_checkpoint



        # Restore exactly the checkpoint selected using only
        # the internal-validation partition.
        model.load_weights(selected_checkpoint)

        print("\nSelected stage:", selected_stage)
        print("Selected internal-validation loss:", selected_validation_loss)


        # ====================================================
        # SAVE BOTH TRAINING HISTORIES
        # ====================================================

        stage_1_history_dataframe = (pd.DataFrame(stage_1_history.history))

        stage_1_history_dataframe.insert(0, "stage_epoch",
            np.arange(1, len(stage_1_history_dataframe) + 1))

        stage_1_history_dataframe.insert(1, "stage", "frozen_head")

        stage_2_history_dataframe = pd.DataFrame(stage_2_history.history)

        stage_2_history_dataframe.insert(0,"stage_epoch",
            np.arange(1, len(stage_2_history_dataframe) + 1))

        stage_2_history_dataframe.insert(1, "stage", "fine_tuning")


        pd.concat([stage_1_history_dataframe, stage_2_history_dataframe],ignore_index=True,
                  ).to_csv(training_history_path, index=False)


        # ====================================================
        # EVALUATE ON THE UNTOUCHED OUTER FOLD ONCE
        # ====================================================

        print("\nEvaluating untouched "f"outer Fold {outer_fold}...")

        inference_start_time = time.perf_counter()

        probabilities = model.predict(outer_validation_dataset, verbose=1)

        inference_seconds = time.perf_counter() - inference_start_time

        probabilities = np.asarray(probabilities, dtype=np.float32)

        expected_probability_shape = (len(outer_validation_dataframe), NUMBER_OF_CLASSES)

        if probabilities.shape != (expected_probability_shape):
            raise RuntimeError("Unexpected outer-validation prediction shape. "f"Found {probabilities.shape}; expected {expected_probability_shape}.")

        if not np.isfinite(probabilities).all():
            raise RuntimeError("Non-finite outer-validation probabilities were produced.")

        probability_sums = probabilities.sum(axis=1)

        if not np.allclose(probability_sums, 1.0, atol=1e-5):
            raise RuntimeError("Some outer-validation probability rows do not sum to one.")

        true_labels = (outer_validation_dataframe["class_index"].to_numpy(dtype=np.int32))

        predicted_labels = np.argmax(probabilities,axis=1).astype(np.int32)

        fold_metrics = calculate_metrics(true_labels, predicted_labels, probabilities)

        fold_metrics.update(
            {
                "configuration_number": configuration_number,
                "configuration_id": configuration_id,
                "batch_size": batch_size,
                "head_learning_rate": head_learning_rate,
                "fine_tune_learning_rate": fine_tune_learning_rate,
                "outer_fold": outer_fold,
                "fold_seed": fold_seed,
                "model_training_images": len(model_training_dataframe),
                "internal_validation_images": len(internal_validation_dataframe),
                "outer_validation_images": len(outer_validation_dataframe),
                "selected_stage": selected_stage,
                "selected_best_epoch": selected_epoch,
                "selected_inner_validation_loss": selected_validation_loss,
                "stage_1_best_epoch": stage_1_best_epoch,
                "stage_1_best_validation_loss": stage_1_best_validation_loss,
                "stage_2_best_epoch": stage_2_best_epoch,
                "stage_2_best_validation_loss": stage_2_best_validation_loss,
                "trainable_base_layer_count": len(trainable_base_layers),
                "first_trainable_base_layer": trainable_base_layers[0],
                "last_trainable_base_layer": trainable_base_layers[-1],
                "stage_1_training_seconds": stage_1_training_seconds,
                "stage_2_training_seconds": stage_2_training_seconds,
                "total_training_seconds": stage_1_training_seconds + stage_2_training_seconds,
                "inference_seconds": inference_seconds,
                "inference_milliseconds_per_image": inference_seconds / len(outer_validation_dataframe) * 1000,
                "fold_total_seconds": time.perf_counter() - fold_start_time,
                "data_augmentation": False,
                "testing_images_loaded": False,
            }
        )


        # ====================================================
        # SAVE OUTER-FOLD PREDICTIONS
        # ====================================================

        predictions_dataframe = outer_validation_dataframe[["relative_path", "class", "class_index", "fold"]].copy()

        # Rename the original class columns so they clearly represent the true labels
        predictions_dataframe = predictions_dataframe.rename(columns={"class": "true_class", "class_index": "true_class_index"})

        # Add the predicted class index for each image
        predictions_dataframe["predicted_class_index"] = predicted_labels

        # Convert each predicted class index into its class name
        predictions_dataframe["predicted_class"] = [INDEX_TO_CLASS[int(index)] for index in predicted_labels]

        # Loop through each class so its predicted probability can be saved in a separate column
        for class_index, class_name in enumerate(CLASS_NAMES):

            # Add the predicted probability for the current class
            predictions_dataframe[f"probability_{class_name}"] = probabilities[:, class_index]

        # Save the completed predictions DataFrame as a CSV file
        predictions_dataframe.to_csv(predictions_path, index=False)


        # ====================================================
        # SAVE CONFUSION MATRIX
        # ====================================================

        # Create the confusion matrix from the true and predicted labels
        fold_confusion_matrix = confusion_matrix(true_labels, predicted_labels, labels=list(range(NUMBER_OF_CLASSES)))

        # Convert the confusion matrix to a DataFrame and save it as a CSV file
        pd.DataFrame(fold_confusion_matrix, index=CLASS_NAMES, columns=CLASS_NAMES).to_csv(
            confusion_matrix_path, index_label="true_class")


        # ====================================================
        # SAVE CLASSIFICATION REPORT
        # ====================================================

        # Create a detailed classification report for the true and predicted labels
        fold_report = classification_report(
            true_labels,
            predicted_labels,
            labels=list(range(NUMBER_OF_CLASSES)),
            target_names=(CLASS_NAMES),
            output_dict=True,
            zero_division=0,
            )

        # Convert the classifiaction report to a DataFrame and save it as a CSV file
        pd.DataFrame(fold_report).transpose().to_csv(classification_report_path,index_label=("class_or_average"))



        # Remove large tepmprary checkpoints
        for checkpoint_path in [stage_1_checkpoint, stage_2_checkpoint]:
            if checkpoint_path.exists():
                checkpoint_path.unlink()

        # Save to json file
        save_json_atomic(fold_metrics, fold_metrics_path)


        completion_information = {
        "status": "completed",
        "configuration_id": configuration_id,
        "outer_fold": outer_fold,
        "macro_f1": fold_metrics["macro_f1"],
        "balanced_accuracy": fold_metrics["balanced_accuracy"],
        "selected_stage": selected_stage,
        "testing_images_loaded": False,
        "completed_at": datetime.now().isoformat(),
        }

        save_json_atomic(completion_information, completion_path)

        print(f"\nFold {outer_fold} completed.")
        print("Accuracy:", f"{fold_metrics['accuracy']:.6f}")
        print("Balanced accuracy:", f"{fold_metrics['balanced_accuracy']:.6f}")
        print("Macro F1:", f"{fold_metrics['macro_f1']:.6f}")
        print("Selected stage:", selected_stage)



        # ----------------------------------------------------
        # Release the model and datasets before the next run
        # ----------------------------------------------------
        del model
        del base_model
        del training_dataset
        del internal_validation_dataset
        del outer_validation_dataset
        del stage_1_history
        del stage_2_history
        del probabilities
        tf.keras.backend.clear_session()
        gc.collect()


# ============================================================
# 8. Verify that all 60 fold runs are complete
# ============================================================

missing_fold_outputs = []

for configuration in search_configurations:
    configuration_directory = CV_RESULTS_DIR / configuration["configuration_id"]


    for outer_fold in OUTER_FOLDS:
        fold_directory = configuration_directory / f"fold_{outer_fold}"


        required_paths = [
            fold_directory / "fold_complete.json",
            fold_directory / "fold_metrics.json",
            fold_directory / "outer_validation_predictions.csv",
            fold_directory / "partition_manifest.csv",
            fold_directory / "training_history.csv",
            fold_directory / "confusion_matrix.csv",
            fold_directory / "classification_report.csv",
            ]

        for path in required_paths:
            if not path.exists():
                missing_fold_outputs.append(str(path))


if missing_fold_outputs:
    raise RuntimeError("The ResNet50 search cannot be aggregated because some required fold outputs are missing:\n"
        + "\n".join(missing_fold_outputs))


# ============================================================
# 9. Aggregate all 60 fold metrics
# ============================================================

fold_metric_records = []

# Loop through each hyperparameter configuration
for configuration in search_configurations:
    configuration_id = configuration["configuration_id"]

    # Load the metrics from every outer fold for the current configuration
    for outer_fold in OUTER_FOLDS:
        fold_metrics_path = CV_RESULTS_DIR / configuration_id / f"fold_{outer_fold}" / "fold_metrics.json"

        with fold_metrics_path.open("r", encoding="utf-8") as file:
            fold_metric_records.append(json.load(file))

# Convert all fold metrics into one DataFrame
all_fold_metrics = pd.DataFrame(fold_metric_records)

# Sort the metrics by configuration number and outer fold
all_fold_metrics = all_fold_metrics.sort_values(["configuration_number", "outer_fold"]).reset_index(drop=True)

# Check that the expected number of configuration-fold results were loaded
if len(all_fold_metrics) != NUMBER_OF_FOLD_RUNS:
    raise RuntimeError(f"Expected {NUMBER_OF_FOLD_RUNS} configuration-fold metric records, but found {len(all_fold_metrics)}.")

# Save all fold metrics as a CSV file
all_fold_metrics.to_csv(ALL_FOLD_METRICS_PATH, index=False)



# ============================================================
# 10. Build the hyperparameter leaderboard
# ============================================================

leaderboard = (
    all_fold_metrics
    .groupby(
        ["configuration_number", "configuration_id", "batch_size", "head_learning_rate", "fine_tune_learning_rate"],
        as_index=False)
    .agg(
        completed_folds=("outer_fold", "count"),
        mean_accuracy=("accuracy", "mean"),
        mean_balanced_accuracy=("balanced_accuracy","mean"),
        mean_macro_f1=("macro_f1", "mean"),
        macro_f1_standard_deviation=("macro_f1", lambda values: values.std(ddof=1)),
        mean_weighted_f1=("weighted_f1", "mean"),
        mean_log_loss=("log_loss", "mean"),
        mean_total_training_seconds=("total_training_seconds", "mean")))


if not (leaderboard["completed_folds"] == 5).all():
    raise RuntimeError("At least one hyperparameter configuration does not contain five completed folds.")


# Count how often each training stage was selected
stage_counts = (
    all_fold_metrics.groupby("configuration_id")["selected_stage"]
    .value_counts()
    .unstack(fill_value=0)
    .reset_index())


if "frozen_head" not in stage_counts.columns:
    stage_counts["frozen_head"] = 0


if "fine_tuning" not in stage_counts.columns:
    stage_counts["fine_tuning"] = 0


stage_counts = stage_counts.rename(
    columns={
        "frozen_head": "frozen_head_selected_folds",
        "fine_tuning": "fine_tuning_selected_folds",
    })

leaderboard = leaderboard.merge(stage_counts, on="configuration_id", how="left")


# Apply the predetermined configuration-selection rule
leaderboard = leaderboard.sort_values(
    by=["mean_macro_f1", "mean_balanced_accuracy", "macro_f1_standard_deviation", "mean_log_loss", "configuration_number"],
    ascending=[False, False, True, True, True],
    kind="mergesort",
).reset_index(drop=True)

# Add leaderboard ranks starting from 1
leaderboard.insert(0, "rank", np.arange(1, len(leaderboard) + 1))


leaderboard.to_csv(LEADERBOARD_PATH, index=False)


print("\n" + "=" * 70)
print("RESNET50 HYPERPARAMETER LEADERBOARD")
print("=" * 70)
display(leaderboard)


# ============================================================
# 11. Select the winning configuration
# ============================================================

# Select the highest-ranked configuration
winning_row = leaderboard.iloc[0]
selected_configuration_id = str(winning_row["configuration_id"])

# Keep only the five folds belonging to the selected configuration
selected_fold_metrics = all_fold_metrics[
    all_fold_metrics["configuration_id"] == selected_configuration_id
].copy().sort_values("outer_fold").reset_index(drop=True)

# Check that the selected configuration contains all five folds
if len(selected_fold_metrics) != 5:
    raise RuntimeError("The selected configuration does not contain exactly five completed folds.")



# ============================================================
# 12. Select the final training stage
# ============================================================

# Count how many folds selected each training stage
selected_stage_counts = selected_fold_metrics["selected_stage"].value_counts().to_dict()
frozen_head_count = int(selected_stage_counts.get("frozen_head", 0))
fine_tuning_count = int(selected_stage_counts.get("fine_tuning", 0))

# Select the final stage by five-fold majority
if fine_tuning_count > frozen_head_count:
    selected_final_stage = "fine_tuning"
elif frozen_head_count > fine_tuning_count:
    selected_final_stage = "frozen_head"
else:
    raise RuntimeError("The selected final stage could not be determined by five-fold majority.")



# ============================================================
# 13. Select final epoch counts
# ============================================================


# Use the median best Stage 1 epoch across the five folds
selected_head_epochs = int(np.median(selected_fold_metrics["stage_1_best_epoch"].to_numpy(dtype=np.int32)))


# Use the median best Stage 2 epoch only if fine-tuning was selected
if selected_final_stage == "fine_tuning":
    selected_fine_tune_epochs = int(np.median(selected_fold_metrics["stage_2_best_epoch"].to_numpy(dtype=np.int32)))
else:
    selected_fine_tune_epochs = 0


# Check that the selected Stage 1 epoch count is valid
if selected_head_epochs < 1:
    raise RuntimeError("The selected Stage 1 epoch count is invalid.")


# Check that the selected fine-tuning epoch count is valid when fine-tuning is used
if selected_final_stage == "fine_tuning" and selected_fine_tune_epochs < 1:
    raise RuntimeError("The selected fine-tuning epoch count is invalid.")



# ============================================================
# 14. Save the selected ResNet50 configuration
# ============================================================

selected_configuration = {
    "configuration_id": selected_configuration_id,
    "architecture": "ResNet50",
    "batch_size": int(winning_row["batch_size"]),
    "head_learning_rate": float(winning_row["head_learning_rate"]),
    "fine_tune_learning_rate": float(winning_row["fine_tune_learning_rate"]),
    "selected_final_stage": selected_final_stage,
    "selected_head_epochs": selected_head_epochs,
    "selected_fine_tune_epochs": selected_fine_tune_epochs,
    "fine_tune_from_layer": FINE_TUNE_FROM_LAYER,
    "dropout_rate": DROPOUT_RATE,
    "five_fold_mean_macro_f1": float(winning_row["mean_macro_f1"]),
    "five_fold_mean_balanced_accuracy": float(winning_row["mean_balanced_accuracy"]),
    "five_fold_macro_f1_standard_deviation": float(winning_row["macro_f1_standard_deviation"]),
    "five_fold_mean_log_loss": float(winning_row["mean_log_loss"]),
    "stage_selection_counts": {"frozen_head": frozen_head_count, "fine_tuning": fine_tuning_count},
    "epoch_selection_rule": {
        "head": "median best Stage 1 internal-validation epoch across the five folds",
        "fine_tuning": "median best Stage 2 internal-validation epoch across the five folds when fine-tuning wins the stage majority",
    },
    "configuration_selection_rule": [
        "highest mean five-fold macro F1",
        "highest mean five-fold balanced accuracy",
        "lowest five-fold macro-F1 standard deviation",
        "lowest mean five-fold log loss",
    ],
    "data_augmentation": False,
    "testing_images_loaded": False,
    "selected_at": datetime.now().isoformat(),
}

save_json_atomic(selected_configuration, SELECTED_CONFIGURATION_PATH)

# ============================================================
# 15. Save the search completion marker LAST
# ============================================================

search_total_seconds = time.perf_counter() - search_start_time

search_completion = {
    "status": "completed",
    "architecture": "ResNet50",
    "number_of_configurations": NUMBER_OF_CONFIGURATIONS,
    "outer_folds": len(OUTER_FOLDS),
    "completed_fold_runs": len(all_fold_metrics),
    "selected_configuration_id": selected_configuration_id,
    "selected_final_stage": selected_final_stage,
    "selected_head_epochs": selected_head_epochs,
    "selected_fine_tune_epochs": selected_fine_tune_epochs,
    "search_total_seconds_this_execution": search_total_seconds,
    "testing_images_loaded": False,
    "completed_at": datetime.now().isoformat(),
}

save_json_atomic(search_completion, SEARCH_COMPLETION_PATH)


# ============================================================
# 16. Final search output
# ============================================================

print("\n" + "=" * 70)
print("RESNET50 HYPERPARAMETER SEARCH COMPLETED")
print("=" * 70)

print("Completed fold runs:", len(all_fold_metrics))
print("\nSelected configuration:", selected_configuration_id)
print("Batch size:", selected_configuration["batch_size"])
print("Head learning rate:", selected_configuration["head_learning_rate"])
print("Fine-tuning learning rate:", selected_configuration["fine_tune_learning_rate"])
print("Selected final stage:", selected_final_stage)
print("Selected Stage 1 epochs:", selected_head_epochs)
print("Selected Stage 2 epochs:", selected_fine_tune_epochs)
print("\nFive-fold mean macro F1:", f"{selected_configuration['five_fold_mean_macro_f1']:.6f}")
print("Five-fold mean balanced accuracy:", f"{selected_configuration['five_fold_mean_balanced_accuracy']:.6f}")
print("Five-fold macro-F1 standard deviation:", f"{selected_configuration['five_fold_macro_f1_standard_deviation']:.6f}")
print("Five-fold mean log loss:", f"{selected_configuration['five_fold_mean_log_loss']:.6f}")
print("\nTesting images were not loaded or evaluated during this search.")
print("\nSearch results saved to:")
print(CV_RESULTS_DIR)



In [ ]:
# ============================================================
# FINAL RESNET50 TRAINING ON ALL 5,600 TRAINING IMAGES
# ============================================================
#
# The hyperparameters and epoch counts used here are loaded
# directly from the completed five-fold hyperparameter search.
#
# Important:
#   - A fresh ImageNet ResNet50 model is created
#   - All 5,600 Training images are used
#   - No validation partition is used
#   - No early stopping is used
#   - No image augmentation is used
#   - Testing images are not loaded or evaluated
#   - Batch Normalization remains frozen during fine-tuning
#
# The final model is saved only after the selected training
# stages have completed successfully.
# ============================================================

import gc
import json
import shutil
import time
from datetime import datetime
from pathlib import Path
import numpy as np
import pandas as pd
import tensorflow as tf


# ============================================================
# 1. Final-training paths
# ============================================================

# Search outputs that determine the final training procedure
SELECTED_CONFIGURATION_PATH = CV_RESULTS_DIR / "selected_configuration.json"
SEARCH_COMPLETION_PATH = CV_RESULTS_DIR / "search_completion.json"


# Final ResNet50 model
FINAL_MODEL_PATH = FINAL_RESULTS_DIR / "final_resnet50_cropped.keras"


# Training-history outputs
STAGE_1_LOG_PATH = FINAL_RESULTS_DIR / "stage_1_training_log.csv"
STAGE_2_LOG_PATH = FINAL_RESULTS_DIR / "stage_2_training_log.csv"
COMBINED_HISTORY_PATH = FINAL_RESULTS_DIR / "combined_training_history.csv"
HISTORY_JSON_PATH = FINAL_RESULTS_DIR / "training_history.json"


# Model and training metadata
FINAL_CONFIGURATION_PATH = FINAL_RESULTS_DIR / "final_training_configuration.json"
FINAL_MANIFEST_PATH = FINAL_RESULTS_DIR / "final_training_manifest.csv"
MODEL_SUMMARY_PATH = FINAL_RESULTS_DIR / "model_summary.txt"
TIMING_PATH = FINAL_RESULTS_DIR / "final_training_timing.json"


# Intermediate Stage 1 and Stage 2 weights
STAGE_1_WEIGHTS_PATH = FINAL_RESULTS_DIR / "stage_1_final.weights.h5"
STAGE_2_WEIGHTS_PATH = FINAL_RESULTS_DIR / "stage_2_final.weights.h5"


# Copy of the selected search configuration used for training
SELECTED_CONFIGURATION_COPY_PATH = FINAL_RESULTS_DIR / "selected_configuration_used.json"


# This file is written LAST and marks a successfully completed final-training run
TRAINING_COMPLETION_PATH = FINAL_RESULTS_DIR / "training_complete.json"


# Save the model locally before copying it atomically
# to persistent Google Drive storage
LOCAL_MODEL_PATH = Path("/content/final_resnet50_cropped.keras")


# ============================================================
# 2. Utility functions
# ============================================================

def save_json_atomic(data, destination):
    """
    Save JSON through a temporary file before replacing the destination.
    """

    destination = Path(destination)
    temporary_path = destination.with_name(destination.name + ".tmp")

    with temporary_path.open("w", encoding="utf-8") as file:
      json.dump(data, file, indent=4, default=str)

    temporary_path.replace(destination)


def count_weight_parameters(weights):
    """
    Count the number of scalar parameters represented by a list of Keras weights.
    """
    return int(sum(tf.keras.backend.count_params(weight) for weight in weights))


# ============================================================
# 3. Verify the completed hyperparameter search
# ============================================================

print("=" * 70)
print("FINAL RESNET50 TRAINING")
print("=" * 70)


if not SEARCH_COMPLETION_PATH.exists():
    raise FileNotFoundError("The ResNet50 hyperparameter-search completion marker was not found:\n" f"{SEARCH_COMPLETION_PATH}")

if not SELECTED_CONFIGURATION_PATH.exists():
    raise FileNotFoundError("The selected ResNet50 configuration was not found:\n" f"{SELECTED_CONFIGURATION_PATH}")

# Load the search completion information
with SEARCH_COMPLETION_PATH.open("r", encoding="utf-8") as file:
    search_completion = json.load(file)

# Load the winning hyperparameter configuration
with SELECTED_CONFIGURATION_PATH.open("r", encoding="utf-8") as file:
    selected_configuration = json.load(file)

if search_completion.get("status") != "completed":
    raise RuntimeError("The ResNet50 hyperparameter search is not marked as completed.")

if search_completion.get("completed_fold_runs") != 60:
    raise RuntimeError(f"Expected 60 completed configuration-fold runs, but the search completion file reports {search_completion.get('completed_fold_runs')}.")

if search_completion.get("selected_configuration_id") != selected_configuration.get("configuration_id"):
    raise RuntimeError("The selected configuration does not match the completed hyperparameter search.")





# ============================================================
# 4. Load the selected final-training configuration
# ============================================================

SELECTED_CONFIGURATION_ID = selected_configuration["configuration_id"]
BATCH_SIZE = int(selected_configuration["batch_size"])
HEAD_LEARNING_RATE = float(selected_configuration["head_learning_rate"])
FINE_TUNE_LEARNING_RATE = float(selected_configuration["fine_tune_learning_rate"])
SELECTED_FINAL_STAGE = selected_configuration["selected_final_stage"]
HEAD_EPOCHS = int(selected_configuration["selected_head_epochs"])
FINE_TUNE_EPOCHS = int(selected_configuration["selected_fine_tune_epochs"])


# Verify that the architecture-specific settings match the current notebook
if selected_configuration["fine_tune_from_layer"] != FINE_TUNE_FROM_LAYER:
    raise RuntimeError("The selected fine-tuning boundary does not match the current ResNet50 notebook.")

if not np.isclose(float(selected_configuration["dropout_rate"]), DROPOUT_RATE):
    raise RuntimeError("The selected dropout rate does not match the current ResNet50 model definition.")

if SELECTED_FINAL_STAGE not in {"frozen_head", "fine_tuning"}:
    raise RuntimeError(f"Unexpected selected final stage: {SELECTED_FINAL_STAGE}")

if HEAD_EPOCHS < 1:
    raise RuntimeError("The selected Stage 1 epoch count is invalid.")

if SELECTED_FINAL_STAGE == "fine_tuning" and FINE_TUNE_EPOCHS < 1:
    raise RuntimeError("The selected Stage 2 epoch count is invalid.")

print("\n--- Selected configuration ---")
print("Configuration:", SELECTED_CONFIGURATION_ID)
print("Batch size:", BATCH_SIZE)
print("Head learning rate:", HEAD_LEARNING_RATE)
print("Fine-tuning learning rate:", FINE_TUNE_LEARNING_RATE)
print("Selected final stage:", SELECTED_FINAL_STAGE)
print("Stage 1 epochs:", HEAD_EPOCHS)
print("Stage 2 epochs:", FINE_TUNE_EPOCHS)


# ============================================================
# 5. Verify final-training completion state
# ============================================================

required_final_outputs = [
    FINAL_MODEL_PATH,
    FINAL_CONFIGURATION_PATH,
    FINAL_MANIFEST_PATH,
    MODEL_SUMMARY_PATH,
    COMBINED_HISTORY_PATH,
    HISTORY_JSON_PATH,
    TIMING_PATH,
    SELECTED_CONFIGURATION_COPY_PATH,
]


final_training_is_complete = TRAINING_COMPLETION_PATH.exists() and all(path.exists() for path in required_final_outputs)


# If a completed run already exists, verify that it belongs to the same selected configuration
if final_training_is_complete:
    with TRAINING_COMPLETION_PATH.open("r", encoding="utf-8") as file:
        existing_completion = json.load(file)


    if existing_completion.get("selected_configuration_id") == SELECTED_CONFIGURATION_ID:
        print("\nA complete final ResNet50 training run already exists.")
        print("Final model:", FINAL_MODEL_PATH)
        print("No training was repeated.")
    else:
        print("\nThe existing final-training output belongs to a different configuration.")
        print("Removing the existing final-training directory...")
        shutil.rmtree(FINAL_RESULTS_DIR)
        final_training_is_complete = False


# Remove incomplete final-training output so training can restart from the beginning
elif FINAL_RESULTS_DIR.exists():
    print("\nIncomplete final-training output was detected.")
    print("Removing the incomplete final-training directory...")
    shutil.rmtree(FINAL_RESULTS_DIR)


# ============================================================
# 6. Train the final model when required
# ============================================================

if not final_training_is_complete:

    FINAL_RESULTS_DIR.mkdir(
        parents=True,
        exist_ok=True,
    )


    final_training_start = (
        time.perf_counter()
    )


    # --------------------------------------------------------
    # Reproducibility
    # --------------------------------------------------------

    tf.keras.mixed_precision.set_global_policy(
        "float32"
    )


    tf.keras.utils.set_random_seed(
        RANDOM_SEED
    )


    try:

        tf.config.experimental.enable_op_determinism()

        deterministic_status = (
            "enabled"
        )


    except Exception as error:

        deterministic_status = (
            "requested, but TensorFlow returned: "
            f"{error}"
        )


    # --------------------------------------------------------
    # Verify GPU availability
    # --------------------------------------------------------

    gpu_devices = (
        tf.config.list_physical_devices(
            "GPU"
        )
    )


    if not gpu_devices:

        raise RuntimeError(
            "No TensorFlow GPU is available."
        )


    print(
        "\nGPU devices:",
        gpu_devices
    )

    print(
        "Deterministic TensorFlow operations:",
        deterministic_status
    )


    # ========================================================
    # 7. Prepare all 5,600 Training images
    # ========================================================

    dataset_preparation_start = (
        time.perf_counter()
    )


    final_training_dataframe = (
        assignments
        .copy()
        .reset_index(
            drop=True
        )
    )


    if len(
        final_training_dataframe
    ) != 5600:

        raise RuntimeError(
            "Expected exactly 5,600 Training images, "
            f"but found "
            f"{len(final_training_dataframe)}."
        )


    if final_training_dataframe[
        "relative_path"
    ].duplicated().any():

        raise RuntimeError(
            "Duplicate Training image paths were found."
        )


    if final_training_dataframe[
        "class_index"
    ].isna().any():

        raise RuntimeError(
            "Missing Training class indices were found."
        )


    # Verify the final class distribution
    final_class_counts = (
        final_training_dataframe[
            "class"
        ]
        .value_counts()
        .reindex(
            CLASS_NAMES,
            fill_value=0,
        )
    )


    for class_name in CLASS_NAMES:

        if (
            int(
                final_class_counts[
                    class_name
                ]
            )
            != 1400
        ):

            raise RuntimeError(
                "Expected 1,400 Training images for "
                f"{class_name}, but found "
                f"{final_class_counts[class_name]}."
            )


    # Save the exact 5,600 records used for final training
    final_training_dataframe[
        [
            "relative_path",
            "class",
            "class_index",
            "fold",
        ]
    ].to_csv(
        FINAL_MANIFEST_PATH,
        index=False,
    )


    print(
        "\n--- Final Training data ---"
    )

    print(
        "Training images:",
        len(
            final_training_dataframe
        )
    )


    print(
        "Class mapping:",
        CLASS_TO_INDEX
    )


    for class_name in CLASS_NAMES:

        print(
            f"{class_name:<12}: "
            f"{int(final_class_counts[class_name])}"
        )


    # --------------------------------------------------------
    # Create the final deterministic Training dataset
    # --------------------------------------------------------

    final_training_dataset = (
        make_dataset(
            final_training_dataframe,
            training=True,
            seed=RANDOM_SEED,
            batch_size=BATCH_SIZE,
        )
    )


    dataset_preparation_seconds = (
        time.perf_counter()
        - dataset_preparation_start
    )


    # ========================================================
    # 8. Build a fresh ImageNet ResNet50
    # ========================================================

    tf.keras.backend.clear_session()


    tf.keras.utils.set_random_seed(
        RANDOM_SEED
    )


    gc.collect()


    (
        model,
        base_model,
    ) = build_model()


    stage_1_trainable_parameters = (
        count_weight_parameters(
            model.trainable_weights
        )
    )


    print(
        "\n--- Fresh final model ---"
    )

    print(
        "Model name:",
        model.name
    )

    print(
        "Total parameters:",
        f"{model.count_params():,}"
    )

    print(
        "Stage 1 trainable parameters:",
        f"{stage_1_trainable_parameters:,}"
    )

    print(
        "Backbone trainable:",
        base_model.trainable
    )


    if base_model.trainable:

        raise RuntimeError(
            "The ResNet50 backbone should be completely "
            "frozen before Stage 1."
        )


    # ========================================================
    # 9. Stage 1 — train the classifier head
    # ========================================================

    print(
        "\n"
        + "=" * 70
    )

    print(
        "STAGE 1 OF 2 — CLASSIFIER-HEAD TRAINING"
    )

    print(
        "=" * 70
    )


    print(
        "Training images:",
        len(
            final_training_dataframe
        )
    )

    print(
        "Epochs:",
        HEAD_EPOCHS
    )

    print(
        "Batch size:",
        BATCH_SIZE
    )

    print(
        "Learning rate:",
        HEAD_LEARNING_RATE
    )


    compile_model(
        model,
        HEAD_LEARNING_RATE,
    )


    stage_1_start = (
        time.perf_counter()
    )


    stage_1_history = model.fit(
        final_training_dataset,
        epochs=HEAD_EPOCHS,
        verbose=1,
        callbacks=[
            tf.keras.callbacks.TerminateOnNaN(),

            tf.keras.callbacks.CSVLogger(
                str(
                    STAGE_1_LOG_PATH
                ),
                append=False,
            ),
        ],
    )


    stage_1_seconds = (
        time.perf_counter()
        - stage_1_start
    )


    model.save_weights(
        STAGE_1_WEIGHTS_PATH
    )


    print(
        "\nStage 1 completed in:",
        round(
            stage_1_seconds / 60,
            2
        ),
        "minutes"
    )


    # ========================================================
    # 10. Stage 2 — fine-tune ResNet50 when selected
    # ========================================================

    stage_2_history = None

    stage_2_seconds = 0.0

    trainable_base_layers = []

    stage_2_trainable_parameters = (
        stage_1_trainable_parameters
    )


    if SELECTED_FINAL_STAGE == (
        "fine_tuning"
    ):

        print(
            "\n"
            + "=" * 70
        )

        print(
            "STAGE 2 OF 2 — RESNET50 FINE-TUNING"
        )

        print(
            "=" * 70
        )


        # Enable the exact same upper ResNet50 layers
        # that were fine-tuned during cross-validation
        trainable_base_layers = (
            enable_fine_tuning(
                base_model
            )
        )


        # Recompile after changing layer trainability
        compile_model(
            model,
            FINE_TUNE_LEARNING_RATE,
        )


        stage_2_trainable_parameters = (
            count_weight_parameters(
                model.trainable_weights
            )
        )


        print(
            "Fine-tuning begins from:",
            FINE_TUNE_FROM_LAYER
        )

        print(
            "Trainable backbone layers:",
            len(
                trainable_base_layers
            )
        )

        print(
            "First trainable backbone layer:",
            trainable_base_layers[
                0
            ]
        )

        print(
            "Last trainable backbone layer:",
            trainable_base_layers[
                -1
            ]
        )

        print(
            "Total trainable parameters:",
            f"{stage_2_trainable_parameters:,}"
        )

        print(
            "Epochs:",
            FINE_TUNE_EPOCHS
        )

        print(
            "Learning rate:",
            FINE_TUNE_LEARNING_RATE
        )


        stage_2_start = (
            time.perf_counter()
        )


        stage_2_history = model.fit(
            final_training_dataset,
            epochs=(
                FINE_TUNE_EPOCHS
            ),
            verbose=1,
            callbacks=[
                tf.keras.callbacks.TerminateOnNaN(),

                tf.keras.callbacks.CSVLogger(
                    str(
                        STAGE_2_LOG_PATH
                    ),
                    append=False,
                ),
            ],
        )


        stage_2_seconds = (
            time.perf_counter()
            - stage_2_start
        )


        model.save_weights(
            STAGE_2_WEIGHTS_PATH
        )


        print(
            "\nStage 2 completed in:",
            round(
                stage_2_seconds / 60,
                2
            ),
            "minutes"
        )


    else:

        print(
            "\nThe five-fold search selected the "
            "frozen-head stage as the final stage."
        )

        print(
            "Stage 2 fine-tuning was therefore skipped."
        )


    # ========================================================
    # 11. Save the complete training history
    # ========================================================

    stage_1_history_dataframe = (
        pd.DataFrame(
            stage_1_history.history
        )
    )


    stage_1_history_dataframe.insert(
        0,
        "stage",
        "frozen_head",
    )


    stage_1_history_dataframe.insert(
        1,
        "stage_epoch",
        np.arange(
            1,
            len(
                stage_1_history_dataframe
            )
            + 1,
        ),
    )


    stage_1_history_dataframe.insert(
        2,
        "global_epoch",
        np.arange(
            1,
            len(
                stage_1_history_dataframe
            )
            + 1,
        ),
    )


    history_frames = [
        stage_1_history_dataframe
    ]


    history_for_json = {
        "stage_1_frozen_head": {
            key: [
                float(
                    value
                )
                for value
                in values
            ]
            for key, values
            in stage_1_history.history.items()
        }
    }


    if stage_2_history is not None:

        stage_2_history_dataframe = (
            pd.DataFrame(
                stage_2_history.history
            )
        )


        stage_2_history_dataframe.insert(
            0,
            "stage",
            "fine_tuning",
        )


        stage_2_history_dataframe.insert(
            1,
            "stage_epoch",
            np.arange(
                1,
                len(
                    stage_2_history_dataframe
                )
                + 1,
            ),
        )


        stage_2_history_dataframe.insert(
            2,
            "global_epoch",
            np.arange(
                HEAD_EPOCHS + 1,
                HEAD_EPOCHS
                + len(
                    stage_2_history_dataframe
                )
                + 1,
            ),
        )


        history_frames.append(
            stage_2_history_dataframe
        )


        history_for_json[
            "stage_2_fine_tuning"
        ] = {
            key: [
                float(
                    value
                )
                for value
                in values
            ]
            for key, values
            in stage_2_history.history.items()
        }


    combined_history = pd.concat(
        history_frames,
        ignore_index=True,
    )


    combined_history.to_csv(
        COMBINED_HISTORY_PATH,
        index=False,
    )


    save_json_atomic(
        history_for_json,
        HISTORY_JSON_PATH,
    )


    # ========================================================
    # 12. Save model summary
    # ========================================================

    with MODEL_SUMMARY_PATH.open(
        "w",
        encoding="utf-8",
    ) as summary_file:

        model.summary(
            print_fn=(
                lambda line:
                summary_file.write(
                    line + "\n"
                )
            )
        )


    # ========================================================
    # 13. Save final-training configuration
    # ========================================================

    final_configuration = {
        "architecture": "ResNet50",
        "experiment": (
            "selected_final_no_augmentation_v2"
        ),
        "selected_configuration_id": (
            SELECTED_CONFIGURATION_ID
        ),
        "dataset_variant": "cropped",
        "dataset_partition": "Training",
        "training_image_count": (
            int(
                len(
                    final_training_dataframe
                )
            )
        ),
        "testing_images_loaded": False,
        "validation_used": False,
        "early_stopping_used": False,
        "data_augmentation": False,
        "class_names": (
            list(
                CLASS_NAMES
            )
        ),
        "class_to_index": (
            CLASS_TO_INDEX
        ),
        "class_counts": {
            class_name: int(
                final_class_counts[
                    class_name
                ]
            )
            for class_name
            in CLASS_NAMES
        },
        "input_shape": [
            IMAGE_HEIGHT,
            IMAGE_WIDTH,
            IMAGE_CHANNELS,
        ],
        "input_representation": (
            "single-channel grayscale converted "
            "to three-channel pseudo-RGB"
        ),
        "preprocessing": (
            "tf.keras.applications.resnet50."
            "preprocess_input"
        ),
        "imagenet_pretrained": True,
        "batch_size": (
            BATCH_SIZE
        ),
        "random_seed": (
            RANDOM_SEED
        ),
        "dropout_rate": (
            DROPOUT_RATE
        ),
        "selected_final_stage": (
            SELECTED_FINAL_STAGE
        ),
        "stage_1": {
            "description": (
                "Frozen ImageNet ResNet50 backbone "
                "with classifier-head training"
            ),
            "epochs": (
                HEAD_EPOCHS
            ),
            "learning_rate": (
                HEAD_LEARNING_RATE
            ),
            "trainable_parameters": (
                stage_1_trainable_parameters
            ),
        },
        "stage_2": {
            "performed": (
                SELECTED_FINAL_STAGE
                == "fine_tuning"
            ),
            "description": (
                "Fine-tuning of the upper ResNet50 "
                "conv5_x stage"
            ),
            "epochs": (
                FINE_TUNE_EPOCHS
                if (
                    SELECTED_FINAL_STAGE
                    == "fine_tuning"
                )
                else 0
            ),
            "learning_rate": (
                FINE_TUNE_LEARNING_RATE
            ),
            "fine_tune_from_layer": (
                FINE_TUNE_FROM_LAYER
            ),
            "batch_normalisation_frozen": True,
            "backbone_called_with_training_false": True,
            "trainable_backbone_layer_count": (
                len(
                    trainable_base_layers
                )
            ),
            "first_trainable_backbone_layer": (
                trainable_base_layers[
                    0
                ]
                if trainable_base_layers
                else None
            ),
            "last_trainable_backbone_layer": (
                trainable_base_layers[
                    -1
                ]
                if trainable_base_layers
                else None
            ),
            "total_trainable_parameters": (
                stage_2_trainable_parameters
            ),
        },
        "epoch_selection_rule": (
            selected_configuration[
                "epoch_selection_rule"
            ]
        ),
        "five_fold_mean_macro_f1": (
            selected_configuration[
                "five_fold_mean_macro_f1"
            ]
        ),
        "five_fold_mean_balanced_accuracy": (
            selected_configuration[
                "five_fold_mean_balanced_accuracy"
            ]
        ),
        "five_fold_macro_f1_standard_deviation": (
            selected_configuration[
                "five_fold_macro_f1_standard_deviation"
            ]
        ),
        "five_fold_mean_log_loss": (
            selected_configuration[
                "five_fold_mean_log_loss"
            ]
        ),
        "tensorflow_version": (
            tf.__version__
        ),
        "keras_version": (
            tf.keras.__version__
        ),
        "deterministic_operations": (
            deterministic_status
        ),
        "created_at": (
            datetime.now().isoformat()
        ),
    }


    save_json_atomic(
        final_configuration,
        FINAL_CONFIGURATION_PATH,
    )


    # Save a copy of the exact search result used
    save_json_atomic(
        selected_configuration,
        SELECTED_CONFIGURATION_COPY_PATH,
    )


    # ========================================================
    # 14. Save the complete final ResNet50 model
    # ========================================================

    print(
        "\n"
        + "=" * 70
    )

    print(
        "SAVING FINAL RESNET50 MODEL"
    )

    print(
        "=" * 70
    )


    model_saving_start = (
        time.perf_counter()
    )


    # Remove an incomplete temporary local model if present
    if LOCAL_MODEL_PATH.exists():

        LOCAL_MODEL_PATH.unlink()


    # Save the complete model locally first
    model.save(
        LOCAL_MODEL_PATH,
        overwrite=True,
    )


    # Copy to a temporary file on Google Drive
    temporary_drive_model = (
        FINAL_RESULTS_DIR
        / "final_resnet50_cropped.tmp.keras"
    )


    if temporary_drive_model.exists():

        temporary_drive_model.unlink()


    shutil.copy2(
        LOCAL_MODEL_PATH,
        temporary_drive_model,
    )


    # Atomically rename the temporary Drive copy
    temporary_drive_model.replace(
        FINAL_MODEL_PATH
    )


    # Remove the temporary local model
    if LOCAL_MODEL_PATH.exists():

        LOCAL_MODEL_PATH.unlink()


    model_saving_seconds = (
        time.perf_counter()
        - model_saving_start
    )


    if not FINAL_MODEL_PATH.exists():

        raise RuntimeError(
            "The final ResNet50 model was not saved "
            "successfully."
        )


    # ========================================================
    # 15. Save timing information
    # ========================================================

    total_model_training_seconds = (
        stage_1_seconds
        + stage_2_seconds
    )


    complete_final_training_seconds = (
        time.perf_counter()
        - final_training_start
    )


    timing_information = {
        "dataset_preparation_seconds": (
            dataset_preparation_seconds
        ),
        "stage_1_training_seconds": (
            stage_1_seconds
        ),
        "stage_1_training_minutes": (
            stage_1_seconds / 60
        ),
        "stage_2_training_seconds": (
            stage_2_seconds
        ),
        "stage_2_training_minutes": (
            stage_2_seconds / 60
        ),
        "total_model_training_seconds": (
            total_model_training_seconds
        ),
        "total_model_training_minutes": (
            total_model_training_seconds
            / 60
        ),
        "model_saving_seconds": (
            model_saving_seconds
        ),
        "complete_final_training_seconds": (
            complete_final_training_seconds
        ),
        "complete_final_training_minutes": (
            complete_final_training_seconds
            / 60
        ),
    }


    save_json_atomic(
        timing_information,
        TIMING_PATH,
    )


    # ========================================================
    # 16. Save completion marker LAST
    # ========================================================

    completion_information = {
        "status": "completed",
        "architecture": "ResNet50",
        "selected_configuration_id": (
            SELECTED_CONFIGURATION_ID
        ),
        "selected_final_stage": (
            SELECTED_FINAL_STAGE
        ),
        "model_path": (
            str(
                FINAL_MODEL_PATH
            )
        ),
        "training_images": (
            int(
                len(
                    final_training_dataframe
                )
            )
        ),
        "batch_size": (
            BATCH_SIZE
        ),
        "head_learning_rate": (
            HEAD_LEARNING_RATE
        ),
        "fine_tune_learning_rate": (
            FINE_TUNE_LEARNING_RATE
        ),
        "head_epochs": (
            HEAD_EPOCHS
        ),
        "fine_tune_epochs": (
            FINE_TUNE_EPOCHS
            if (
                SELECTED_FINAL_STAGE
                == "fine_tuning"
            )
            else 0
        ),
        "validation_used": False,
        "early_stopping_used": False,
        "testing_images_loaded": False,
        "completed_at": (
            datetime.now().isoformat()
        ),
    }


    save_json_atomic(
        completion_information,
        TRAINING_COMPLETION_PATH,
    )


    # ========================================================
    # 17. Final output
    # ========================================================

    print(
        "\n"
        + "=" * 70
    )

    print(
        "FINAL RESNET50 TRAINING COMPLETED"
    )

    print(
        "=" * 70
    )


    print(
        "Selected configuration:",
        SELECTED_CONFIGURATION_ID
    )


    print(
        "Training images:",
        len(
            final_training_dataframe
        )
    )


    print(
        "Batch size:",
        BATCH_SIZE
    )


    print(
        "Stage 1 epochs:",
        HEAD_EPOCHS
    )


    print(
        "Stage 1 learning rate:",
        HEAD_LEARNING_RATE
    )


    print(
        "Stage 2 epochs:",
        (
            FINE_TUNE_EPOCHS
            if (
                SELECTED_FINAL_STAGE
                == "fine_tuning"
            )
            else 0
        )
    )


    print(
        "Stage 2 learning rate:",
        FINE_TUNE_LEARNING_RATE
    )


    print(
        "\nStage 1 training time:",
        round(
            stage_1_seconds / 60,
            2
        ),
        "minutes"
    )


    print(
        "Stage 2 training time:",
        round(
            stage_2_seconds / 60,
            2
        ),
        "minutes"
    )


    print(
        "Total model training time:",
        round(
            total_model_training_seconds
            / 60,
            2
        ),
        "minutes"
    )


    print(
        "Model saving time:",
        round(
            model_saving_seconds,
            2
        ),
        "seconds"
    )


    print(
        "Complete final-training time:",
        round(
            complete_final_training_seconds
            / 60,
            2
        ),
        "minutes"
    )


    print(
        "\nFinal model:"
    )

    print(
        FINAL_MODEL_PATH
    )


    print(
        "\nResults directory:"
    )

    print(
        FINAL_RESULTS_DIR
    )


    print(
        "\nTesting images were not loaded "
        "or evaluated during final training."
    )

FINAL RESNET50 TRAINING

--- Selected configuration ---
Configuration: resnet50_bs16_headlr1e-03_ftlr1e-05
Batch size: 16
Head learning rate: 0.001
Fine-tuning learning rate: 1e-05
Selected final stage: fine_tuning
Stage 1 epochs: 13
Stage 2 epochs: 5

Incomplete final-training output was detected.
Removing the incomplete final-training directory...

GPU devices: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
Deterministic TensorFlow operations: enabled

--- Final Training data ---
Training images: 5600
Class mapping: {'glioma': 0, 'meningioma': 1, 'notumor': 2, 'pituitary': 3}
glioma      : 1400
meningioma  : 1400
notumor     : 1400
pituitary   : 1400
94765736/94765736 ━━━━━━━━━━━━━━━━━━━━ 5s 0us/step

--- Fresh final model ---
Model name: resnet50_transfer_learning
Total parameters: 23,595,908
Stage 1 trainable parameters: 8,196
Backbone trainable: False

STAGE 1 OF 2 — CLASSIFIER-HEAD TRAINING
Training images: 5600
Epochs: 13
Batch size: 16
Learning rate: 0.001
E


SAVING FINAL RESNET50 MODEL

FINAL RESNET50 TRAINING COMPLETED
Selected configuration: resnet50_bs16_headlr1e-03_ftlr1e-05
Training images: 5600
Batch size: 16
Stage 1 epochs: 13
Stage 1 learning rate: 0.001
Stage 2 epochs: 5
Stage 2 learning rate: 1e-05

Stage 1 training time: 1.54 minutes
Stage 2 training time: 0.91 minutes
Total model training time: 2.46 minutes
Model saving time: 1.67 seconds
Complete final-training time: 2.7 minutes

Final model:
/content/drive/MyDrive/brain_tumour_colab/results/resnet50_transfer_learning/resnet50_selected_final_no_augmentation_v2/final_resnet50_cropped.keras

Results directory:
/content/drive/MyDrive/brain_tumour_colab/results/resnet50_transfer_learning/resnet50_selected_final_no_augmentation_v2

Testing images were not loaded or evaluated during final training.


In [ ]:
# ============================================================
# FINAL RESNET50 EVALUATION ON HELD-OUT TESTING PARTITION
#
# Important:
#   - Loads the previously saved final ResNet50 v2 model
#   - Requires successful final-training completion first
#   - Uses the 1,598 cropped Testing images
#   - Does not train or modify the model
#   - Does not shuffle the Testing data
#   - Uses the same pseudo-RGB conversion and ResNet50
#     preprocessing used during training
#   - Verifies that Testing paths do not overlap the 5,600
#     training paths used during cross-validation/final training
#   - Writes the completion marker only after all Testing
#     evaluation outputs have been saved successfully
# ============================================================

import gc
import json
import shutil
import time

from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
import tensorflow as tf

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    cohen_kappa_score,
    confusion_matrix,
    f1_score,
    log_loss,
    matthews_corrcoef,
    precision_score,
    recall_score,
)


# ============================================================
# 1. Configuration
# ============================================================

CELL_START_TIME = time.perf_counter()

RANDOM_SEED = 42

IMAGE_HEIGHT = 224
IMAGE_WIDTH = 224
IMAGE_CHANNELS = 3
NUMBER_OF_CLASSES = 4

CLASS_NAMES = (
    "glioma",
    "meningioma",
    "notumor",
    "pituitary",
)

CLASS_TO_INDEX = {
    class_name: index
    for index, class_name in enumerate(CLASS_NAMES)
}

INDEX_TO_CLASS = {
    index: class_name
    for class_name, index in CLASS_TO_INDEX.items()
}


# ------------------------------------------------------------
# Current project/data paths
# ------------------------------------------------------------

PROJECT_ROOT = Path(
    "/content/brain-tumour-mri-classification"
)

DATA_DIRECTORY = (
    PROJECT_ROOT
    / "processed_data_cropped"
)

TESTING_DIRECTORY = (
    DATA_DIRECTORY
    / "Testing"
)

FOLDS_FILE = (
    PROJECT_ROOT
    / "splits"
    / "five_fold_cross_validation.csv"
)


# ------------------------------------------------------------
# Current ResNet50 v2 final-training paths
# ------------------------------------------------------------

DRIVE_ROOT = Path(
    "/content/drive/MyDrive/brain_tumour_colab"
)

FINAL_TRAINING_DIRECTORY = (
    DRIVE_ROOT
    / "results"
    / "resnet50_transfer_learning"
    / "resnet50_selected_final_no_augmentation_v2"
)

FINAL_MODEL_PATH = (
    FINAL_TRAINING_DIRECTORY
    / "final_resnet50_cropped.keras"
)

FINAL_TRAINING_CONFIGURATION_PATH = (
    FINAL_TRAINING_DIRECTORY
    / "final_training_configuration.json"
)

SELECTED_CONFIGURATION_USED_PATH = (
    FINAL_TRAINING_DIRECTORY
    / "selected_configuration_used.json"
)

TRAINING_COMPLETION_PATH = (
    FINAL_TRAINING_DIRECTORY
    / "training_complete.json"
)


# ------------------------------------------------------------
# Testing-evaluation output paths
# ------------------------------------------------------------

EVALUATION_DIRECTORY = (
    FINAL_TRAINING_DIRECTORY
    / "testing_evaluation"
)

PREDICTIONS_PATH = (
    EVALUATION_DIRECTORY
    / "testing_predictions.csv"
)

CONFUSION_MATRIX_PATH = (
    EVALUATION_DIRECTORY
    / "confusion_matrix.csv"
)

CLASSIFICATION_REPORT_CSV_PATH = (
    EVALUATION_DIRECTORY
    / "classification_report.csv"
)

CLASSIFICATION_REPORT_JSON_PATH = (
    EVALUATION_DIRECTORY
    / "classification_report.json"
)

METRICS_PATH = (
    EVALUATION_DIRECTORY
    / "testing_metrics.json"
)

TIMING_PATH = (
    EVALUATION_DIRECTORY
    / "testing_timing.json"
)

TESTING_MANIFEST_PATH = (
    EVALUATION_DIRECTORY
    / "testing_manifest.csv"
)

PROBABILITIES_PATH = (
    EVALUATION_DIRECTORY
    / "testing_probabilities.npz"
)

COMPLETION_PATH = (
    EVALUATION_DIRECTORY
    / "evaluation_complete.json"
)


EXPECTED_CLASS_COUNTS = {
    "glioma": 400,
    "meningioma": 400,
    "notumor": 398,
    "pituitary": 400,
}

EXPECTED_TESTING_IMAGES = 1598
EXPECTED_TRAINING_IMAGES = 5600


# ============================================================
# 2. Utility functions
# ============================================================

def save_json_atomic(
    data,
    destination,
):
    """Save JSON through a temporary file."""

    destination = Path(destination)

    temporary_path = destination.with_name(
        destination.name + ".tmp"
    )

    with temporary_path.open(
        "w",
        encoding="utf-8",
    ) as file:
        json.dump(
            data,
            file,
            indent=4,
            default=str,
        )

    temporary_path.replace(destination)


def save_dataframe_atomic(
    dataframe,
    destination,
    index=False,
):
    """Save a DataFrame through a temporary CSV file."""

    destination = Path(destination)

    temporary_path = destination.with_name(
        destination.name + ".tmp"
    )

    dataframe.to_csv(
        temporary_path,
        index=index,
    )

    temporary_path.replace(destination)


def save_probabilities_atomic(
    destination,
    **arrays,
):
    """Save a compressed NumPy archive through a temporary file."""

    destination = Path(destination)

    temporary_path = destination.with_name(
        destination.stem + ".tmp.npz"
    )

    np.savez_compressed(
        temporary_path,
        **arrays,
    )

    temporary_path.replace(destination)


def load_json_file(path):
    """Load one JSON file."""

    path = Path(path)

    with path.open(
        "r",
        encoding="utf-8",
    ) as file:
        return json.load(file)


def load_and_preprocess_testing_image(
    image_path,
):
    """
    Load one grayscale PNG, convert it to pseudo-RGB,
    cast to float32 and apply ResNet50 preprocessing.
    """

    image_bytes = tf.io.read_file(
        image_path
    )

    image = tf.io.decode_png(
        image_bytes,
        channels=1,
    )

    image = tf.ensure_shape(
        image,
        (
            IMAGE_HEIGHT,
            IMAGE_WIDTH,
            1,
        ),
    )

    image = tf.image.grayscale_to_rgb(
        image
    )

    image = tf.cast(
        image,
        tf.float32,
    )

    image = (
        tf.keras.applications.resnet50
        .preprocess_input(image)
    )

    image = tf.ensure_shape(
        image,
        (
            IMAGE_HEIGHT,
            IMAGE_WIDTH,
            IMAGE_CHANNELS,
        ),
    )

    return image


# ============================================================
# 3. Reproducibility
# ============================================================

np.random.seed(
    RANDOM_SEED
)

tf.keras.utils.set_random_seed(
    RANDOM_SEED
)

try:

    tf.config.experimental.enable_op_determinism()

    deterministic_status = "enabled"

except Exception as error:

    deterministic_status = (
        "requested, but TensorFlow returned: "
        f"{error}"
    )


# ============================================================
# 4. Run the final held-out Testing evaluation
# ============================================================

def run_final_resnet50_testing_evaluation():

    print("=" * 70)
    print("FINAL RESNET50 TESTING EVALUATION")
    print("=" * 70)

    print(
        "TensorFlow version:",
        tf.__version__,
    )

    print(
        "GPU devices:",
        tf.config.list_physical_devices(
            "GPU"
        ),
    )

    print(
        "Testing directory:",
        TESTING_DIRECTORY,
    )

    print(
        "Final model:",
        FINAL_MODEL_PATH,
    )

    print(
        "Evaluation directory:",
        EVALUATION_DIRECTORY,
    )

    print(
        "Data augmentation:",
        False,
    )

    print(
        "Testing data shuffled:",
        False,
    )

    print(
        "Deterministic TensorFlow operations:",
        deterministic_status,
    )


    # ========================================================
    # 5. Verify final-training completion
    # ========================================================

    if not TESTING_DIRECTORY.exists():

        raise FileNotFoundError(
            "The cropped Testing directory was not found:\n"
            f"{TESTING_DIRECTORY}"
        )


    if not FOLDS_FILE.exists():

        raise FileNotFoundError(
            "The fixed five-fold assignment file was not found:\n"
            f"{FOLDS_FILE}"
        )


    if not FINAL_MODEL_PATH.exists():

        raise FileNotFoundError(
            "The final trained ResNet50 v2 model was not found:\n"
            f"{FINAL_MODEL_PATH}"
        )


    if not FINAL_TRAINING_CONFIGURATION_PATH.exists():

        raise FileNotFoundError(
            "The final-training configuration was not found:\n"
            f"{FINAL_TRAINING_CONFIGURATION_PATH}"
        )


    if not SELECTED_CONFIGURATION_USED_PATH.exists():

        raise FileNotFoundError(
            "The selected-configuration record was not found:\n"
            f"{SELECTED_CONFIGURATION_USED_PATH}"
        )


    if not TRAINING_COMPLETION_PATH.exists():

        raise FileNotFoundError(
            "The final-training completion marker was not found:\n"
            f"{TRAINING_COMPLETION_PATH}\n\n"
            "Testing evaluation must not run until final "
            "training has completed successfully."
        )


    training_completion = load_json_file(
        TRAINING_COMPLETION_PATH
    )

    if (
        training_completion.get("status")
        != "completed"
    ):

        raise RuntimeError(
            "The final-training completion marker does not "
            "report status='completed'."
        )


    final_training_configuration = (
        load_json_file(
            FINAL_TRAINING_CONFIGURATION_PATH
        )
    )

    selected_configuration_used = (
        load_json_file(
            SELECTED_CONFIGURATION_USED_PATH
        )
    )


    # --------------------------------------------------------
    # Obtain the batch size from the saved final-training
    # configuration rather than hard-coding it here.
    # --------------------------------------------------------

    batch_size_value = (
        final_training_configuration.get(
            "batch_size"
        )
    )

    if batch_size_value is None:

        batch_size_value = (
            selected_configuration_used.get(
                "batch_size"
            )
        )


    if batch_size_value is None:

        raise KeyError(
            "The saved final-training configuration does not "
            "contain a batch_size value."
        )


    batch_size = int(
        batch_size_value
    )


    selected_configuration_id = (
        final_training_configuration.get(
            "configuration_id"
        )
        or final_training_configuration.get(
            "selected_configuration_id"
        )
        or selected_configuration_used.get(
            "configuration_id"
        )
        or selected_configuration_used.get(
            "selected_configuration_id"
        )
        or "not recorded"
    )


    print(
        "\n--- Final-training verification ---"
    )

    print(
        "Training completion status:",
        training_completion.get(
            "status"
        ),
    )

    print(
        "Selected configuration:",
        selected_configuration_id,
    )

    print(
        "Testing batch size:",
        batch_size,
    )

    print(
        "Final model exists:",
        FINAL_MODEL_PATH.exists(),
    )


    # ========================================================
    # 6. Completion/resume protection
    # ========================================================

    required_evaluation_outputs = [
        PREDICTIONS_PATH,
        CONFUSION_MATRIX_PATH,
        CLASSIFICATION_REPORT_CSV_PATH,
        CLASSIFICATION_REPORT_JSON_PATH,
        METRICS_PATH,
        TIMING_PATH,
        TESTING_MANIFEST_PATH,
        PROBABILITIES_PATH,
        COMPLETION_PATH,
    ]


    evaluation_is_complete = False

    if COMPLETION_PATH.exists():

        try:

            existing_completion = load_json_file(
                COMPLETION_PATH
            )

            all_required_outputs_exist = all(
                path.exists()
                for path
                in required_evaluation_outputs
            )

            evaluation_is_complete = (
                existing_completion.get(
                    "status"
                )
                == "completed"
                and all_required_outputs_exist
            )

        except Exception:

            evaluation_is_complete = False


    if evaluation_is_complete:

        print("\n" + "=" * 70)
        print(
            "TESTING EVALUATION ALREADY COMPLETED"
        )
        print("=" * 70)

        print(
            "The completed held-out Testing evaluation "
            "was found."
        )

        print(
            "The model will NOT be evaluated again."
        )

        existing_metrics = load_json_file(
            METRICS_PATH
        )

        print(
            "\nAccuracy:",
            f"{existing_metrics['accuracy']:.6f}",
        )

        print(
            "Balanced accuracy:",
            f"{existing_metrics['balanced_accuracy']:.6f}",
        )

        print(
            "Macro F1-score:",
            f"{existing_metrics['macro_f1']:.6f}",
        )

        print(
            "\nExisting evaluation results:"
        )

        print(
            EVALUATION_DIRECTORY
        )

        return


    # --------------------------------------------------------
    # If an earlier attempt created only partial output,
    # remove that incomplete evaluation directory and rerun
    # the evaluation cleanly.
    # --------------------------------------------------------

    if EVALUATION_DIRECTORY.exists():

        print(
            "\nRemoving incomplete Testing evaluation "
            "output from an earlier attempt..."
        )

        shutil.rmtree(
            EVALUATION_DIRECTORY
        )


    EVALUATION_DIRECTORY.mkdir(
        parents=True,
        exist_ok=True,
    )


    # ========================================================
    # 7. Enumerate and verify Testing images
    # ========================================================

    manifest_start = time.perf_counter()

    testing_paths = []
    testing_labels = []
    testing_class_names = []
    testing_relative_paths = []

    class_counts = {}


    for class_name in CLASS_NAMES:

        class_directory = (
            TESTING_DIRECTORY
            / class_name
        )

        if not class_directory.exists():

            raise FileNotFoundError(
                "Testing class directory not found: "
                f"{class_directory}"
            )


        class_image_paths = sorted(
            class_directory.glob(
                "*.png"
            )
        )


        class_counts[class_name] = len(
            class_image_paths
        )


        expected_count = (
            EXPECTED_CLASS_COUNTS[
                class_name
            ]
        )


        if (
            len(class_image_paths)
            != expected_count
        ):

            raise RuntimeError(
                f"Expected {expected_count} Testing images "
                f"for {class_name}, but found "
                f"{len(class_image_paths)}."
            )


        class_index = (
            CLASS_TO_INDEX[
                class_name
            ]
        )


        for image_path in class_image_paths:

            testing_paths.append(
                str(image_path)
            )

            testing_labels.append(
                class_index
            )

            testing_class_names.append(
                class_name
            )

            # Store paths relative to processed_data_cropped,
            # e.g. Testing/glioma/image.png.
            testing_relative_paths.append(
                image_path
                .relative_to(
                    DATA_DIRECTORY
                )
                .as_posix()
            )


    testing_paths = np.asarray(
        testing_paths,
        dtype=str,
    )

    testing_labels = np.asarray(
        testing_labels,
        dtype=np.int32,
    )

    testing_class_names = np.asarray(
        testing_class_names,
        dtype=str,
    )

    testing_relative_paths = np.asarray(
        testing_relative_paths,
        dtype=str,
    )


    if (
        len(testing_paths)
        != EXPECTED_TESTING_IMAGES
    ):

        raise RuntimeError(
            f"Expected {EXPECTED_TESTING_IMAGES:,} "
            "cropped Testing images, but found "
            f"{len(testing_paths):,}."
        )


    if (
        len(testing_paths)
        != len(testing_labels)
    ):

        raise RuntimeError(
            "The number of Testing labels does not "
            "match the number of Testing paths."
        )


    if (
        len(np.unique(testing_relative_paths))
        != EXPECTED_TESTING_IMAGES
    ):

        raise RuntimeError(
            "Duplicate Testing relative paths were detected."
        )


    # ========================================================
    # 8. Verify no Training/Testing path overlap
    # ========================================================

    fold_assignments = pd.read_csv(
        FOLDS_FILE
    )


    if "relative_path" not in fold_assignments.columns:

        raise RuntimeError(
            "The fixed fold CSV does not contain the "
            "required relative_path column."
        )


    if (
        len(fold_assignments)
        != EXPECTED_TRAINING_IMAGES
    ):

        raise RuntimeError(
            f"Expected {EXPECTED_TRAINING_IMAGES:,} "
            "training assignments in the fold CSV, "
            f"but found {len(fold_assignments):,}."
        )


    training_relative_paths = {
        Path(str(path)).as_posix()
        for path
        in fold_assignments[
            "relative_path"
        ]
    }


    testing_relative_path_set = set(
        testing_relative_paths.tolist()
    )


    path_overlap = (
        training_relative_paths
        & testing_relative_path_set
    )


    if path_overlap:

        overlap_preview = sorted(
            path_overlap
        )[:10]

        raise RuntimeError(
            "Training/Testing path overlap was detected.\n"
            f"Examples: {overlap_preview}"
        )


    manifest_seconds = (
        time.perf_counter()
        - manifest_start
    )


    print(
        "\n--- Testing data verification ---"
    )

    print(
        "Testing images:",
        len(testing_paths),
    )

    print(
        "Class mapping:",
        CLASS_TO_INDEX,
    )


    for class_name in CLASS_NAMES:

        print(
            f"{class_name:<12}: "
            f"{class_counts[class_name]}"
        )


    print(
        "Training paths checked:",
        len(training_relative_paths),
    )

    print(
        "Training/Testing path overlap:",
        len(path_overlap),
    )


    testing_manifest = pd.DataFrame(
        {
            "relative_path":
                testing_relative_paths,
            "true_class":
                testing_class_names,
            "true_index":
                testing_labels,
        }
    )


    save_dataframe_atomic(
        testing_manifest,
        TESTING_MANIFEST_PATH,
        index=False,
    )


    # ========================================================
    # 9. Create deterministic Testing dataset
    # ========================================================

    testing_dataset = (
        tf.data.Dataset
        .from_tensor_slices(
            testing_paths
        )
    )


    dataset_options = (
        tf.data.Options()
    )

    dataset_options.experimental_deterministic = True


    testing_dataset = (
        testing_dataset
        .with_options(
            dataset_options
        )
    )


    testing_dataset = (
        testing_dataset
        .map(
            load_and_preprocess_testing_image,
            num_parallel_calls=
                tf.data.AUTOTUNE,
            deterministic=True,
        )
    )


    testing_dataset = (
        testing_dataset
        .batch(
            batch_size,
            drop_remainder=False,
        )
    )


    # Cache the preprocessed images in system memory.
    #
    # The first complete iteration measures PNG decoding,
    # grayscale-to-RGB conversion and ResNet50 preprocessing.
    #
    # The later prediction pass uses the cached tensors so
    # its timing excludes PNG decoding/preprocessing.

    cached_testing_dataset = (
        testing_dataset
        .cache()
        .prefetch(
            tf.data.AUTOTUNE
        )
    )


    # ========================================================
    # 10. Load the saved final ResNet50 model
    # ========================================================

    # Release references left by the final-training cell.

    for variable_name in (
        "model",
        "base_model",
        "training_dataset",
        "sample_images",
        "sample_labels",
    ):

        globals().pop(
            variable_name,
            None,
        )


    gc.collect()

    tf.keras.backend.clear_session()


    model_loading_start = (
        time.perf_counter()
    )


    final_model = (
        tf.keras.models.load_model(
            FINAL_MODEL_PATH,
            compile=False,
        )
    )


    model_loading_seconds = (
        time.perf_counter()
        - model_loading_start
    )


    print(
        "\n--- Loaded model verification ---"
    )

    print(
        "Model name:",
        final_model.name,
    )

    print(
        "Input shape:",
        final_model.input_shape,
    )

    print(
        "Output shape:",
        final_model.output_shape,
    )

    print(
        "Total parameters:",
        f"{final_model.count_params():,}",
    )

    print(
        "Model loading time:",
        round(
            model_loading_seconds,
            2,
        ),
        "seconds",
    )


    if (
        tuple(
            final_model.input_shape[1:]
        )
        != (
            IMAGE_HEIGHT,
            IMAGE_WIDTH,
            IMAGE_CHANNELS,
        )
    ):

        raise RuntimeError(
            "Unexpected final-model input shape: "
            f"{final_model.input_shape}"
        )


    if (
        int(
            final_model.output_shape[-1]
        )
        != NUMBER_OF_CLASSES
    ):

        raise RuntimeError(
            "Expected four output probabilities, "
            "but the model output shape is "
            f"{final_model.output_shape}."
        )


    # ========================================================
    # 11. Populate preprocessing cache
    # ========================================================

    print("\n" + "=" * 70)
    print("TESTING PREPROCESSING")
    print("=" * 70)

    print(
        "Decoding and preprocessing all "
        f"{EXPECTED_TESTING_IMAGES:,} Testing images..."
    )


    preprocessing_start = (
        time.perf_counter()
    )

    cached_image_count = 0
    sample_testing_batch = None


    for image_batch in cached_testing_dataset:

        if sample_testing_batch is None:

            sample_testing_batch = (
                image_batch
            )


        cached_image_count += int(
            image_batch.shape[0]
        )


    preprocessing_seconds = (
        time.perf_counter()
        - preprocessing_start
    )


    if (
        cached_image_count
        != EXPECTED_TESTING_IMAGES
    ):

        raise RuntimeError(
            f"Expected to preprocess "
            f"{EXPECTED_TESTING_IMAGES:,} "
            "Testing images, but processed "
            f"{cached_image_count:,}."
        )


    if sample_testing_batch is None:

        raise RuntimeError(
            "No Testing image batch was produced."
        )


    if not bool(
        tf.reduce_all(
            tf.math.is_finite(
                sample_testing_batch
            )
        ).numpy()
    ):

        raise RuntimeError(
            "Non-finite values were detected in the "
            "preprocessed Testing images."
        )


    print(
        "Preprocessed images:",
        cached_image_count,
    )

    print(
        "Sample batch shape:",
        sample_testing_batch.shape,
    )

    print(
        "Testing preprocessing time:",
        round(
            preprocessing_seconds,
            2,
        ),
        "seconds",
    )


    # ========================================================
    # 12. Warm up the model
    # ========================================================

    # This warm-up avoids including TensorFlow graph tracing
    # and initial GPU setup in the measured inference time.

    warmup_output = final_model(
        sample_testing_batch,
        training=False,
    )

    _ = warmup_output.numpy()


    # ========================================================
    # 13. Timed held-out Testing inference
    # ========================================================

    print("\n" + "=" * 70)
    print("HELD-OUT TESTING INFERENCE")
    print("=" * 70)


    inference_start = (
        time.perf_counter()
    )


    testing_probabilities = (
        final_model.predict(
            cached_testing_dataset,
            verbose=1,
        )
    )


    inference_seconds = (
        time.perf_counter()
        - inference_start
    )


    testing_probabilities = np.asarray(
        testing_probabilities,
        dtype=np.float32,
    )


    if (
        testing_probabilities.shape
        != (
            EXPECTED_TESTING_IMAGES,
            NUMBER_OF_CLASSES,
        )
    ):

        raise RuntimeError(
            "Unexpected prediction matrix shape: "
            f"{testing_probabilities.shape}"
        )


    if not np.isfinite(
        testing_probabilities
    ).all():

        raise RuntimeError(
            "Non-finite values were detected in the "
            "Testing probabilities."
        )


    probability_sums = (
        testing_probabilities.sum(
            axis=1
        )
    )


    if not np.allclose(
        probability_sums,
        1.0,
        atol=1e-5,
    ):

        raise RuntimeError(
            "Some Testing probability rows do not "
            "sum to one."
        )


    predicted_indices = np.argmax(
        testing_probabilities,
        axis=1,
    ).astype(
        np.int32
    )


    predicted_class_names = np.asarray(
        [
            INDEX_TO_CLASS[
                int(index)
            ]
            for index
            in predicted_indices
        ],
        dtype=str,
    )


    prediction_confidences = np.max(
        testing_probabilities,
        axis=1,
    )


    correct_predictions = (
        predicted_indices
        == testing_labels
    )


    inference_milliseconds_per_image = (
        inference_seconds
        / len(testing_paths)
        * 1000
    )


    end_to_end_testing_seconds = (
        preprocessing_seconds
        + inference_seconds
    )


    print(
        "\nCached model inference time:",
        round(
            inference_seconds,
            2,
        ),
        "seconds",
    )

    print(
        "Cached inference time per image:",
        round(
            inference_milliseconds_per_image,
            3,
        ),
        "milliseconds",
    )

    print(
        "Preprocessing + inference time:",
        round(
            end_to_end_testing_seconds,
            2,
        ),
        "seconds",
    )


    # ========================================================
    # 14. Calculate final Testing metrics
    # ========================================================

    labels = list(
        range(
            NUMBER_OF_CLASSES
        )
    )


    accuracy = accuracy_score(
        testing_labels,
        predicted_indices,
    )


    balanced_accuracy = (
        balanced_accuracy_score(
            testing_labels,
            predicted_indices,
        )
    )


    macro_precision = precision_score(
        testing_labels,
        predicted_indices,
        labels=labels,
        average="macro",
        zero_division=0,
    )


    macro_recall = recall_score(
        testing_labels,
        predicted_indices,
        labels=labels,
        average="macro",
        zero_division=0,
    )


    macro_f1 = f1_score(
        testing_labels,
        predicted_indices,
        labels=labels,
        average="macro",
        zero_division=0,
    )


    weighted_precision = precision_score(
        testing_labels,
        predicted_indices,
        labels=labels,
        average="weighted",
        zero_division=0,
    )


    weighted_recall = recall_score(
        testing_labels,
        predicted_indices,
        labels=labels,
        average="weighted",
        zero_division=0,
    )


    weighted_f1 = f1_score(
        testing_labels,
        predicted_indices,
        labels=labels,
        average="weighted",
        zero_division=0,
    )


    matthews_correlation = (
        matthews_corrcoef(
            testing_labels,
            predicted_indices,
        )
    )


    cohen_kappa = (
        cohen_kappa_score(
            testing_labels,
            predicted_indices,
        )
    )


    testing_log_loss = log_loss(
        testing_labels,
        testing_probabilities,
        labels=labels,
    )


    confusion = confusion_matrix(
        testing_labels,
        predicted_indices,
        labels=labels,
    )


    report_dictionary = (
        classification_report(
            testing_labels,
            predicted_indices,
            labels=labels,
            target_names=list(
                CLASS_NAMES
            ),
            output_dict=True,
            zero_division=0,
        )
    )


    number_correct = int(
        correct_predictions.sum()
    )


    number_incorrect = int(
        len(correct_predictions)
        - number_correct
    )


    metrics = {
        "model_name":
            "ResNet50 transfer learning",
        "configuration_id":
            selected_configuration_id,
        "dataset_variant":
            "cropped",
        "dataset_partition":
            "Testing",
        "testing_image_count":
            int(
                len(testing_paths)
            ),
        "testing_batch_size":
            int(batch_size),
        "class_names":
            list(CLASS_NAMES),
        "class_to_index":
            CLASS_TO_INDEX,
        "class_counts":
            class_counts,
        "accuracy":
            float(accuracy),
        "balanced_accuracy":
            float(
                balanced_accuracy
            ),
        "macro_precision":
            float(
                macro_precision
            ),
        "macro_recall":
            float(
                macro_recall
            ),
        "macro_f1":
            float(
                macro_f1
            ),
        "weighted_precision":
            float(
                weighted_precision
            ),
        "weighted_recall":
            float(
                weighted_recall
            ),
        "weighted_f1":
            float(
                weighted_f1
            ),
        "matthews_correlation_coefficient":
            float(
                matthews_correlation
            ),
        "cohen_kappa":
            float(
                cohen_kappa
            ),
        "multiclass_log_loss":
            float(
                testing_log_loss
            ),
        "correct_predictions":
            number_correct,
        "incorrect_predictions":
            number_incorrect,
        "mean_prediction_confidence":
            float(
                prediction_confidences.mean()
            ),
        "data_augmentation":
            False,
        "testing_data_shuffled":
            False,
        "model_modified_during_testing":
            False,
        "training_testing_path_overlap":
            int(
                len(path_overlap)
            ),
        "tensorflow_version":
            tf.__version__,
        "evaluated_at":
            datetime.now().isoformat(),
    }


    # ========================================================
    # 15. Save prediction-level results
    # ========================================================

    predictions_frame = pd.DataFrame(
        {
            "relative_path":
                testing_relative_paths,
            "true_class":
                testing_class_names,
            "true_index":
                testing_labels,
            "predicted_class":
                predicted_class_names,
            "predicted_index":
                predicted_indices,
            "correct":
                correct_predictions,
            "prediction_confidence":
                prediction_confidences,
            "probability_glioma":
                testing_probabilities[
                    :, 0
                ],
            "probability_meningioma":
                testing_probabilities[
                    :, 1
                ],
            "probability_notumor":
                testing_probabilities[
                    :, 2
                ],
            "probability_pituitary":
                testing_probabilities[
                    :, 3
                ],
        }
    )


    save_dataframe_atomic(
        predictions_frame,
        PREDICTIONS_PATH,
        index=False,
    )


    save_probabilities_atomic(
        PROBABILITIES_PATH,
        probabilities=
            testing_probabilities,
        true_labels=
            testing_labels,
        predicted_labels=
            predicted_indices,
        relative_paths=
            testing_relative_paths,
        class_names=
            np.asarray(
                CLASS_NAMES,
                dtype=str,
            ),
    )


    # ========================================================
    # 16. Save confusion matrix and classification report
    # ========================================================

    confusion_frame = pd.DataFrame(
        confusion,
        index=[
            f"actual_{class_name}"
            for class_name
            in CLASS_NAMES
        ],
        columns=[
            f"predicted_{class_name}"
            for class_name
            in CLASS_NAMES
        ],
    )


    save_dataframe_atomic(
        confusion_frame,
        CONFUSION_MATRIX_PATH,
        index=True,
    )


    report_frame = pd.DataFrame(
        report_dictionary
    ).transpose()


    save_dataframe_atomic(
        report_frame,
        CLASSIFICATION_REPORT_CSV_PATH,
        index=True,
    )


    save_json_atomic(
        report_dictionary,
        CLASSIFICATION_REPORT_JSON_PATH,
    )


    save_json_atomic(
        metrics,
        METRICS_PATH,
    )


    # ========================================================
    # 17. Save timing information
    # ========================================================

    complete_cell_seconds = (
        time.perf_counter()
        - CELL_START_TIME
    )


    timing_information = {
        "testing_manifest_preparation_seconds":
            manifest_seconds,
        "model_loading_seconds":
            model_loading_seconds,
        "testing_preprocessing_seconds":
            preprocessing_seconds,
        "cached_model_inference_seconds":
            inference_seconds,
        "cached_model_inference_milliseconds_per_image":
            inference_milliseconds_per_image,
        "preprocessing_plus_inference_seconds":
            end_to_end_testing_seconds,
        "complete_evaluation_cell_seconds":
            complete_cell_seconds,
        "complete_evaluation_cell_minutes":
            complete_cell_seconds / 60,
        "timing_notes": {
            "testing_preprocessing_seconds": (
                "Includes PNG decoding, grayscale-to-RGB "
                "conversion, ResNet50 preprocessing and "
                "population of the in-memory cache."
            ),
            "cached_model_inference_seconds": (
                "Measured after preprocessing was cached "
                "and after one warm-up batch. Includes "
                "cached dataset iteration, host-to-device "
                "transfer and model forward inference."
            ),
            "preprocessing_plus_inference_seconds": (
                "Sum of Testing preprocessing and cached "
                "model inference. Model loading is excluded."
            ),
        },
    }


    save_json_atomic(
        timing_information,
        TIMING_PATH,
    )


    # ========================================================
    # 18. Save completion marker LAST
    # ========================================================

    completion_information = {
        "status":
            "completed",
        "model_path":
            str(
                FINAL_MODEL_PATH
            ),
        "evaluation_directory":
            str(
                EVALUATION_DIRECTORY
            ),
        "configuration_id":
            selected_configuration_id,
        "testing_images":
            int(
                len(testing_paths)
            ),
        "accuracy":
            float(
                accuracy
            ),
        "balanced_accuracy":
            float(
                balanced_accuracy
            ),
        "macro_f1":
            float(
                macro_f1
            ),
        "completed_at":
            datetime.now().isoformat(),
    }


    save_json_atomic(
        completion_information,
        COMPLETION_PATH,
    )


    # ========================================================
    # 19. Final output
    # ========================================================

    print("\n" + "=" * 70)
    print("FINAL RESNET50 TESTING RESULTS")
    print("=" * 70)

    print(
        "Selected configuration:",
        selected_configuration_id,
    )

    print(
        "Testing images:",
        len(testing_paths),
    )

    print(
        "Testing batch size:",
        batch_size,
    )

    print(
        f"\nAccuracy:            "
        f"{accuracy:.6f}"
    )

    print(
        f"Balanced accuracy:   "
        f"{balanced_accuracy:.6f}"
    )

    print(
        f"Macro precision:     "
        f"{macro_precision:.6f}"
    )

    print(
        f"Macro recall:        "
        f"{macro_recall:.6f}"
    )

    print(
        f"Macro F1-score:      "
        f"{macro_f1:.6f}"
    )

    print(
        f"Weighted F1-score:   "
        f"{weighted_f1:.6f}"
    )

    print(
        f"MCC:                 "
        f"{matthews_correlation:.6f}"
    )

    print(
        f"Cohen's kappa:       "
        f"{cohen_kappa:.6f}"
    )

    print(
        f"Multiclass log loss: "
        f"{testing_log_loss:.6f}"
    )


    print(
        "\nCorrect predictions:",
        number_correct,
    )

    print(
        "Incorrect predictions:",
        number_incorrect,
    )


    print(
        "\nConfusion matrix"
    )

    print(
        "Rows = actual classes; "
        "columns = predicted classes"
    )

    print(
        confusion_frame
    )


    print(
        "\nClassification report"
    )

    print(
        classification_report(
            testing_labels,
            predicted_indices,
            labels=labels,
            target_names=list(
                CLASS_NAMES
            ),
            digits=6,
            zero_division=0,
        )
    )


    print(
        "\n--- Timing ---"
    )

    print(
        "Testing preprocessing:",
        round(
            preprocessing_seconds,
            2,
        ),
        "seconds",
    )

    print(
        "Cached model inference:",
        round(
            inference_seconds,
            2,
        ),
        "seconds",
    )

    print(
        "Inference per image:",
        round(
            inference_milliseconds_per_image,
            3,
        ),
        "milliseconds",
    )

    print(
        "Preprocessing + inference:",
        round(
            end_to_end_testing_seconds,
            2,
        ),
        "seconds",
    )

    print(
        "Complete evaluation cell:",
        round(
            complete_cell_seconds,
            2,
        ),
        "seconds",
    )


    print(
        "\nTraining/Testing path overlap:",
        len(path_overlap),
    )

    print(
        "\nEvaluation results saved to:"
    )

    print(
        EVALUATION_DIRECTORY
    )

    print(
        "\nTesting evaluation completion marker:"
    )

    print(
        COMPLETION_PATH
    )


# ============================================================
# 20. Execute the evaluation
# ============================================================

run_final_resnet50_testing_evaluation()

FINAL RESNET50 TESTING EVALUATION
TensorFlow version: 2.20.0
GPU devices: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
Testing directory: /content/brain-tumour-mri-classification/processed_data_cropped/Testing
Final model: /content/drive/MyDrive/brain_tumour_colab/results/resnet50_transfer_learning/resnet50_selected_final_no_augmentation_v2/final_resnet50_cropped.keras
Evaluation directory: /content/drive/MyDrive/brain_tumour_colab/results/resnet50_transfer_learning/resnet50_selected_final_no_augmentation_v2/testing_evaluation
Data augmentation: False
Testing data shuffled: False
Deterministic TensorFlow operations: enabled

--- Final-training verification ---
Training completion status: completed
Selected configuration: resnet50_bs16_headlr1e-03_ftlr1e-05
Testing batch size: 16
Final model exists: True

--- Testing data verification ---
Testing images: 1598
Class mapping: {'glioma': 0, 'meningioma': 1, 'notumor': 2, 'pituitary': 3}
glioma      : 400
meningioma  :